# 01 — UniTox Dataset Audit & Baseline Setup

This notebook is the first step of the Reliability-Aware UniTox project.

## Goal

Before changing the UniTox method, we will:

1. Load the official small-molecule benchmark.
2. Audit its structure and labels.
3. Verify the supplied Morgan/RDKit fingerprint files.
4. Reproduce the UniTox Random Forest configuration for one task:
   - Liver toxicity
   - Binary label
   - Morgan fingerprint
   - 10-fold cross-validation
   - ROC-AUC

**Reference configuration from the UniTox repository:** Morgan fingerprint = 2048 bits, radius = 2, Random Forest = 500 trees, seed = 0, 10-fold CV, metric = AUC.


In [1]:
import os

# Folder containing the UniTox repository
PROJECT_ROOT = r"D:\pro dataset\UniTox-main"

DATA_DIR = os.path.join(PROJECT_ROOT, "Data")

CSV_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark.csv"
)

MORGAN_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark-morgan.npz"
)

RDKIT_PATH = os.path.join(
    DATA_DIR,
    "UniTox-Small-Molecule-Benchmark-rdkit.npz"
)

In [2]:
import os
import numpy as np
import pandas as pd

print("CSV exists:", os.path.exists(CSV_PATH))
print("Morgan NPZ exists:", os.path.exists(MORGAN_PATH))
print("RDKit NPZ exists:", os.path.exists(RDKIT_PATH))


CSV exists: True
Morgan NPZ exists: True
RDKit NPZ exists: True


In [3]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumns:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")


Shape: (1349, 43)

Columns:
 1. generic_name
 2. cardio_toxicity_reasoning
 3. cardio_toxicity_ternary_rating
 4. cardio_toxicity_binary_rating
 5. cardio_toxicity_confident_ternary_rating_0_1
 6. cardio_toxicity_binary_rating_0_1
 7. dermatological_toxicity_reasoning
 8. dermatological_toxicity_ternary_rating
 9. dermatological_toxicity_binary_rating
10. dermatological_toxicity_confident_ternary_rating_0_1
11. dermatological_toxicity_binary_rating_0_1
12. hematological_reasoning
13. hematological_ternary_rating
14. hematological_binary_rating
15. hematological_confident_ternary_rating_0_1
16. hematological_binary_rating_0_1
17. infertility_reasoning
18. infertility_ternary_rating
19. infertility_binary_rating
20. infertility_confident_ternary_rating_0_1
21. infertility_binary_rating_0_1
22. liver_toxicity_reasoning
23. liver_toxicity_ternary_rating
24. liver_toxicity_binary_rating
25. liver_toxicity_confident_ternary_rating_0_1
26. liver_toxicity_binary_rating_0_1
27. ototoxicity_reas

In [4]:
print("Unique drugs:", df["generic_name"].nunique())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate drug names:", df["generic_name"].duplicated().sum())
print("Missing SMILES:", df["smiles"].isna().sum())
print("Missing all_smiles:", df["all_smiles"].isna().sum())
print("Rows:", len(df))


Unique drugs: 1349
Duplicate rows: 0
Duplicate drug names: 0
Missing SMILES: 0
Missing all_smiles: 0
Rows: 1349


In [5]:
toxicity_targets = [
    "cardio_toxicity",
    "dermatological_toxicity",
    "hematological",
    "infertility",
    "liver_toxicity",
    "ototoxicity",
    "pulmonary_toxicity",
    "renal_toxicity",
]

summary = []
for tox in toxicity_targets:
    ternary = f"{tox}_ternary_rating"
    binary = f"{tox}_binary_rating"
    confident = f"{tox}_confident_ternary_rating_0_1"
    binary01 = f"{tox}_binary_rating_0_1"

    summary.append({
        "toxicity": tox,
        "ternary_missing": df[ternary].isna().sum(),
        "binary_missing": df[binary].isna().sum(),
        "confident_missing": df[confident].isna().sum(),
        "binary01_missing": df[binary01].isna().sum(),
        "binary_yes": (df[binary] == "Yes").sum(),
        "binary_no": (df[binary] == "No").sum(),
        "ternary_less": (df[ternary] == "Less").sum(),
        "ternary_most": (df[ternary] == "Most").sum(),
        "ternary_no": (df[ternary] == "No").sum(),
    })

pd.DataFrame(summary)


,toxicity,ternary_missing,binary_missing,confident_missing,binary01_missing,binary_yes,binary_no,ternary_less,ternary_most,ternary_no
0,cardio_toxicity,0,0,448,0,755,594,429,594,307
1,dermatological_toxicity,0,0,394,0,1199,150,394,828,127
2,hematological,0,0,274,0,876,473,274,655,420
3,infertility,0,0,328,0,503,846,328,336,685
4,liver_toxicity,0,0,471,0,736,613,471,481,397
5,ototoxicity,0,0,166,0,121,1228,166,54,1129
6,pulmonary_toxicity,0,0,352,0,498,851,352,357,640
7,renal_toxicity,0,0,401,0,508,841,401,369,579


In [6]:
tox = "liver_toxicity"

print("Liver ternary:")
print(df[f"{tox}_ternary_rating"].value_counts(dropna=False))

print("\nLiver binary:")
print(df[f"{tox}_binary_rating"].value_counts(dropna=False))

print("\nLiver confident 0/1:")
print(df[f"{tox}_confident_ternary_rating_0_1"].value_counts(dropna=False))

print("\nLiver binary 0/1:")
print(df[f"{tox}_binary_rating_0_1"].value_counts(dropna=False))


Liver ternary:
liver_toxicity_ternary_rating
Most    481
Less    471
No      397
Name: count, dtype: int64

Liver binary:
liver_toxicity_binary_rating
Yes    736
No     613
Name: count, dtype: int64

Liver confident 0/1:
liver_toxicity_confident_ternary_rating_0_1
1.0    481
NaN    471
0.0    397
Name: count, dtype: int64

Liver binary 0/1:
liver_toxicity_binary_rating_0_1
1    736
0    613
Name: count, dtype: int64


In [7]:
morgan = np.load(MORGAN_PATH)["features"]
rdkit = np.load(RDKIT_PATH)["features"]

print("Morgan shape:", morgan.shape, "dtype:", morgan.dtype)
print("RDKit shape:", rdkit.shape, "dtype:", rdkit.dtype)


Morgan shape: (1349, 2048) dtype: float32
RDKit shape: (1349, 200) dtype: float32


In [8]:
pip install rdkit

Note: you may need to restart the kernel to use updated packages.


## Important observation

The supplied fingerprint files have the same row count as the benchmark:

- Morgan: **1349 × 2048**
- RDKit: **1349 × 200**

Therefore, row order must be preserved when joining fingerprints to the CSV.

We will verify this against independently generated RDKit fingerprints before using the supplied arrays as our baseline features.


In [9]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# Create Morgan fingerprint generator
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def morgan_fp_array(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros((2048,), dtype=np.int8)

    # Convert RDKit fingerprint to NumPy array
    from rdkit import DataStructs
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [10]:
# Compare all rows with the supplied Morgan fingerprint.
# This can take a little time.

generated = np.vstack([
    morgan_fp_array(s)
    for s in df["smiles"]
])

print("Generated shape:", generated.shape)
print("Supplied shape:", morgan.shape)

same = np.array_equal(generated, morgan)
print("Exact array equality:", same)

if not same:
    differing_rows = np.where(np.any(generated != morgan, axis=1))[0]
    print("Rows differing:", len(differing_rows))
    print("First differing row:", differing_rows[:10])


[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors
[16:03:10] WARNING: not removing hydrogen atom without neighbors


Generated shape: (1349, 2048)
Supplied shape: (1349, 2048)
Exact array equality: True


[16:03:11] WARNING: not removing hydrogen atom without neighbors
[16:03:11] WARNING: not removing hydrogen atom without neighbors


# First baseline: Liver + Binary + Morgan + Random Forest

This follows the UniTox reference configuration:

- Target: `liver_toxicity_binary_rating_0_1`
- Features: Morgan fingerprint
- Bits: 2048
- Radius: 2
- Random Forest: 500 trees
- Seed: 0
- CV: 10 folds
- Metric: ROC-AUC

The published repository result we will use as our reference is approximately:

**ROC-AUC = 0.7631 ± 0.0321**

Our reproduction does not need to match every fold bit-for-bit unless the original split implementation is identical; it should be reasonably close before we proceed.


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

target = "liver_toxicity_binary_rating_0_1"

X = morgan
y = df[target].astype(int).to_numpy()

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    n_jobs=-1
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Fold AUCs:")
print(scores)
print("\nMean AUC:", scores.mean())
print("Std AUC:", scores.std())


Fold AUCs:
[0.69200177 0.76347768 0.73840035 0.82310589 0.78511298 0.78001772
 0.76517501 0.83340718 0.79209127 0.72557826]

Mean AUC: 0.7698368125012359
Std AUC: 0.040961683135931265


# Reference comparison

UniTox repository reference for:

`Random Forest + Morgan + Binary + 10-fold CV + Liver`

**0.7630554777 ± 0.0320785795**

If our result differs noticeably, do **not** modify the model to force agreement. Instead, we will inspect the original split implementation and data preparation.

That reproducibility check comes before our proposed reliability-aware model.


In [12]:
same = np.array_equal(generated, morgan)

print("Exact array equality:", same)

if not same:
    differing_rows = np.where(
        np.any(generated != morgan, axis=1)
    )[0]

    print("Rows differing:", len(differing_rows))
    print("First differing rows:", differing_rows[:20])

Exact array equality: True


In [13]:
confident_target = "liver_toxicity_confident_ternary_rating_0_1"

mask = df[confident_target].notna()

X_conf = morgan[mask]
y_conf = df.loc[mask, confident_target].astype(int).to_numpy()

print("Total samples:", len(df))
print("Confident samples:", len(y_conf))
print("Removed samples:", len(df) - len(y_conf))

print("\nClass distribution:")
print(pd.Series(y_conf).value_counts())

Total samples: 1349
Confident samples: 878
Removed samples: 471

Class distribution:
1    481
0    397
Name: count, dtype: int64


In [14]:
rf_conf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    n_jobs=-1
)

cv_conf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

scores_conf = cross_val_score(
    rf_conf,
    X_conf,
    y_conf,
    cv=cv_conf,
    scoring="roc_auc",
    n_jobs=-1
)

print("Confident-label fold AUCs:")
print(scores_conf)

print("\nMean AUC:", scores_conf.mean())
print("Std AUC:", scores_conf.std())

Confident-label fold AUCs:
[0.82369792 0.8015625  0.8296875  0.82473958 0.76692708 0.8171875
 0.85130208 0.83150183 0.84161325 0.91773504]

Mean AUC: 0.8305954288766788
Std AUC: 0.03647372262303214


In [15]:
same = np.array_equal(generated, morgan)

print("Exact array equality:", same)

if not same:
    differing_rows = np.where(
        np.any(generated != morgan, axis=1)
    )[0]

    print("Rows differing:", len(differing_rows))
    print("First differing rows:", differing_rows[:20])

Exact array equality: True


# Next experiments

After this notebook passes:

1. Reproduce all 8 toxicity tasks.
2. Reproduce the confident-label version.
3. Compare binary vs confident-label performance.
4. Add scaffold-balanced evaluation.
5. Inspect DICTrank, DILIrank and DIRIL.
6. Generate leakage-safe out-of-fold predictions.
7. Test external agreement, model agreement and molecular similarity as reliability signals.
8. Train reliability-weighted models.
9. Run ablations.
10. Evaluate calibration and robustness.


In [16]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


# ============================================================
# 1. EXACT UniTox toxicity column prefixes
# ============================================================

toxicity_prefixes = {
    "Cardio": "cardio_toxicity",
    "Dermatological": "dermatological_toxicity",
    "Hematological": "hematological",
    "Infertility": "infertility",
    "Liver": "liver_toxicity",
    "Ototoxicity": "ototoxicity",
    "Pulmonary": "pulmonary_toxicity",
    "Renal": "renal_toxicity"
}


# ============================================================
# 2. Common RF + CV configuration
# ============================================================

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

rf_params = {
    "n_estimators": 500,
    "random_state": 0,
    "n_jobs": -1
}


# ============================================================
# 3. Evaluation function
# ============================================================

def evaluate_target(target, X, y, label_type):

    rf = RandomForestClassifier(**rf_params)

    scores = cross_val_score(
        rf,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return {
        "target": target,
        "label_type": label_type,
        "n_samples": len(y),
        "positive": int(np.sum(y == 1)),
        "negative": int(np.sum(y == 0)),
        "mean_auc": scores.mean(),
        "std_auc": scores.std(),
        "fold_scores": scores
    }


# ============================================================
# 4. ALL BINARY LABELS
# ============================================================

binary_results = []

print("=" * 70)
print("ALL BINARY LABEL EXPERIMENTS")
print("=" * 70)

for toxicity_name, prefix in toxicity_prefixes.items():

    target = f"{prefix}_binary_rating_0_1"

    # Check that the column actually exists
    if target not in df.columns:
        print(f"\nWARNING: Missing column: {target}")
        continue

    y = df[target].astype(int).to_numpy()

    print(f"\nRunning: {toxicity_name}")
    print(f"Target: {target}")
    print(f"Samples: {len(y)}")
    print(f"Class distribution: {np.bincount(y)}")

    result = evaluate_target(
        target=target,
        X=morgan,
        y=y,
        label_type="All Binary"
    )

    binary_results.append(result)

    print(
        f"AUC = {result['mean_auc']:.4f} "
        f"+/- {result['std_auc']:.4f}"
    )


# ============================================================
# 5. CONFIDENT LABELS ONLY
# ============================================================

confident_results = []

print("\n")
print("=" * 70)
print("CONFIDENT LABEL EXPERIMENTS")
print("=" * 70)

for toxicity_name, prefix in toxicity_prefixes.items():

    target = f"{prefix}_confident_ternary_rating_0_1"

    if target not in df.columns:
        print(f"\nWARNING: Missing column: {target}")
        continue

    mask = df[target].notna()

    X_conf = morgan[mask]

    y_conf = (
        df.loc[mask, target]
        .astype(int)
        .to_numpy()
    )

    print(f"\nRunning: {toxicity_name}")
    print(f"Target: {target}")
    print(f"Confident samples: {len(y_conf)}")
    print(f"Removed: {len(df) - len(y_conf)}")
    print(f"Class distribution: {np.bincount(y_conf)}")

    result = evaluate_target(
        target=target,
        X=X_conf,
        y=y_conf,
        label_type="Confident Only"
    )

    confident_results.append(result)

    print(
        f"AUC = {result['mean_auc']:.4f} "
        f"+/- {result['std_auc']:.4f}"
    )


# ============================================================
# 6. CREATE SUMMARY DATAFRAMES
# ============================================================

binary_summary = pd.DataFrame([
    {
        "Toxicity": toxicity_name,
        "Samples": r["n_samples"],
        "Positive": r["positive"],
        "Negative": r["negative"],
        "Mean AUC": r["mean_auc"],
        "Std AUC": r["std_auc"]
    }
    for toxicity_name, r in zip(
        [x for x in toxicity_prefixes if
         f"{toxicity_prefixes[x]}_binary_rating_0_1" in df.columns],
        binary_results
    )
])


confident_summary = pd.DataFrame([
    {
        "Toxicity": toxicity_name,
        "Samples": r["n_samples"],
        "Positive": r["positive"],
        "Negative": r["negative"],
        "Mean AUC": r["mean_auc"],
        "Std AUC": r["std_auc"]
    }
    for toxicity_name, r in zip(
        [x for x in toxicity_prefixes if
         f"{toxicity_prefixes[x]}_confident_ternary_rating_0_1"
         in df.columns],
        confident_results
    )
])


# ============================================================
# 7. MERGE RESULTS
# ============================================================

comparison = binary_summary[
    ["Toxicity", "Samples", "Mean AUC", "Std AUC"]
].copy()

comparison = comparison.rename(columns={
    "Samples": "All Samples",
    "Mean AUC": "All Binary AUC",
    "Std AUC": "All Binary Std"
})

comparison["Confident Samples"] = confident_summary["Samples"]

comparison["Confident AUC"] = confident_summary["Mean AUC"]

comparison["Confident Std"] = confident_summary["Std AUC"]

comparison["AUC Difference"] = (
    comparison["Confident AUC"]
    - comparison["All Binary AUC"]
)

comparison["Removed %"] = (
    (comparison["All Samples"]
     - comparison["Confident Samples"])
    / comparison["All Samples"]
    * 100
)


# ============================================================
# 8. DISPLAY
# ============================================================

print("\n")
print("=" * 100)
print("ALL LABELS vs CONFIDENT LABELS")
print("=" * 100)

display(
    comparison.round(4)
)


# ============================================================
# 9. SAVE
# ============================================================

comparison.to_csv(
    "UniTox_RF_Morgan_All_vs_Confident.csv",
    index=False
)

print("\nResults saved successfully.")

ALL BINARY LABEL EXPERIMENTS

Running: Cardio
Target: cardio_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [594 755]
AUC = 0.7701 +/- 0.0310

Running: Dermatological
Target: dermatological_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [ 150 1199]
AUC = 0.7032 +/- 0.0777

Running: Hematological
Target: hematological_binary_rating_0_1
Samples: 1349
Class distribution: [473 876]
AUC = 0.7788 +/- 0.0383

Running: Infertility
Target: infertility_binary_rating_0_1
Samples: 1349
Class distribution: [846 503]
AUC = 0.7837 +/- 0.0369

Running: Liver
Target: liver_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [613 736]
AUC = 0.7698 +/- 0.0410

Running: Ototoxicity
Target: ototoxicity_binary_rating_0_1
Samples: 1349
Class distribution: [1228  121]
AUC = 0.6790 +/- 0.0963

Running: Pulmonary
Target: pulmonary_toxicity_binary_rating_0_1
Samples: 1349
Class distribution: [851 498]
AUC = 0.7367 +/- 0.0523

Running: Renal
Target: renal_toxicity_binary_ratin

,Toxicity,All Samples,All Binary AUC,All Binary Std,Confident Samples,Confident AUC,Confident Std,AUC Difference,Removed %
0,Cardio,1349,0.7701,0.0310,901,0.8030,0.0570,0.0329,33.2098
1,Dermatological,1349,0.7032,0.0777,955,0.7321,0.0635,0.0289,29.2068
2,Hematological,1349,0.7788,0.0383,1075,0.8221,0.0297,0.0433,20.3113
3,Infertility,1349,0.7837,0.0369,1021,0.8274,0.0321,0.0437,24.3143
4,Liver,1349,0.7698,0.0410,878,0.8306,0.0365,0.0608,34.9148
5,Ototoxicity,1349,0.6790,0.0963,1183,0.7741,0.1300,0.0951,12.3054
6,Pulmonary,1349,0.7367,0.0523,997,0.7603,0.0484,0.0236,26.0934
7,Renal,1349,0.7616,0.0265,948,0.8037,0.0296,0.0421,29.7257



Results saved successfully.


In [17]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd
import numpy as np


def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol,
        includeChirality=False
    )


df["scaffold"] = df["smiles"].apply(get_scaffold)

print("Total molecules:", len(df))
print("Valid scaffolds:", df["scaffold"].notna().sum())
print("Unique scaffolds:", df["scaffold"].nunique())

print("\nTop 20 scaffold frequencies:")
print(
    df["scaffold"]
    .value_counts()
    .head(20)
)

print("\nMolecules without scaffold:")
print(df["scaffold"].isna().sum())

[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors
[16:06:40] WARNING: not removing hydrogen atom without neighbors


Total molecules: 1349
Valid scaffolds: 1349
Unique scaffolds: 802

Top 20 scaffold frequencies:
scaffold
                                      122
c1ccccc1                               77
O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21        15
O=C1C=C2CCC3C4CCCC4CCC3C2CC1           12
O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1     10
c1ccncc1                                8
c1ccccc1.c1ccccc1                       8
c1ccc(Cc2ccccc2)cc1                     8
c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1       8
c1c[nH]cn1                              7
O=C(c1ccccc1)c1ccccc1                   7
O=c1[nH]cnc2[nH]cnc12                   6
c1ncc2ncn(C3CCCO3)c2n1                  6
O=C1CN=C(c2ccccc2)c2ccccc2N1            6
c1ccc(CC2CCCCC2)cc1                     6
c1ncc2nc[nH]c2n1                        5
O=C(Cc1ccccc1)NC1C(=O)N2CCSC12          5
C1CCC2C(C1)CCC1C3CCCC3CCC21             5
c1ccc(Oc2ccccc2)cc1                     5
O=C(Cc1ccccc1)NC1C(=O)N2C=CCSC12        5
Name: count, dtype: int64

Molecules without scaffold:


[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors


In [18]:
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# Scaffold groups
# ============================================================

scaffold_groups = defaultdict(list)

for idx, scaffold in enumerate(df["scaffold"]):
    scaffold_groups[scaffold].append(idx)

# Convert to lists
scaffold_groups = {
    scaffold: np.array(indices)
    for scaffold, indices in scaffold_groups.items()
}

print("Number of scaffold groups:", len(scaffold_groups))

print("\nLargest scaffold groups:")
largest_groups = sorted(
    scaffold_groups.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

for i, (scaffold, indices) in enumerate(largest_groups, 1):
    display_scaffold = "<EMPTY>" if scaffold == "" else scaffold

    print(
        f"{i:2d}. "
        f"size={len(indices):3d} | "
        f"{display_scaffold}"
    )


# ============================================================
# Create scaffold-balanced folds
# ============================================================

n_folds = 10
rng = np.random.RandomState(0)

# Shuffle scaffold groups first so ties don't depend on
# dictionary ordering.
groups = list(scaffold_groups.items())
rng.shuffle(groups)

# Sort largest groups first
groups = sorted(
    groups,
    key=lambda x: len(x[1]),
    reverse=True
)

fold_indices = [[] for _ in range(n_folds)]
fold_sizes = np.zeros(n_folds, dtype=int)

# Greedy balancing:
# assign each new scaffold to the currently smallest fold
for scaffold, indices in groups:

    fold_id = np.argmin(fold_sizes)

    fold_indices[fold_id].extend(indices.tolist())
    fold_sizes[fold_id] += len(indices)


# Convert to numpy arrays
fold_indices = [
    np.array(sorted(indices), dtype=int)
    for indices in fold_indices
]


# ============================================================
# Basic fold information
# ============================================================

print("\n" + "=" * 70)
print("SCAFFOLD-BALANCED FOLD SIZES")
print("=" * 70)

for i, indices in enumerate(fold_indices):

    print(
        f"Fold {i + 1}: "
        f"{len(indices)} molecules | "
        f"{len(set(df.loc[indices, 'scaffold']))} scaffolds"
    )

print(
    "\nTotal assigned:",
    sum(len(x) for x in fold_indices)
)

print(
    "Original molecules:",
    len(df)
)


# ============================================================
# VERIFY every molecule appears exactly once
# ============================================================

all_test_indices = np.concatenate(fold_indices)

print(
    "\nUnique assigned molecules:",
    len(np.unique(all_test_indices))
)

print(
    "Expected:",
    len(df)
)


# ============================================================
# VERIFY NO SCAFFOLD LEAKAGE
# ============================================================

print("\n" + "=" * 70)
print("SCAFFOLD LEAKAGE CHECK")
print("=" * 70)

leakage_found = False

for test_fold in range(n_folds):

    test_idx = fold_indices[test_fold]

    train_idx = np.concatenate([
        fold_indices[j]
        for j in range(n_folds)
        if j != test_fold
    ])

    train_scaffolds = set(
        df.loc[train_idx, "scaffold"]
    )

    test_scaffolds = set(
        df.loc[test_idx, "scaffold"]
    )

    overlap = train_scaffolds.intersection(
        test_scaffolds
    )

    print(
        f"Fold {test_fold + 1}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"scaffold overlap={len(overlap)}"
    )

    if len(overlap) > 0:
        leakage_found = True


print("\nLeakage detected:", leakage_found)

Number of scaffold groups: 802

Largest scaffold groups:
 1. size=122 | <EMPTY>
 2. size= 77 | c1ccccc1
 3. size= 15 | O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21
 4. size= 12 | O=C1C=C2CCC3C4CCCC4CCC3C2CC1
 5. size= 10 | O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1
 6. size=  8 | c1ccncc1
 7. size=  8 | c1ccccc1.c1ccccc1
 8. size=  8 | c1ccc(Cc2ccccc2)cc1
 9. size=  8 | c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1
10. size=  7 | c1c[nH]cn1
11. size=  7 | O=C(c1ccccc1)c1ccccc1
12. size=  6 | O=c1[nH]cnc2[nH]cnc12
13. size=  6 | c1ncc2ncn(C3CCCO3)c2n1
14. size=  6 | O=C1CN=C(c2ccccc2)c2ccccc2N1
15. size=  6 | c1ccc(CC2CCCCC2)cc1
16. size=  5 | c1ncc2nc[nH]c2n1
17. size=  5 | O=C(Cc1ccccc1)NC1C(=O)N2CCSC12
18. size=  5 | C1CCC2C(C1)CCC1C3CCCC3CCC21
19. size=  5 | c1ccc(Oc2ccccc2)cc1
20. size=  5 | O=C(Cc1ccccc1)NC1C(=O)N2C=CCSC12

SCAFFOLD-BALANCED FOLD SIZES
Fold 1: 135 molecules | 14 scaffolds
Fold 2: 135 molecules | 59 scaffolds
Fold 3: 135 molecules | 90 scaffolds
Fold 4: 135 molecules | 91 scaffolds
Fold 5: 135 mol

In [19]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# ---------------------------------------------------------
# Generate Morgan fingerprints
# ---------------------------------------------------------
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

X_morgan = np.array([
    morgan_generator.GetFingerprintAsNumPy(Chem.MolFromSmiles(smiles))
    for smiles in df["smiles"]
])

print("Morgan fingerprint shape:", X_morgan.shape)

[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors


Morgan fingerprint shape: (1349, 2048)


[16:06:42] WARNING: not removing hydrogen atom without neighbors
[16:06:42] WARNING: not removing hydrogen atom without neighbors


In [20]:
# Show all columns containing "liver"
[c for c in df.columns if "liver" in c.lower()]

['liver_toxicity_reasoning',
 'liver_toxicity_ternary_rating',
 'liver_toxicity_binary_rating',
 'liver_toxicity_confident_ternary_rating_0_1',
 'liver_toxicity_binary_rating_0_1']

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# ---------------------------------------------------------
# Liver binary target
# ---------------------------------------------------------
target_col = "liver_toxicity_binary_rating_0_1"

X = X_morgan
y = df[target_col].values.astype(int)

print("Target:", target_col)
print("Class distribution:", np.bincount(y))

# ---------------------------------------------------------
# Scaffold-balanced 10-fold evaluation
# ---------------------------------------------------------
fold_aucs = []

for fold in range(10):

    test_idx = fold_indices[fold]

    train_idx = np.concatenate([
        fold_indices[j]
        for j in range(10)
        if j != fold
    ])

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = RandomForestClassifier(
        n_estimators=500,
        random_state=0,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    fold_aucs.append(auc)

    print(
        f"Fold {fold+1}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"AUC={auc:.4f}"
    )

print("\n" + "="*60)
print("SCAFFOLD-BALANCED RF + MORGAN — LIVER")
print("="*60)

print("Fold AUCs:", [round(x, 4) for x in fold_aucs])
print(f"Mean AUC: {np.mean(fold_aucs):.4f}")
print(f"Std AUC : {np.std(fold_aucs):.4f}")

Target: liver_toxicity_binary_rating_0_1
Class distribution: [613 736]
Fold 1: train=1214, test=135, AUC=0.5541
Fold 2: train=1214, test=135, AUC=0.6697
Fold 3: train=1214, test=135, AUC=0.7705
Fold 4: train=1214, test=135, AUC=0.7004
Fold 5: train=1214, test=135, AUC=0.7666
Fold 6: train=1214, test=135, AUC=0.7198
Fold 7: train=1214, test=135, AUC=0.7582
Fold 8: train=1214, test=135, AUC=0.6447
Fold 9: train=1214, test=135, AUC=0.7627
Fold 10: train=1215, test=134, AUC=0.6242

SCAFFOLD-BALANCED RF + MORGAN — LIVER
Fold AUCs: [0.5541, 0.6697, 0.7705, 0.7004, 0.7666, 0.7198, 0.7582, 0.6447, 0.7627, 0.6242]
Mean AUC: 0.6971
Std AUC : 0.0693


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# =========================================================
# EXACT TARGET COLUMNS FROM YOUR DATAFRAME
# =========================================================
toxicity_targets = {
    "Cardiotoxicity": "cardio_toxicity_binary_rating_0_1",
    "Dermatological": "dermatological_toxicity_binary_rating_0_1",
    "Hematological": "hematological_binary_rating_0_1",
    "Infertility": "infertility_binary_rating_0_1",
    "Liver": "liver_toxicity_binary_rating_0_1",
    "Ototoxicity": "ototoxicity_binary_rating_0_1",
    "Pulmonary": "pulmonary_toxicity_binary_rating_0_1",
    "Renal": "renal_toxicity_binary_rating_0_1"
}

results = []

# =========================================================
# SCAFFOLD-BALANCED 10-FOLD RF + MORGAN
# =========================================================

for toxicity, target_col in toxicity_targets.items():

    y = df[target_col].values.astype(int)

    fold_aucs = []

    for fold in range(10):

        test_idx = fold_indices[fold]

        train_idx = np.concatenate([
            fold_indices[j]
            for j in range(10)
            if j != fold
        ])

        X_train = X_morgan[train_idx]
        X_test = X_morgan[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model = RandomForestClassifier(
            n_estimators=500,
            random_state=0,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y_test, y_prob)
        fold_aucs.append(auc)

    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)

    results.append({
        "Toxicity": toxicity,
        "Mean AUC": mean_auc,
        "Std AUC": std_auc
    })

    print(
        f"{toxicity:16s} | "
        f"AUC = {mean_auc:.4f} ± {std_auc:.4f}"
    )

# =========================================================
# FINAL RESULTS TABLE
# =========================================================

scaffold_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SCAFFOLD-BALANCED RF + MORGAN — ALL 8 TOXICITY TASKS")
print("=" * 70)

display(
    scaffold_results.style.format({
        "Mean AUC": "{:.4f}",
        "Std AUC": "{:.4f}"
    })
)

Cardiotoxicity   | AUC = 0.6934 ± 0.0846
Dermatological   | AUC = 0.5804 ± 0.1105
Hematological    | AUC = 0.7096 ± 0.0844
Infertility      | AUC = 0.6719 ± 0.0799
Liver            | AUC = 0.6971 ± 0.0693
Ototoxicity      | AUC = 0.6162 ± 0.1028
Pulmonary        | AUC = 0.6679 ± 0.0781
Renal            | AUC = 0.7032 ± 0.0828

SCAFFOLD-BALANCED RF + MORGAN — ALL 8 TOXICITY TASKS


,Toxicity,Mean AUC,Std AUC
0,Cardiotoxicity,0.6934,0.0846
1,Dermatological,0.5804,0.1105
2,Hematological,0.7096,0.0844
3,Infertility,0.6719,0.0799
4,Liver,0.6971,0.0693
5,Ototoxicity,0.6162,0.1028
6,Pulmonary,0.6679,0.0781
7,Renal,0.7032,0.0828


In [23]:
import pandas as pd
from pathlib import Path

# =========================================================
# FIND THE THREE EXTERNAL VALIDATION FILES
# =========================================================

BASE_DIR = Path.cwd()

print("Notebook working directory:")
print(BASE_DIR)

# ---------------------------------------------------------
# File paths
# ---------------------------------------------------------

dictrank_path = BASE_DIR / "dictrank_raw.csv"
dilirank_path = BASE_DIR / "dilirank_raw.xlsx"
diril_path = BASE_DIR / "A. DIRIL (317)-Table 1.csv"

# Check that all files exist
print("\nFile check:")
print("DICTrank:", dictrank_path.exists(), dictrank_path)
print("DILIrank:", dilirank_path.exists(), dilirank_path)
print("DIRIL:", diril_path.exists(), diril_path)

# =========================================================
# LOAD DATASETS
# =========================================================

if not dictrank_path.exists():
    raise FileNotFoundError(
        f"DICTrank not found: {dictrank_path}\n"
        "Download dictrank_raw.csv and place it beside this notebook."
    )

if not dilirank_path.exists():
    raise FileNotFoundError(
        f"DILIrank not found: {dilirank_path}\n"
        "Download dilirank_raw.xlsx and place it beside this notebook."
    )

if not diril_path.exists():
    raise FileNotFoundError(
        f"DIRIL not found: {diril_path}\n"
        "Download A. DIRIL (317)-Table 1.csv and place it beside this notebook."
    )

# =========================================================
# READ FILES
# =========================================================

dictrank = pd.read_csv(dictrank_path)

dilirank = pd.read_excel(
    dilirank_path,
    sheet_name="supporting table 1"
)

diril = pd.read_csv(diril_path)

# =========================================================
# AUDIT
# =========================================================

print("\n" + "=" * 70)
print("EXTERNAL DATASET AUDIT")
print("=" * 70)

print("\nDICTrank")
print("-" * 70)
print("Shape:", dictrank.shape)
print("Columns:")
print(dictrank.columns.tolist())

print("\nDILIrank")
print("-" * 70)
print("Shape:", dilirank.shape)
print("Columns:")
print(dilirank.columns.tolist())

print("\nDIRIL")
print("-" * 70)
print("Shape:", diril.shape)
print("Columns:")
print(diril.columns.tolist())

Notebook working directory:
C:\Users\DEEPAJYOTI

File check:
DICTrank: True C:\Users\DEEPAJYOTI\dictrank_raw.csv
DILIrank: True C:\Users\DEEPAJYOTI\dilirank_raw.xlsx
DIRIL: True C:\Users\DEEPAJYOTI\A. DIRIL (317)-Table 1.csv

EXTERNAL DATASET AUDIT

DICTrank
----------------------------------------------------------------------
Shape: (1318, 10)
Columns:
['Unnamed: 0', 'Trade Name', 'Generic/Proper Name(s)', 'Active Ingredient(s)', 'Cardiotoxicity', 'Label Section ', 'DICT _ Concern', 'Keywords', 'DIC Severity Level', 'split']

DILIrank
----------------------------------------------------------------------
Shape: (1036, 16)
Columns:
['LTKBID', 'PubChem_CID', 'LabelCompoundName', 'ApprovalYear', 'DILIConcern', 'SeverityClass', 'LabelSection', 'AdjudicatedDILI', 'vDILIConcern', 'Greene_Annotation', 'Sakatis_Annotation', 'Xu_Annotation', 'Zhu_Annotation', 'SMILES', 'URL (accessed on 1/21/2016)', 'VER_DISP (1=LTKB-BD, 2=others)']

DIRIL
-----------------------------------------------------

In [24]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [25]:
# =========================================================
# DILIrank LABEL AUDIT
# =========================================================

print("=" * 70)
print("DILIrank — DILIConcern")
print("=" * 70)

print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — AdjudicatedDILI")
print("=" * 70)

print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — SeverityClass")
print("=" * 70)

print(dilirank["SeverityClass"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("Examples")
print("=" * 70)

display(
    dilirank[
        [
            "LabelCompoundName",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI",
            "SMILES"
        ]
    ].head(20)
)

DILIrank — DILIConcern
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

DILIrank — AdjudicatedDILI
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

DILIrank — SeverityClass
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4
Name: count, dtype: int64

Examples


,LabelCompoundName,DILIConcern,SeverityClass,AdjudicatedDILI,SMILES
0,ceftriaxone,Less-DILI-Concern,4,Yes,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...
1,trimethoprim,Less-DILI-Concern,4,Yes,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
2,tetracycline,Less-DILI-Concern,2,Yes,CC1(C2CC3C(C(=O)/C(=C(\N)/O)/C(=O)C3(C(=O)C2=C...
3,dapsone,Less-DILI-Concern,3,Yes,C1=CC(=CC=C1N)S(=O)(=O)C2=CC=C(C=C2)N
4,pyrazinamide,Less-DILI-Concern,3,Yes,C1=CN=C(C=N1)C(=O)N
5,fenofibrate,Less-DILI-Concern,3,Yes,CC(C)OC(=O)C(C)(C)OC1=CC=C(C=C1)C(=O)C2=CC=C(C...
6,cyclophosphamide,Less-DILI-Concern,5,Yes,C1CNP(=O)(OC1)N(CCCl)CCCl
7,chlorpromazine,Less-DILI-Concern,2,Yes,CN(C)CCCN1C2=CC=CC=C2SC3=C1C=C(C=C3)Cl
8,doxorubicin,Less-DILI-Concern,3,Yes,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...
9,clindamycin,Less-DILI-Concern,3,Yes,CCC[C@@H]1C[C@H](N(C1)C)C(=O)NC([C@@H]2[C@@H](...


In [26]:
# =========================================================
# UNITOX LIVER LABEL AUDIT
# =========================================================

print("=" * 70)
print("UniTox liver binary labels")
print("=" * 70)

print(
    df["liver_toxicity_binary_rating_0_1"]
    .value_counts(dropna=False)
)

print("\nRelevant UniTox columns:")

for c in df.columns:
    if "liver" in c.lower():
        print(" -", c)

UniTox liver binary labels
liver_toxicity_binary_rating_0_1
1    736
0    613
Name: count, dtype: int64

Relevant UniTox columns:
 - liver_toxicity_reasoning
 - liver_toxicity_ternary_rating
 - liver_toxicity_binary_rating
 - liver_toxicity_confident_ternary_rating_0_1
 - liver_toxicity_binary_rating_0_1


In [27]:
print("\nPotential drug identity columns:")

for c in df.columns:
    c_lower = c.lower()

    if any(
        keyword in c_lower
        for keyword in [
            "drug",
            "name",
            "compound",
            "smiles",
            "pubchem",
            "inchikey",
            "drugbank"
        ]
    ):
        print(" -", c)


Potential drug identity columns:
 - generic_name
 - smiles
 - all_smiles


In [28]:
# =========================================================
# DILIrank LABEL AUDIT
# =========================================================

print("=" * 70)
print("DILIrank — DILIConcern")
print("=" * 70)
print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — AdjudicatedDILI")
print("=" * 70)
print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — SeverityClass")
print("=" * 70)
print(dilirank["SeverityClass"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DILIrank — Example records")
print("=" * 70)

display(
    dilirank[
        [
            "LabelCompoundName",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI",
            "SMILES"
        ]
    ].head(20)
)

DILIrank — DILIConcern
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

DILIrank — AdjudicatedDILI
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

DILIrank — SeverityClass
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4
Name: count, dtype: int64

DILIrank — Example records


,LabelCompoundName,DILIConcern,SeverityClass,AdjudicatedDILI,SMILES
0,ceftriaxone,Less-DILI-Concern,4,Yes,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...
1,trimethoprim,Less-DILI-Concern,4,Yes,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
2,tetracycline,Less-DILI-Concern,2,Yes,CC1(C2CC3C(C(=O)/C(=C(\N)/O)/C(=O)C3(C(=O)C2=C...
3,dapsone,Less-DILI-Concern,3,Yes,C1=CC(=CC=C1N)S(=O)(=O)C2=CC=C(C=C2)N
4,pyrazinamide,Less-DILI-Concern,3,Yes,C1=CN=C(C=N1)C(=O)N
5,fenofibrate,Less-DILI-Concern,3,Yes,CC(C)OC(=O)C(C)(C)OC1=CC=C(C=C1)C(=O)C2=CC=C(C...
6,cyclophosphamide,Less-DILI-Concern,5,Yes,C1CNP(=O)(OC1)N(CCCl)CCCl
7,chlorpromazine,Less-DILI-Concern,2,Yes,CN(C)CCCN1C2=CC=CC=C2SC3=C1C=C(C=C3)Cl
8,doxorubicin,Less-DILI-Concern,3,Yes,C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...
9,clindamycin,Less-DILI-Concern,3,Yes,CCC[C@@H]1C[C@H](N(C1)C)C(=O)NC([C@@H]2[C@@H](...


In [29]:
# =========================================================
# 1. DILIrank LABEL DISTRIBUTIONS
# =========================================================

for col in ["DILIConcern", "SeverityClass", "AdjudicatedDILI"]:
    print("\n" + "=" * 60)
    print(f"{col} — value counts")
    print("=" * 60)
    print(dilirank[col].value_counts(dropna=False).to_string())


# =========================================================
# 2. CROSS-TABULATE DILI CONCERN AND ADJUDICATION
# =========================================================

print("\nDILIConcern vs AdjudicatedDILI")

display(
    pd.crosstab(
        dilirank["DILIConcern"],
        dilirank["AdjudicatedDILI"],
        margins=True,
        dropna=False
    )
)


# =========================================================
# 3. UNITOX DRUG IDENTITY QUALITY
# =========================================================

print("\n" + "=" * 60)
print("UniTox identity audit")
print("=" * 60)

print("Rows:", len(df))
print("Unique generic names:", df["generic_name"].nunique())
print("Missing generic names:", df["generic_name"].isna().sum())
print("Missing SMILES:", df["smiles"].isna().sum())
print("Missing all_smiles:", df["all_smiles"].isna().sum())

display(
    df[
        [
            "generic_name",
            "smiles",
            "liver_toxicity_binary_rating_0_1"
        ]
    ].head(15)
)


DILIConcern — value counts
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1

SeverityClass — value counts
SeverityClass
0    338
3    309
8    159
5     62
7     60
4     53
2     46
6      5
1      4

AdjudicatedDILI — value counts
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12

DILIConcern vs AdjudicatedDILI


AdjudicatedDILI,Yes,Yes(literature),NaN,All
DILIConcern,,,,
Less-DILI-Concern,249,2,237,488
Most-DILI-Concern,124,9,75,208
Most-DILI-concern,0,0,1,1
No-DILI-Concern,26,1,294,321
No-DILI-concern,0,0,18,18
All,399,12,625,1036



UniTox identity audit
Rows: 1349
Unique generic names: 1349
Missing generic names: 0
Missing SMILES: 0
Missing all_smiles: 0


,generic_name,smiles,liver_toxicity_binary_rating_0_1
0,68GA-DOTATATE,CC(O)C(NC(=O)C1CSSCC(NC(=O)C(Cc2ccccc2)NC(=O)C...,0
1,ABACAVIR,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,1
2,ABALOPARATIDE,CCC(C)C(NC(=O)C(CO)NC(=O)C(CCCCN)NC(=O)CNC(=O)...,0
3,ABEMACICLIB,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C(C...,1
4,ABIRATERONE ACETATE,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,1
5,ACALABRUTINIB,CC#CC(=O)N1CCCC1c1nc(-c2ccc(C(=O)Nc3ccccn3)cc2...,1
6,ACARBOSE,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,1
7,ACEBUTOLOL HYDROCHLORIDE,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1.Cl,1
8,ACETAZOLAMIDE,CC(=O)Nc1nnc(S(N)(=O)=O)s1,1
9,ACETOHYDROXAMIC ACID,CC(=O)NO,0


In [30]:
import pandas as pd
import re
import numpy as np

# =========================================================
# NORMALIZE DRUG NAMES
# =========================================================

def normalize_drug_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Normalize punctuation and whitespace
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else None


# Work on copies to preserve the original datasets
unitox_match = df[
    ["generic_name", "smiles", "liver_toxicity_binary_rating_0_1"]
].copy()

dilirank_match = dilirank[
    ["LabelCompoundName", "SMILES", "DILIConcern",
     "SeverityClass", "AdjudicatedDILI"]
].copy()

unitox_match["normalized_name"] = (
    unitox_match["generic_name"].apply(normalize_drug_name)
)

dilirank_match["normalized_name"] = (
    dilirank_match["LabelCompoundName"].apply(normalize_drug_name)
)

# =========================================================
# CHECK DUPLICATE KEYS BEFORE MERGING
# =========================================================

print("UniTox rows:", len(unitox_match))
print("DILIrank rows:", len(dilirank_match))

print(
    "UniTox duplicate normalized names:",
    unitox_match["normalized_name"].duplicated().sum()
)

print(
    "DILIrank duplicate normalized names:",
    dilirank_match["normalized_name"].duplicated().sum()
)

# Keep only records with usable normalized names
unitox_valid_names = unitox_match.dropna(
    subset=["normalized_name"]
).copy()

dilirank_valid_names = dilirank_match.dropna(
    subset=["normalized_name"]
).copy()

# Identify names occurring in both datasets
unitox_names = set(unitox_valid_names["normalized_name"])
dilirank_names = set(dilirank_valid_names["normalized_name"])

common_names = unitox_names.intersection(dilirank_names)

print("\nUnique normalized UniTox names:", len(unitox_names))
print("Unique normalized DILIrank names:", len(dilirank_names))
print("Common normalized names:", len(common_names))

# =========================================================
# SHOW CANDIDATE MATCHES
# =========================================================

candidate_matches = unitox_valid_names[
    unitox_valid_names["normalized_name"].isin(common_names)
].merge(
    dilirank_valid_names,
    on="normalized_name",
    how="inner",
    suffixes=("_unitox", "_dilirank"),
    validate="many_to_many"
)

print("\nCandidate matched rows:", len(candidate_matches))
print(
    "Unique UniTox drug names matched:",
    candidate_matches["normalized_name"].nunique()
)

display(
    candidate_matches[
        [
            "generic_name",
            "LabelCompoundName",
            "normalized_name",
            "smiles",
            "SMILES",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "SeverityClass",
            "AdjudicatedDILI"
        ]
    ].head(30)
)

UniTox rows: 1349
DILIrank rows: 1036
UniTox duplicate normalized names: 1
DILIrank duplicate normalized names: 0

Unique normalized UniTox names: 1348
Unique normalized DILIrank names: 1036
Common normalized names: 497

Candidate matched rows: 497
Unique UniTox drug names matched: 497


,generic_name,LabelCompoundName,normalized_name,smiles,SMILES,liver_toxicity_binary_rating_0_1,DILIConcern,SeverityClass,AdjudicatedDILI
0,ABACAVIR,abacavir,abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,C1CC1NC2=NC(=NC3=C2N=CN3C4C[C@H](C=C4)CO)N,1,Most-DILI-Concern,8,Yes
1,ACARBOSE,acarbose,acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O[C@@H]2[C...,1,Most-DILI-Concern,8,Yes
2,ACETAZOLAMIDE,acetazolamide,acetazolamide,CC(=O)Nc1nnc(S(N)(=O)=O)s1,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,1,Most-DILI-Concern,8,Yes
3,ACETOHYDROXAMIC ACID,acetohydroxamic acid,acetohydroxamic acid,CC(=O)NO,CC(=O)NO,0,Less-DILI-Concern,3,NaN
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,acetylcholine chloride,CC(=O)OCC[N+](C)(C)C.[Cl-],CC(=O)OCC[N+](C)(C)C,0,No-DILI-concern,0,NaN
5,ACITRETIN,acitretin,acitretin,COc1cc(C)c(C=CC(C)=CC=CC(C)=CC(=O)O)c(C)c1C,CC1=CC(=C(C(=C1/C=C/C(=C/C=C/C(=C\C(=O)O)/C)/C...,1,Most-DILI-Concern,5,Yes
6,ADENOSINE,adenosine,adenosine,Nc1ncnc2c1ncn2C1OC(CO)C(O)C1O,C1=NC2=C(C(=N1)N)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,0,No-DILI-Concern,0,NaN
7,ALBENDAZOLE,albendazole,albendazole,CCCSc1ccc2nc(NC(=O)OC)[nH]c2c1,CCCSC1=CC2=C(C=C1)N=C(N2)NC(=O)OC,1,Most-DILI-Concern,7,Yes
8,ALLOPURINOL,allopurinol,allopurinol,O=c1[nH]cnc2[nH]ncc12,C1=C2C(=NC=NC2=O)NN1,1,Most-DILI-Concern,8,Yes
9,ALMOTRIPTAN,almotriptan,almotriptan,CN(C)CCc1c[nH]c2ccc(CS(=O)(=O)N3CCCC3)cc12,CN(C)CCC1=CNC2=C1C=C(C=C2)CS(=O)(=O)N3CCCC3,0,No-DILI-Concern,0,NaN


In [31]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import inchi

# =========================================================
# CANONICALIZE SMILES
# =========================================================

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return None

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return None

        return inchi.MolToInchiKey(mol)

    except Exception:
        return None


# Use the existing candidate_matches dataframe
identity_check = candidate_matches.copy()

identity_check["unitox_inchikey"] = (
    identity_check["smiles"].apply(smiles_to_inchikey)
)

identity_check["dilirank_inchikey"] = (
    identity_check["SMILES"].apply(smiles_to_inchikey)
)

# =========================================================
# CLASSIFY STRUCTURE MATCHES
# =========================================================

def compare_structures(row):
    key1 = row["unitox_inchikey"]
    key2 = row["dilirank_inchikey"]

    if pd.isna(key1) or pd.isna(key2):
        return "Structure unavailable/invalid"

    if key1 == key2:
        return "Exact InChIKey match"

    return "Name match, structure differs"


identity_check["structure_status"] = identity_check.apply(
    compare_structures,
    axis=1
)

print("=" * 70)
print("MOLECULAR IDENTITY AUDIT")
print("=" * 70)

print(identity_check["structure_status"].value_counts(dropna=False))

print("\nExamples where names match but structures differ:")

display(
    identity_check[
        identity_check["structure_status"]
        == "Name match, structure differs"
    ][
        [
            "generic_name",
            "LabelCompoundName",
            "unitox_inchikey",
            "dilirank_inchikey",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI"
        ]
    ].head(20)
)

print("\nExamples with exact structure matches:")

display(
    identity_check[
        identity_check["structure_status"] == "Exact InChIKey match"
    ][
        [
            "generic_name",
            "LabelCompoundName",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI"
        ]
    ].head(20)
)

MOLECULAR IDENTITY AUDIT
structure_status
Exact InChIKey match             253
Name match, structure differs    238
Structure unavailable/invalid      6
Name: count, dtype: int64

Examples where names match but structures differ:


,generic_name,LabelCompoundName,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,abacavir,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,acarbose,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,JUGOREOARAHOCO-UHFFFAOYSA-M,OIPILFWXSMYKGL-UHFFFAOYSA-N,0,No-DILI-concern,NaN
5,ACITRETIN,acitretin,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,adenosine,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,amoxicillin,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,amphotericin b,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,ampicillin,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,anidulafungin,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,aprepitant,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN



Examples with exact structure matches:


,generic_name,LabelCompoundName,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
2,ACETAZOLAMIDE,acetazolamide,1,Most-DILI-Concern,Yes
3,ACETOHYDROXAMIC ACID,acetohydroxamic acid,0,Less-DILI-Concern,NaN
7,ALBENDAZOLE,albendazole,1,Most-DILI-Concern,Yes
8,ALLOPURINOL,allopurinol,1,Most-DILI-Concern,Yes
9,ALMOTRIPTAN,almotriptan,0,No-DILI-Concern,NaN
10,AMANTADINE,amantadine,0,Less-DILI-Concern,NaN
11,AMBRISENTAN,ambrisentan,0,Less-DILI-Concern,NaN
12,AMIFOSTINE,amifostine,0,No-DILI-Concern,NaN
13,AMINOCAPROIC ACID,aminocaproic acid,1,No-DILI-Concern,NaN
14,AMINOPHYLLINE,aminophylline,1,No-DILI-Concern,NaN


In [32]:
print("=" * 60)
print("MOLECULAR IDENTITY AUDIT")
print("=" * 60)

print(identity_check["structure_status"].value_counts(dropna=False))

print("\nTotal candidate rows:", len(identity_check))
print("Unique matched drug names:",
      identity_check["normalized_name"].nunique())

print("\nMissing UniTox InChIKeys:",
      identity_check["unitox_inchikey"].isna().sum())

print("Missing DILIrank InChIKeys:",
      identity_check["dilirank_inchikey"].isna().sum())

MOLECULAR IDENTITY AUDIT
structure_status
Exact InChIKey match             253
Name match, structure differs    238
Structure unavailable/invalid      6
Name: count, dtype: int64

Total candidate rows: 497
Unique matched drug names: 497

Missing UniTox InChIKeys: 0
Missing DILIrank InChIKeys: 6


In [33]:
print("DILI concern categories:")
print(dilirank["DILIConcern"].value_counts(dropna=False))

print("\nAdjudication categories:")
print(dilirank["AdjudicatedDILI"].value_counts(dropna=False))

print("\nCross-tabulation:")
display(
    pd.crosstab(
        dilirank["DILIConcern"],
        dilirank["AdjudicatedDILI"],
        margins=True,
        dropna=False
    )
)

DILI concern categories:
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

Adjudication categories:
AdjudicatedDILI
NaN                625
Yes                399
Yes(literature)     12
Name: count, dtype: int64

Cross-tabulation:


AdjudicatedDILI,Yes,Yes(literature),NaN,All
DILIConcern,,,,
Less-DILI-Concern,249,2,237,488
Most-DILI-Concern,124,9,75,208
Most-DILI-concern,0,0,1,1
No-DILI-Concern,26,1,294,321
No-DILI-concern,0,0,18,18
All,399,12,625,1036


In [34]:
# Inspect name matches with different InChIKeys

mismatch_cols = [
    "generic_name",
    "LabelCompoundName",
    "smiles",
    "SMILES",
    "unitox_inchikey",
    "dilirank_inchikey",
    "liver_toxicity_binary_rating_0_1",
    "DILIConcern",
    "AdjudicatedDILI"
]

mismatches = identity_check[
    identity_check["structure_status"] == "Name match, structure differs"
].copy()

print("Total mismatches:", len(mismatches))

display(mismatches[mismatch_cols].head(30))

Total mismatches: 238


,generic_name,LabelCompoundName,smiles,SMILES,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,C1CC1NC2=NC(=NC3=C2N=CN3C4C[C@H](C=C4)CO)N,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O[C@@H]2[C...,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
4,ACETYLCHOLINE CHLORIDE,acetylcholine chloride,CC(=O)OCC[N+](C)(C)C.[Cl-],CC(=O)OCC[N+](C)(C)C,JUGOREOARAHOCO-UHFFFAOYSA-M,OIPILFWXSMYKGL-UHFFFAOYSA-N,0,No-DILI-concern,NaN
5,ACITRETIN,acitretin,COc1cc(C)c(C=CC(C)=CC=CC(C)=CC(=O)O)c(C)c1C,CC1=CC(=C(C(=C1/C=C/C(=C/C=C/C(=C\C(=O)O)/C)/C...,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,adenosine,Nc1ncnc2c1ncn2C1OC(CO)C(O)C1O,C1=NC2=C(C(=N1)N)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,amoxicillin,CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,amphotericin b,CC1C=CC=CC=CC=CC=CC=CC=CC(OC2OC(C)C(O)C(N)C2O)...,C[C@H]1C=CC=CC=CC=CC=CC=CC=C[C@@H](C[C@H]2[C@@...,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,ampicillin,CC1(C)SC2C(NC(=O)C(N)c3ccccc3)C(=O)N2C1C(=O)O,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,anidulafungin,CCCCCOc1ccc(-c2ccc(-c3ccc(C(=O)NC4CC(O)C(O)NC(...,CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,aprepitant,CC(OC1OCCN(Cc2n[nH]c(=O)[nH]2)C1c1ccc(F)cc1)c1...,C[C@H](C1=CC(=CC(=C1)C(F)(F)F)C(F)(F)F)O[C@H]2...,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN


In [35]:
from rdkit import Chem

def structure_summary(smiles):
    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    return {
        "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
        "components": len(Chem.GetMolFrags(mol)),
        "heavy_atoms": mol.GetNumHeavyAtoms()
    }

# Apply to the first 30 mismatches for inspection
sample = mismatches.head(30).copy()

sample["unitox_structure"] = sample["smiles"].apply(structure_summary)
sample["dilirank_structure"] = sample["SMILES"].apply(structure_summary)

display(
    sample[
        [
            "generic_name",
            "unitox_structure",
            "dilirank_structure"
        ]
    ]
)

,generic_name,unitox_structure,dilirank_structure
0,ABACAVIR,{'canonical_smiles': 'Nc1nc(NC2CC2)c2ncn(C3C=C...,{'canonical_smiles': 'Nc1nc(NC2CC2)c2ncn(C3C=C...
1,ACARBOSE,{'canonical_smiles': 'CC1OC(OC2C(CO)OC(OC3C(CO...,{'canonical_smiles': 'C[C@H]1O[C@H](O[C@H]2[C@...
4,ACETYLCHOLINE CHLORIDE,{'canonical_smiles': 'CC(=O)OCC[N+](C)(C)C.[Cl...,"{'canonical_smiles': 'CC(=O)OCC[N+](C)(C)C', '..."
5,ACITRETIN,{'canonical_smiles': 'COc1cc(C)c(C=CC(C)=CC=CC...,{'canonical_smiles': 'COc1cc(C)c(/C=C/C(C)=C/C...
6,ADENOSINE,{'canonical_smiles': 'Nc1ncnc2c1ncn2C1OC(CO)C(...,{'canonical_smiles': 'Nc1ncnc2c1ncn2[C@@H]1O[C...
16,AMOXICILLIN,{'canonical_smiles': 'CC1(C)SC2C(NC(=O)C(N)c3c...,{'canonical_smiles': 'CC1(C)S[C@@H]2[C@H](NC(=...
17,AMPHOTERICIN B,{'canonical_smiles': 'CC1C=CC=CC=CC=CC=CC=CC=C...,{'canonical_smiles': 'C[C@@H]1[C@H](O)[C@@H](C...
18,AMPICILLIN,{'canonical_smiles': 'CC1(C)SC2C(NC(=O)C(N)c3c...,{'canonical_smiles': 'CC1(C)S[C@@H]2[C@H](NC(=...
19,ANIDULAFUNGIN,{'canonical_smiles': 'CCCCCOc1ccc(-c2ccc(-c3cc...,{'canonical_smiles': 'CCCCCOc1ccc(-c2ccc(-c3cc...
20,APREPITANT,{'canonical_smiles': 'CC(OC1OCCN(Cc2n[nH]c(=O)...,{'canonical_smiles': 'C[C@@H](O[C@@H]1OCCN(Cc2...


In [36]:

# Classify chemical identity matches more carefully

def classify_inchikeys(row):
    k1 = row["unitox_inchikey"]
    k2 = row["dilirank_inchikey"]

    if pd.isna(k1) or pd.isna(k2):
        return "Missing InChIKey"

    if k1 == k2:
        return "Exact InChIKey match"

    # First 14 characters encode the connectivity block
    if k1[:14] == k2[:14]:
        return "Same connectivity block; stereochemistry/other layers differ"

    return "Different connectivity block"

identity_check["identity_class"] = identity_check.apply(
    classify_inchikeys, axis=1
)

print(identity_check["identity_class"].value_counts())

# Inspect the first 20 cases with matching connectivity
connectivity_cases = identity_check[
    identity_check["identity_class"]
    == "Same connectivity block; stereochemistry/other layers differ"
]

display(
    connectivity_cases[
        [
            "generic_name",
            "unitox_inchikey",
            "dilirank_inchikey",
            "liver_toxicity_binary_rating_0_1",
            "DILIConcern",
            "AdjudicatedDILI",
        ]
    ].head(20)
)


identity_class
Exact InChIKey match                                            253
Same connectivity block; stereochemistry/other layers differ    193
Different connectivity block                                     45
Missing InChIKey                                                  6
Name: count, dtype: int64


,generic_name,unitox_inchikey,dilirank_inchikey,liver_toxicity_binary_rating_0_1,DILIConcern,AdjudicatedDILI
0,ABACAVIR,MCGSCOLBFJQGHM-UHFFFAOYSA-N,MCGSCOLBFJQGHM-PEHGTWAWSA-N,1,Most-DILI-Concern,Yes
1,ACARBOSE,XUFXOAAUWZOOIT-UHFFFAOYSA-N,XUFXOAAUWZOOIT-JMPDRRIHSA-N,1,Most-DILI-Concern,Yes
5,ACITRETIN,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UGOGCBOOSA-N,1,Most-DILI-Concern,Yes
6,ADENOSINE,OIRDTQYFTABQOQ-UHFFFAOYSA-N,OIRDTQYFTABQOQ-KQYNXXCUSA-N,0,No-DILI-Concern,NaN
16,AMOXICILLIN,LSQZJLSUYDQPKJ-UHFFFAOYSA-N,LSQZJLSUYDQPKJ-NJBDSQKTSA-N,1,Less-DILI-Concern,Yes
17,AMPHOTERICIN B,APKFDSVGJQXUKY-UHFFFAOYSA-N,APKFDSVGJQXUKY-KKGHZKTASA-N,0,Less-DILI-Concern,Yes
18,AMPICILLIN,AVKUERGKIZMTKX-UHFFFAOYSA-N,AVKUERGKIZMTKX-NJBDSQKTSA-N,0,Less-DILI-Concern,Yes
19,ANIDULAFUNGIN,JHVAMHSQVVQIOT-UHFFFAOYSA-N,JHVAMHSQVVQIOT-MFAJLEFUSA-N,1,Most-DILI-Concern,NaN
20,APREPITANT,ATALOFNDEOCMKK-UHFFFAOYSA-N,ATALOFNDEOCMKK-BYYRLHKVSA-N,0,Less-DILI-Concern,NaN
21,ARGATROBAN,KXNPVXPOPUZYGB-UHFFFAOYSA-N,KXNPVXPOPUZYGB-XYVMCAHJSA-N,1,No-DILI-Concern,NaN


In [37]:

# 1. Count the chemical identity groups
print("Chemical identity classification:")
print(identity_check["identity_class"].value_counts(dropna=False))

# 2. Normalize DILIrank category capitalization and whitespace
identity_check["DILIConcern_clean"] = (
    identity_check["DILIConcern"]
    .astype("string")
    .str.strip()
    .str.lower()
)

identity_check["DILIConcern_clean"] = (
    identity_check["DILIConcern_clean"]
    .replace({
        "most-dili-concern": "Most",
        "less-dili-concern": "Less",
        "no-dili-concern": "No"
    })
)

# 3. First, use only exact full InChIKey matches
exact = identity_check[
    identity_check["identity_class"] == "Exact InChIKey match"
].copy()

print("\nExact-match sample size:", len(exact))
print("\nUniTox label vs DILIrank category:")
display(pd.crosstab(
    exact["DILIConcern_clean"],
    exact["liver_toxicity_binary_rating_0_1"],
    margins=True
))


Chemical identity classification:
identity_class
Exact InChIKey match                                            253
Same connectivity block; stereochemistry/other layers differ    193
Different connectivity block                                     45
Missing InChIKey                                                  6
Name: count, dtype: int64

Exact-match sample size: 253

UniTox label vs DILIrank category:


liver_toxicity_binary_rating_0_1,0,1,All
DILIConcern_clean,,,
Less,37,100,137
Most,2,53,55
No,47,14,61
All,86,167,253


In [38]:

from scipy.stats import spearmanr, chi2_contingency
import numpy as np
import pandas as pd

# Use the exact chemical identity matches only
analysis_df = exact.copy()

# Explicit ordinal encoding; Less remains a separate category
analysis_df["dili_ordinal"] = (
    analysis_df["DILIConcern_clean"]
    .map({"No": 0, "Less": 1, "Most": 2})
)

analysis_df["unitox_label"] = pd.to_numeric(
    analysis_df["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
)

analysis_df = analysis_df.dropna(
    subset=["dili_ordinal", "unitox_label"]
)

# Spearman rank correlation
rho, spearman_p = spearmanr(
    analysis_df["dili_ordinal"],
    analysis_df["unitox_label"]
)

# Chi-square test for the 3 x 2 table
table = pd.crosstab(
    analysis_df["DILIConcern_clean"],
    analysis_df["unitox_label"]
).reindex(
    index=["No", "Less", "Most"],
    columns=[0, 1],
    fill_value=0
)

chi2, chi_p, dof, expected = chi2_contingency(table)

print("N =", len(analysis_df))
print("\nContingency table:")
display(table)

print(f"Spearman rho = {rho:.4f}")
print(f"Spearman p-value = {spearman_p:.6g}")
print(f"Chi-square statistic = {chi2:.4f}")
print(f"Chi-square degrees of freedom = {dof}")
print(f"Chi-square p-value = {chi_p:.6g}")
print("\nExpected cell counts:")
display(pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
))


N = 253

Contingency table:


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Less,37,100
Most,2,53


Spearman rho = 0.5313
Spearman p-value = 7.86018e-20
Chi-square statistic = 75.9682
Chi-square degrees of freedom = 2
Chi-square p-value = 3.18945e-17

Expected cell counts:


unitox_label,0,1
DILIConcern_clean,,
No,20.735178,40.264822
Less,46.569170,90.430830
Most,18.695652,36.304348


In [39]:

from scipy.stats import chi2_contingency, spearmanr
import pandas as pd

# Use the existing strict exact-match subset
analysis_df = exact.copy()

analysis_df["dili_ordinal"] = (
    analysis_df["DILIConcern_clean"]
    .map({"No": 0, "Less": 1, "Most": 2})
)

analysis_df["unitox_label"] = pd.to_numeric(
    analysis_df["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
)

analysis_df = analysis_df.dropna(
    subset=["dili_ordinal", "unitox_label"]
)

# Observed counts
observed = pd.crosstab(
    analysis_df["DILIConcern_clean"],
    analysis_df["unitox_label"]
).reindex(
    index=["No", "Less", "Most"],
    columns=[0, 1],
    fill_value=0
)

# Chi-square test
chi2, p_chi, dof, expected = chi2_contingency(observed)

# Spearman correlation
rho, p_spearman = spearmanr(
    analysis_df["dili_ordinal"],
    analysis_df["unitox_label"]
)

print("N =", len(analysis_df))
print("\nObserved counts:")
display(observed)

print("\nExpected counts under independence:")
display(pd.DataFrame(
    expected,
    index=observed.index,
    columns=observed.columns
))

print(f"\nChi-square = {chi2:.4f}")
print(f"Degrees of freedom = {dof}")
print(f"Chi-square p-value = {p_chi:.6g}")
print(f"Spearman rho = {rho:.4f}")
print(f"Spearman p-value = {p_spearman:.6g}")

# Clear-extremes sensitivity analysis: exclude Less
extremes = analysis_df[
    analysis_df["DILIConcern_clean"].isin(["No", "Most"])
].copy()

extreme_table = pd.crosstab(
    extremes["DILIConcern_clean"],
    extremes["unitox_label"]
).reindex(
    index=["No", "Most"],
    columns=[0, 1],
    fill_value=0
)

print("\nClear-extremes table (Less excluded):")
display(extreme_table)

print("\nPositive-label rate by category:")
display(
    extremes.groupby("DILIConcern_clean")["unitox_label"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "positive_rate"})
)


N = 253

Observed counts:


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Less,37,100
Most,2,53



Expected counts under independence:


unitox_label,0,1
DILIConcern_clean,,
No,20.735178,40.264822
Less,46.569170,90.430830
Most,18.695652,36.304348



Chi-square = 75.9682
Degrees of freedom = 2
Chi-square p-value = 3.18945e-17
Spearman rho = 0.5313
Spearman p-value = 7.86018e-20

Clear-extremes table (Less excluded):


unitox_label,0,1
DILIConcern_clean,,
No,47,14
Most,2,53



Positive-label rate by category:


,count,positive_rate
DILIConcern_clean,,
Most,55,0.963636
No,61,0.229508


In [40]:

from scipy.stats import fisher_exact
import numpy as np

# DILIrank Most = reference positive; No = reference negative
# UniTox 1 = predicted positive; UniTox 0 = predicted negative
tn = 47  # DILIrank No, UniTox 0
fp = 14  # DILIrank No, UniTox 1
fn = 2   # DILIrank Most, UniTox 0
tp = 53  # DILIrank Most, UniTox 1

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
npv = tn / (tn + fn)
accuracy = (tp + tn) / (tp + tn + fp + fn)

odds_ratio, fisher_p = fisher_exact(
    [[tp, fn], [fp, tn]],
    alternative="two-sided"
)

print(f"Sample size: {tp + tn + fp + fn}")
print(f"Raw agreement: {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Positive predictive value: {precision:.4f}")
print(f"Negative predictive value: {npv:.4f}")
print(f"Odds ratio: {odds_ratio:.4f}")
print(f"Fisher exact p-value: {fisher_p:.6g}")


Sample size: 116
Raw agreement: 0.8621
Sensitivity: 0.9636
Specificity: 0.7705
Positive predictive value: 0.7910
Negative predictive value: 0.9592
Odds ratio: 88.9643
Fisher exact p-value: 3.2226e-17


In [41]:

import numpy as np
import pandas as pd

# Define the target and molecular structure columns
TARGET = "liver_toxicity_binary_rating_0_1"
SMILES_COL = "smiles"

# Make a separate working copy; preserve the original dataframe
work_df = df.copy().reset_index(drop=True)

print("=== DATASET AUDIT ===")
print("Dataset shape:", work_df.shape)
print("Target column exists:", TARGET in work_df.columns)
print("SMILES column exists:", SMILES_COL in work_df.columns)

if TARGET not in work_df.columns or SMILES_COL not in work_df.columns:
    raise ValueError("Required target or SMILES column is missing.")

# Convert target to numeric for validation
work_df[TARGET] = pd.to_numeric(work_df[TARGET], errors="coerce")

print("\n=== TARGET DISTRIBUTION ===")
print(work_df[TARGET].value_counts(dropna=False).sort_index())

print("\n=== MISSING VALUES ===")
print("Missing targets:", work_df[TARGET].isna().sum())
print("Missing SMILES:", work_df[SMILES_COL].isna().sum())
print("Duplicate SMILES strings:", work_df[SMILES_COL].duplicated().sum())

# Confirm that labels are binary and non-missing
valid_labels = set(work_df[TARGET].dropna().unique())
print("\nObserved non-missing label values:", valid_labels)

if work_df[TARGET].isna().any():
    print("WARNING: Missing target labels need investigation.")

if not valid_labels.issubset({0, 1}):
    print("WARNING: Target contains values other than 0 and 1.")

# Summarize the external-reference subset
print("\n=== EXTERNAL REFERENCE ===")
print("Exact InChIKey matches:", len(exact))
print("DILIrank category counts:")
print(exact["DILIConcern_clean"].value_counts(dropna=False))

print("\nAudit completed.")


=== DATASET AUDIT ===
Dataset shape: (1349, 44)
Target column exists: True
SMILES column exists: True

=== TARGET DISTRIBUTION ===
liver_toxicity_binary_rating_0_1
0    613
1    736
Name: count, dtype: int64

=== MISSING VALUES ===
Missing targets: 0
Missing SMILES: 0
Duplicate SMILES strings: 0

Observed non-missing label values: {np.int64(0), np.int64(1)}

=== EXTERNAL REFERENCE ===
Exact InChIKey matches: 253
DILIrank category counts:
DILIConcern_clean
Less    137
No       61
Most     55
Name: count, dtype: Int64

Audit completed.


In [42]:

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# Create a reproducible working dataset
work_df = work_df.reset_index(drop=True).copy()

# Generate a Bemis-Murcko scaffold for each molecule
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold, canonical=True)

work_df["scaffold"] = work_df[SMILES_COL].apply(get_scaffold)

if work_df["scaffold"].isna().any():
    raise ValueError("Invalid structures found. Investigate before splitting.")

print("Total drugs:", len(work_df))
print("Unique Murcko scaffolds:", work_df["scaffold"].nunique())
print("Largest scaffold groups:")
print(work_df["scaffold"].value_counts().head(10))

# Empty scaffold is retained as a group:
# many acyclic compounds have no Bemis-Murcko scaffold.
print("\nDrugs with empty scaffold:",
      int((work_df["scaffold"] == "").sum()))

# Assign whole scaffold groups to 10 folds.
# Greedy assignment balances fold sizes, while preserving scaffold groups.
def assign_scaffold_folds(data, n_splits=10, random_state=42):
    rng = np.random.default_rng(random_state)

    y = data[TARGET].to_numpy(dtype=int)
    scaffold_values = data["scaffold"].to_numpy()

    groups = {}
    for idx, scaffold in enumerate(scaffold_values):
        groups.setdefault(scaffold, []).append(idx)

    scaffold_items = list(groups.items())
    rng.shuffle(scaffold_items)

    # Large groups first; randomization resolves equal-size ties
    scaffold_items.sort(key=lambda item: len(item[1]), reverse=True)

    fold_indices = [[] for _ in range(n_splits)]

    # Keep approximate class counts and sizes balanced
    fold_class_counts = np.zeros((n_splits, 2), dtype=int)

    for scaffold, indices in scaffold_items:
        labels = y[indices]
        group_counts = np.bincount(labels, minlength=2)

        scores = []
        for fold in range(n_splits):
            proposed = fold_class_counts.copy()
            proposed[fold] += group_counts

            # Penalize imbalance in total class counts across folds
            class_imbalance = np.sum(
                np.std(proposed, axis=0)
            )

            size_imbalance = np.std(
                [len(fold_indices[j]) +
                 (len(indices) if j == fold else 0)
                 for j in range(n_splits)]
            )

            scores.append(class_imbalance + 0.01 * size_imbalance)

        best_fold = int(np.argmin(scores))
        fold_indices[best_fold].extend(indices)
        fold_class_counts[best_fold] += group_counts

    fold_id = np.full(len(data), -1, dtype=int)
    for fold, indices in enumerate(fold_indices):
        fold_id[indices] = fold

    return fold_id

work_df["fold"] = assign_scaffold_folds(
    work_df, n_splits=10, random_state=42
)

# Verify fold sizes and class counts
print("\n=== FOLD SUMMARY ===")
fold_summary = []

for fold in range(10):
    subset = work_df[work_df["fold"] == fold]
    fold_summary.append({
        "fold": fold,
        "n_drugs": len(subset),
        "negative": int((subset[TARGET] == 0).sum()),
        "positive": int((subset[TARGET] == 1).sum()),
        "unique_scaffolds": subset["scaffold"].nunique()
    })

display(pd.DataFrame(fold_summary))

# Verify no scaffold is shared between folds
scaffold_fold_counts = work_df.groupby("scaffold")["fold"].nunique()
leaked_scaffolds = int((scaffold_fold_counts > 1).sum())

print("\nScaffolds appearing in multiple folds:", leaked_scaffolds)

assert work_df["fold"].ge(0).all()
assert leaked_scaffolds == 0

print("Scaffold leakage check passed.")


[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:20] WARNING: not removing hydrogen atom without neighbors
[16:09:22] WARNING: not removing hydrogen atom without neighbors
[16:09:22] WARNING: not removing hydrogen atom without neighbors


Total drugs: 1349
Unique Murcko scaffolds: 802
Largest scaffold groups:
scaffold
                                      122
c1ccccc1                               77
O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21        15
O=C1C=C2CCC3C4CCCC4CCC3C2CC1           12
O=C1C=CC2C(=O)C3=Cc4ccccc4CC3CC2C1     10
c1ccncc1                                8
c1ccccc1.c1ccccc1                       8
c1ccc(Cc2ccccc2)cc1                     8
c1ccc2c(c1)Sc1ccccc1N2CCCN1CCNCC1       8
c1c[nH]cn1                              7
Name: count, dtype: int64

Drugs with empty scaffold: 122

=== FOLD SUMMARY ===


,fold,n_drugs,negative,positive,unique_scaffolds
0,0,163,90,73,40
1,1,132,58,74,56
2,2,132,58,74,87
3,3,132,58,74,88
4,4,132,58,74,89
5,5,132,58,74,88
6,6,132,59,73,89
7,7,132,58,74,89
8,8,131,58,73,88
9,9,131,58,73,88



Scaffolds appearing in multiple folds: 0
Scaffold leakage check passed.


In [43]:

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

def validate_molecule(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return False
    return True

work_df["smiles_valid"] = work_df[SMILES_COL].apply(validate_molecule)

print("Invalid SMILES:", int((~work_df["smiles_valid"]).sum()))
print("Valid SMILES:", int(work_df["smiles_valid"].sum()))

assert work_df["smiles_valid"].all(), (
    "Investigate invalid structures before proceeding."
)


[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors


Invalid SMILES: 0
Valid SMILES: 1349


[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors


In [44]:

# Save the exact split assignment for reproducibility
fold_file = "unitox_liver_scaffold_folds.csv"

work_df[
    ["generic_name", SMILES_COL, TARGET, "scaffold", "fold"]
].to_csv(fold_file, index=False)

print("Saved fold assignments to:", fold_file)


Saved fold assignments to: unitox_liver_scaffold_folds.csv


In [45]:

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Ensure the working dataframe has a clean positional index
work_df = work_df.reset_index(drop=True).copy()

# Generate Morgan fingerprints (radius 2, 2048 bits)
fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = fp_generator.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X = np.vstack(work_df[SMILES_COL].apply(smiles_to_fp).to_numpy())
y = work_df[TARGET].to_numpy(dtype=int)
folds = work_df["fold"].to_numpy(dtype=int)

print("Fingerprint matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Fold IDs:", np.unique(folds))

# Store exactly one out-of-fold probability for every drug
oof_probability = np.full(len(work_df), np.nan)
fold_results = []

for fold in sorted(np.unique(folds)):
    train_idx = np.where(folds != fold)[0]
    test_idx = np.where(folds == fold)[0]

    model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    )

    model.fit(X[train_idx], y[train_idx])

    # Probability of the positive class
    positive_col = list(model.classes_).index(1)
    probabilities = model.predict_proba(X[test_idx])[:, positive_col]

    oof_probability[test_idx] = probabilities

    # AUC is undefined if a held-out fold has only one class
    if len(np.unique(y[test_idx])) == 2:
        fold_auc = roc_auc_score(y[test_idx], probabilities)
    else:
        fold_auc = np.nan

    fold_results.append({
        "fold": int(fold),
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "test_auc": fold_auc
    })

assert np.isfinite(oof_probability).all(), (
    "Some drugs do not have an out-of-fold prediction."
)

# Basic performance and agreement summaries
oof_prediction = (oof_probability >= 0.5).astype(int)

print("\n=== OUT-OF-FOLD RESULTS ===")
print("Number of predictions:", len(oof_probability))
print("Pooled OOF ROC-AUC:", round(
    roc_auc_score(y, oof_probability), 4
))
print("Pooled OOF accuracy:", round(
    accuracy_score(y, oof_prediction), 4
))

print("\nFold-level results:")
display(pd.DataFrame(fold_results))

# Save predictions for the next reliability-score stage
oof_df = work_df[
    ["generic_name", SMILES_COL, TARGET, "scaffold", "fold"]
].copy()

oof_df["oof_probability"] = oof_probability
oof_df["oof_prediction"] = oof_prediction
oof_df["oof_agrees_with_label"] = (
    oof_df["oof_prediction"] == oof_df[TARGET]
)

# Confidence in the model's binary decision, from 0 to 1
oof_df["oof_model_confidence"] = np.abs(
    2 * oof_df["oof_probability"] - 1
)

oof_df.to_csv("unitox_liver_oof_predictions.csv", index=False)

print("\nSaved: unitox_liver_oof_predictions.csv")
print("\nModel agreement with existing labels:")
display(
    oof_df["oof_agrees_with_label"]
    .value_counts()
    .rename(index={True: "Agrees", False: "Disagrees"})
    .to_frame("number_of_drugs")
)


[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:23] WARNING: not removing hydrogen atom without neighbors
[16:09:24] WARNING: not removing hydrogen atom without neighbors
[16:09:24] WARNING: not removing hydrogen atom without neighbors


Fingerprint matrix shape: (1349, 2048)
Target shape: (1349,)
Fold IDs: [0 1 2 3 4 5 6 7 8 9]

=== OUT-OF-FOLD RESULTS ===
Number of predictions: 1349
Pooled OOF ROC-AUC: 0.7204
Pooled OOF accuracy: 0.6723

Fold-level results:


,fold,train_n,test_n,test_auc
0,0,1186,163,0.716134
1,1,1217,132,0.709576
2,2,1217,132,0.738234
3,3,1217,132,0.721808
4,4,1217,132,0.732992
5,5,1217,132,0.592032
6,6,1217,132,0.680984
7,7,1217,132,0.803355
8,8,1218,131,0.751771
9,9,1218,131,0.681153



Saved: unitox_liver_oof_predictions.csv

Model agreement with existing labels:


,number_of_drugs
oof_agrees_with_label,
Agrees,907
Disagrees,442


In [46]:

# Overall performance
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd
import numpy as np

print("Pooled OOF ROC-AUC:",
      round(roc_auc_score(y, oof_probability), 4))
print("Pooled OOF accuracy:",
      round(accuracy_score(y, oof_prediction), 4))

# Confidence distribution by agreement status
print("\nConfidence by agreement with UniTox label:")
display(
    oof_df.groupby("oof_agrees_with_label")[
        "oof_model_confidence"
    ].agg(["count", "mean", "median", "std"])
)

# Confidence distribution by original UniTox label
print("\nConfidence by UniTox class:")
display(
    oof_df.groupby(TARGET)["oof_model_confidence"]
    .agg(["count", "mean", "median", "std"])
)

# How often does the model disagree within each class?
print("\nAgreement rate by UniTox class:")
display(
    oof_df.groupby(TARGET)["oof_agrees_with_label"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "agreement_rate"})
)


Pooled OOF ROC-AUC: 0.7204
Pooled OOF accuracy: 0.6723

Confidence by agreement with UniTox label:


,count,mean,median,std
oof_agrees_with_label,,,,
False,442,0.265103,0.236,0.179759
True,907,0.346853,0.328,0.202884



Confidence by UniTox class:


,count,mean,median,std
liver_toxicity_binary_rating_0_1,,,,
0,613,0.295996,0.256,0.199477
1,736,0.340117,0.328,0.197004



Agreement rate by UniTox class:


,count,agreement_rate
liver_toxicity_binary_rating_0_1,,
0,613,0.505710
1,736,0.811141


In [61]:
print("OOF columns:", oof_df.columns.tolist())
print("Dataset columns:", work_df.columns.tolist())

OOF columns: ['generic_name', 'smiles', 'liver_toxicity_binary_rating_0_1', 'scaffold', 'fold', 'oof_probability', 'oof_prediction', 'oof_agrees_with_label', 'oof_model_confidence']
Dataset columns: ['generic_name', 'cardio_toxicity_reasoning', 'cardio_toxicity_ternary_rating', 'cardio_toxicity_binary_rating', 'cardio_toxicity_confident_ternary_rating_0_1', 'cardio_toxicity_binary_rating_0_1', 'dermatological_toxicity_reasoning', 'dermatological_toxicity_ternary_rating', 'dermatological_toxicity_binary_rating', 'dermatological_toxicity_confident_ternary_rating_0_1', 'dermatological_toxicity_binary_rating_0_1', 'hematological_reasoning', 'hematological_ternary_rating', 'hematological_binary_rating', 'hematological_confident_ternary_rating_0_1', 'hematological_binary_rating_0_1', 'infertility_reasoning', 'infertility_ternary_rating', 'infertility_binary_rating', 'infertility_confident_ternary_rating_0_1', 'infertility_binary_rating_0_1', 'liver_toxicity_reasoning', 'liver_toxicity_ternar

In [62]:
from rdkit import Chem
from rdkit.Chem import inchi

# Create InChIKeys if they are not already available
if "inchikey" not in work_df.columns:
    work_df["inchikey"] = work_df["smiles"].apply(
        lambda s: (
            inchi.MolToInchiKey(Chem.MolFromSmiles(s))
            if isinstance(s, str) and Chem.MolFromSmiles(s) is not None
            else None
        )
    )

# Attach InChIKeys to OOF data using the SMILES as the key
smiles_to_key = (
    work_df.dropna(subset=["smiles"])
    .drop_duplicates(subset=["smiles"])
    .set_index("smiles")["inchikey"]
)

oof_df["inchikey"] = oof_df["smiles"].map(smiles_to_key)

print("OOF rows:", len(oof_df))
print("Non-missing InChIKeys:", oof_df["inchikey"].notna().sum())
print("Unique InChIKeys:", oof_df["inchikey"].nunique())

[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not removing hydrogen atom without neighbors
[16:16:08] WARNING: not r

OOF rows: 1349
Non-missing InChIKeys: 1349
Unique InChIKeys: 1348


[16:16:09] WARNING: not removing hydrogen atom without neighbors
[16:16:09] WARNING: not removing hydrogen atom without neighbors
[16:16:09] WARNING: not removing hydrogen atom without neighbors
[16:16:09] WARNING: not removing hydrogen atom without neighbors


In [63]:
# Diagnose duplicate chemical identifiers
print("UniTox rows:", len(oof_df))
print("Unique non-missing InChIKeys:", oof_df["inchikey"].nunique())
print(
    "Rows with duplicate InChIKeys:",
    int(oof_df["inchikey"].duplicated(keep=False).sum())
)

# Display duplicated chemical identifiers
duplicate_mask = oof_df["inchikey"].notna() & oof_df["inchikey"].duplicated(keep=False)

duplicate_keys = (
    oof_df.loc[duplicate_mask]
    .sort_values("inchikey")
)

display(
    duplicate_keys[
        ["generic_name", "smiles", "inchikey", TARGET, "fold"]
    ].head(30)
)

# Check whether duplicated InChIKeys have conflicting toxicity labels
label_conflicts = (
    duplicate_keys.groupby("inchikey")[TARGET]
    .nunique()
)

print(
    "Duplicated InChIKeys with conflicting UniTox labels:",
    int((label_conflicts > 1).sum())
)

# Check whether identical chemical structures occur in different folds
fold_counts = (
    duplicate_keys.groupby("inchikey")["fold"]
    .nunique()
)

print(
    "Duplicated InChIKeys appearing in multiple folds:",
    int((fold_counts > 1).sum())
)

UniTox rows: 1349
Unique non-missing InChIKeys: 1348
Rows with duplicate InChIKeys: 2


,generic_name,smiles,inchikey,liver_toxicity_binary_rating_0_1,fold
986,ATAZANAVIR SULFATE,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7
1276,REYATAZ,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7


Duplicated InChIKeys with conflicting UniTox labels: 0
Duplicated InChIKeys appearing in multiple folds: 0


In [64]:

# Build one external record per full InChIKey
external = exact[
    ["unitox_inchikey", "DILIConcern_clean"]
].copy()

external = external.dropna(subset=["unitox_inchikey"])

# If an InChIKey has conflicting external categories,
# mark it ambiguous instead of choosing one arbitrarily.
def combine_categories(values):
    values = sorted(set(values.dropna().astype(str)))
    return values[0] if len(values) == 1 else "Conflicting"

external_unique = (
    external.groupby("unitox_inchikey")["DILIConcern_clean"]
    .agg(combine_categories)
    .reset_index()
)

external_unique["external_reference_label"] = (
    external_unique["DILIConcern_clean"]
    .map({"No": 0, "Most": 1})
)

# Remove any previous external columns before rerunning the merge
cols_to_remove = [
    "DILIConcern_clean",
    "unitox_inchikey",
    "external_reference_label",
    "A_external_agreement"
]
oof_df = oof_df.drop(
    columns=[c for c in cols_to_remove if c in oof_df.columns]
)

rows_before = len(oof_df)

oof_df = oof_df.merge(
    external_unique,
    left_on="inchikey",
    right_on="unitox_inchikey",
    how="left",
    validate="many_to_one"
)

assert len(oof_df) == rows_before, "Merge changed the number of rows!"

# Assign agreement only for clear reference categories
eligible = oof_df["external_reference_label"].notna()

oof_df["A_external_agreement"] = np.nan
oof_df.loc[eligible, "A_external_agreement"] = (
    oof_df.loc[eligible, TARGET].astype(int)
    == oof_df.loc[eligible, "external_reference_label"].astype(int)
).astype(int)

print("Rows retained:", len(oof_df))
print("Rows with clear external reference:", int(eligible.sum()))
print("External agreements:",
      int((oof_df["A_external_agreement"] == 1).sum()))
print("External disagreements:",
      int((oof_df["A_external_agreement"] == 0).sum()))
print("Unassigned external signals:",
      int(oof_df["A_external_agreement"].isna().sum()))

display(
    oof_df.groupby("DILIConcern_clean", dropna=False)[
        "A_external_agreement"
    ].agg(["count", "mean"])
)


Rows retained: 1349
Rows with clear external reference: 116
External agreements: 100
External disagreements: 16
Unassigned external signals: 1233


,count,mean
DILIConcern_clean,,
Less,0,NaN
Most,55,0.963636
No,61,0.770492
<NA>,0,NaN


In [65]:

# Evaluate model confidence against external agreement
external_eval = oof_df[
    oof_df["A_external_agreement"].notna()
].copy()

external_eval["external_agreement_label"] = (
    external_eval["A_external_agreement"].astype(int)
)

print("Eligible drugs:", len(external_eval))

print("\nModel confidence by external agreement:")
display(
    external_eval.groupby("external_agreement_label")[
        "oof_model_confidence"
    ].agg(["count", "mean", "median", "std"])
    .rename(index={
        0: "Disagrees with external reference",
        1: "Agrees with external reference"
    })
)

print("\nModel agreement with UniTox by external agreement:")
display(
    external_eval.groupby("external_agreement_label")[
        "oof_agrees_with_label"
    ].agg(["count", "mean"])
    .rename(index={
        0: "Disagrees with external reference",
        1: "Agrees with external reference"
    })
)

print("\nCross-tabulation:")
display(pd.crosstab(
    external_eval["external_agreement_label"],
    external_eval["oof_agrees_with_label"],
    rownames=["External agreement"],
    colnames=["Model agrees with UniTox"]
))


Eligible drugs: 116

Model confidence by external agreement:


,count,mean,median,std
external_agreement_label,,,,
Disagrees with external reference,16,0.27875,0.296,0.139857
Agrees with external reference,100,0.30494,0.270,0.190654



Model agreement with UniTox by external agreement:


,count,mean
external_agreement_label,,
Disagrees with external reference,16,0.625
Agrees with external reference,100,0.690



Cross-tabulation:


Model agrees with UniTox,False,True
External agreement,,
0,6,10
1,31,69


In [66]:

from scipy.stats import fisher_exact, chi2_contingency
from sklearn.metrics import cohen_kappa_score

# Rows: external agreement (0/1)
# Columns: model agrees with UniTox (False/True)
ct = pd.crosstab(
    external_eval["external_agreement_label"],
    external_eval["oof_agrees_with_label"]
).reindex(index=[0, 1], columns=[False, True], fill_value=0)

print("Contingency table:")
display(ct)

# Fisher exact test for association
odds_ratio, p_value = fisher_exact(ct.to_numpy())

print(f"Odds ratio: {odds_ratio:.4f}")
print(f"Fisher exact p-value: {p_value:.6g}")

# Descriptive agreement between the two binary indicators
kappa = cohen_kappa_score(
    external_eval["external_agreement_label"].astype(int),
    external_eval["oof_agrees_with_label"].astype(int)
)

print(f"Cohen's kappa: {kappa:.4f}")


Contingency table:


oof_agrees_with_label,False,True
external_agreement_label,,
0,6,10
1,31,69


Odds ratio: 1.3355
Fisher exact p-value: 0.578676
Cohen's kappa: 0.0419


In [67]:
import numpy as np
from rdkit import DataStructs


def molecular_neighbour_consistency(
    query_fp,
    reference_fps,
    reference_labels,
    k=10,
    exclude_index=None,
):
    """
    Calculate similarity-weighted label consistency.

    query_fp:
        Fingerprint of the molecule being evaluated.

    reference_fps:
        Fingerprints of eligible reference molecules.

    reference_labels:
        Binary labels aligned with reference_fps.

    k:
        Number of nearest molecular neighbours.

    exclude_index:
        Optional reference index to exclude, especially when
        the query molecule is itself part of the reference set.

    Returns:
        A score between 0 and 1, or np.nan if no usable
        neighbours are available.
    """
    labels = np.asarray(reference_labels, dtype=int)

    if len(reference_fps) != len(labels):
        raise ValueError("Fingerprint and label counts differ.")

    if k < 1:
        raise ValueError("k must be at least 1.")

    if not len(reference_fps):
        return np.nan

    similarities = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            query_fp,
            reference_fps
        ),
        dtype=float
    )

    if exclude_index is not None:
        similarities[exclude_index] = -1.0

    # Only use neighbours with positive similarity.
    eligible = np.flatnonzero(similarities > 0)

    if len(eligible) == 0:
        return np.nan

    # Select the k most similar eligible molecules.
    order = eligible[
        np.argsort(similarities[eligible])[::-1]
    ][:k]

    weights = similarities[order]
    neighbour_labels = labels[order]

    agreement = (neighbour_labels == query_label).astype(float)
    return np.sum(weights * agreement) / np.sum(weights)

In [68]:
def molecular_neighbour_consistency(
    query_fp,
    query_label,
    reference_fps,
    reference_labels,
    k=10,
    exclude_index=None,
):
    labels = np.asarray(reference_labels, dtype=int)

    if len(reference_fps) != len(labels):
        raise ValueError("Fingerprint and label counts differ.")

    if k < 1:
        raise ValueError("k must be at least 1.")

    if len(reference_fps) == 0:
        return np.nan

    similarities = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            query_fp, reference_fps
        ),
        dtype=float
    )

    if exclude_index is not None:
        similarities[exclude_index] = -1.0

    eligible = np.flatnonzero(similarities > 0)

    if len(eligible) == 0:
        return np.nan

    order = eligible[
        np.argsort(similarities[eligible])[::-1]
    ][:k]

    weights = similarities[order]
    agreement = (labels[order] == query_label).astype(float)

    return float(np.sum(weights * agreement) / np.sum(weights))

In [50]:
def training_neighbour_scores(
    train_fps,
    train_labels,
    k=10
):
    scores = []

    for i, fp in enumerate(train_fps):
        score = molecular_neighbour_consistency(
            query_fp=fp,
            query_label=train_labels[i],
            reference_fps=train_fps,
            reference_labels=train_labels,
            k=k,
            exclude_index=i
        )
        scores.append(score)

    return np.asarray(scores, dtype=float)

In [69]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Sample molecules for testing
smiles = [
    "CCO",        # Ethanol
    "CCCO",       # Propanol
    "CCCCO",      # Butanol
    "c1ccccc1",   # Benzene
    "Cc1ccccc1",  # Toluene
    "CC(=O)O",    # Acetic acid
]

# Illustrative labels only; not real toxicity annotations
labels = np.array([0, 0, 0, 1, 1, 0])

# Generate Morgan fingerprints
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

molecules = [Chem.MolFromSmiles(s) for s in smiles]
fingerprints = [
    generator.GetFingerprint(mol)
    for mol in molecules
]

def neighbour_consistency(i, k=3):
    similarities = np.array(
        DataStructs.BulkTanimotoSimilarity(
            fingerprints[i], fingerprints
        )
    )

    # Exclude the query molecule itself
    similarities[i] = -1

    # Select the k most similar molecules
    neighbours = np.argsort(similarities)[::-1][:k]
    neighbours = [
        j for j in neighbours if similarities[j] > 0
    ]

    if not neighbours:
        return np.nan

    weights = similarities[neighbours]
    agreements = (labels[neighbours] == labels[i]).astype(float)

    return np.sum(weights * agreements) / np.sum(weights)

# Actually execute the function and print the results
print("Molecular-neighbour consistency scores")
print("-" * 45)

for i, s in enumerate(smiles):
    score = neighbour_consistency(i, k=3)
    print(f"{s:15s} Label={labels[i]}  S_i={score}")

Molecular-neighbour consistency scores
---------------------------------------------
CCO             Label=0  S_i=1.0
CCCO            Label=0  S_i=1.0
CCCCO           Label=0  S_i=1.0
c1ccccc1        Label=1  S_i=0.7105263157894737
Cc1ccccc1       Label=1  S_i=0.5925925925925926
CC(=O)O         Label=0  S_i=1.0


In [70]:

# STEP 1: Check notebook variables and data alignment

print("=== NOTEBOOK DIAGNOSTIC ===")

# Check whether key variables exist
variables_to_check = [
    "work_df",
    "oof_df",
    "TARGET",
    "scaffold_folds",
    "X_morgan"
]

for name in variables_to_check:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'NOT FOUND'}")

# Inspect the main dataframes
if "work_df" in globals():
    print("\nwork_df shape:", work_df.shape)
    print("work_df columns:", work_df.columns.tolist())

if "oof_df" in globals():
    print("\noof_df shape:", oof_df.shape)
    print("oof_df columns:", oof_df.columns.tolist())
    print("Contains inchikey:", "inchikey" in oof_df.columns)

# Check the target
if "TARGET" in globals():
    print("\nTARGET:", TARGET)

# Check the Morgan fingerprint array
if "X_morgan" in globals():
    print("\nX_morgan shape:", X_morgan.shape)

print("\n=== DIAGNOSTIC COMPLETE ===")

=== NOTEBOOK DIAGNOSTIC ===
work_df: AVAILABLE
oof_df: AVAILABLE
TARGET: AVAILABLE
scaffold_folds: NOT FOUND
X_morgan: AVAILABLE

work_df shape: (1349, 47)
work_df columns: ['generic_name', 'cardio_toxicity_reasoning', 'cardio_toxicity_ternary_rating', 'cardio_toxicity_binary_rating', 'cardio_toxicity_confident_ternary_rating_0_1', 'cardio_toxicity_binary_rating_0_1', 'dermatological_toxicity_reasoning', 'dermatological_toxicity_ternary_rating', 'dermatological_toxicity_binary_rating', 'dermatological_toxicity_confident_ternary_rating_0_1', 'dermatological_toxicity_binary_rating_0_1', 'hematological_reasoning', 'hematological_ternary_rating', 'hematological_binary_rating', 'hematological_confident_ternary_rating_0_1', 'hematological_binary_rating_0_1', 'infertility_reasoning', 'infertility_ternary_rating', 'infertility_binary_rating', 'infertility_confident_ternary_rating_0_1', 'infertility_binary_rating_0_1', 'liver_toxicity_reasoning', 'liver_toxicity_ternary_rating', 'liver_toxicity

In [71]:
import numpy as np
from rdkit import Chem, DataStructs

# Generate fingerprints aligned with work_df row order.
# Use the same Morgan settings as the baseline.
molecules = [Chem.MolFromSmiles(s) for s in work_df[SMILES_COL]]

if any(mol is None for mol in molecules):
    raise ValueError("Invalid SMILES found; inspect before proceeding.")

reference_fps = [fp_generator.GetFingerprint(mol) for mol in molecules]
labels = work_df[TARGET].to_numpy(dtype=int)

def calculate_neighbour_score(i, k=10):
    similarities = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            reference_fps[i], reference_fps
        ),
        dtype=float
    )

    # Exclude the molecule itself.
    similarities[i] = -1.0

    eligible = np.flatnonzero(similarities > 0)
    if len(eligible) == 0:
        return np.nan

    neighbours = eligible[
        np.argsort(similarities[eligible])[::-1][:k]
    ]

    weights = similarities[neighbours]
    agreements = (labels[neighbours] == labels[i]).astype(float)

    return float(np.average(agreements, weights=weights))

# Descriptive training-set diagnostic only.
# These scores use other molecules' observed labels.
S = np.array([
    calculate_neighbour_score(i, k=10)
    for i in range(len(work_df))
])

work_df["S_neighbour_consistency"] = S

print("Number of molecules:", len(S))
print("Valid S_i scores:", int(np.isfinite(S).sum()))
print("Missing S_i scores:", int(np.isnan(S).sum()))
print("Mean S_i:", round(float(np.nanmean(S)), 4))
print("Median S_i:", round(float(np.nanmedian(S)), 4))

display(
    work_df[
        ["generic_name", SMILES_COL, TARGET, "S_neighbour_consistency"]
    ].head(10)
)

[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors
[19:03:46] WARNING: not removing hydrogen atom without neighbors


Number of molecules: 1349
Valid S_i scores: 1349
Missing S_i scores: 0
Mean S_i: 0.6249
Median S_i: 0.6486


,generic_name,smiles,liver_toxicity_binary_rating_0_1,S_neighbour_consistency
0,68GA-DOTATATE,CC(O)C(NC(=O)C1CSSCC(NC(=O)C(Cc2ccccc2)NC(=O)C...,0,0.565431
1,ABACAVIR,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,1,0.924758
2,ABALOPARATIDE,CCC(C)C(NC(=O)C(CO)NC(=O)C(CCCCN)NC(=O)CNC(=O)...,0,0.797688
3,ABEMACICLIB,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C(C...,1,0.820553
4,ABIRATERONE ACETATE,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,1,0.559641
5,ACALABRUTINIB,CC#CC(=O)N1CCCC1c1nc(-c2ccc(C(=O)Nc3ccccn3)cc2...,1,0.604714
6,ACARBOSE,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,1,0.301811
7,ACEBUTOLOL HYDROCHLORIDE,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1.Cl,1,0.497893
8,ACETAZOLAMIDE,CC(=O)Nc1nnc(S(N)(=O)=O)s1,1,0.477973
9,ACETOHYDROXAMIC ACID,CC(=O)NO,0,0.555002


In [72]:
import numpy as np
from rdkit import DataStructs

y = work_df[TARGET].to_numpy(dtype=int)
folds = work_df["fold"].to_numpy()

# Uses reference fingerprints already generated in the previous cell.
S_oof = np.full(len(work_df), np.nan)

for fold_id in np.unique(folds):
    train_idx = np.flatnonzero(folds != fold_id)
    test_idx = np.flatnonzero(folds == fold_id)

    train_fps = [reference_fps[j] for j in train_idx]
    train_y = y[train_idx]

    for i in test_idx:
        similarities = np.asarray(
            DataStructs.BulkTanimotoSimilarity(
                reference_fps[i], train_fps
            ),
            dtype=float
        )

        eligible = np.flatnonzero(similarities > 0)

        if len(eligible) == 0:
            continue

        k = min(10, len(eligible))
        neighbours = eligible[
            np.argsort(similarities[eligible])[::-1][:k]
        ]

        weights = similarities[neighbours]
        agreements = (train_y[neighbours] == y[i]).astype(float)

        S_oof[i] = np.average(agreements, weights=weights)

work_df["S_neighbour_oof"] = S_oof

print("Total molecules:", len(S_oof))
print("Valid leakage-safe scores:", np.isfinite(S_oof).sum())
print("Missing scores:", np.isnan(S_oof).sum())
print("Mean:", round(float(np.nanmean(S_oof)), 4))
print("Median:", round(float(np.nanmedian(S_oof)), 4))

Total molecules: 1349
Valid leakage-safe scores: 1349
Missing scores: 0
Mean: 0.5867
Median: 0.6036


In [73]:

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# --------------------------------------------------
# 1. Prepare aligned data
# --------------------------------------------------
y = work_df[TARGET].to_numpy(dtype=int)
folds = work_df["fold"].to_numpy(dtype=int)

assert len(X) == len(y) == len(folds) == len(work_df)
assert np.isfinite(X).all()
assert set(np.unique(y)).issubset({0, 1})

# Reuse the same Morgan fingerprint settings as the baseline.
# These bit vectors are used only for neighbour similarities.
molecules = [Chem.MolFromSmiles(s) for s in work_df[SMILES_COL]]
assert all(m is not None for m in molecules), "Invalid SMILES found."

fps = [fp_generator.GetFingerprint(m) for m in molecules]

# --------------------------------------------------
# 2. Training-only neighbour consistency
# --------------------------------------------------
def training_consistency_scores(train_indices, k=10):
    """
    For each training molecule, calculate similarity-weighted
    label agreement using other training molecules only.
    The molecule itself is excluded.
    """
    train_fps = [fps[i] for i in train_indices]
    train_y = y[train_indices]
    n = len(train_indices)

    scores = np.full(n, np.nan)

    for i in range(n):
        sims = np.asarray(
            DataStructs.BulkTanimotoSimilarity(
                train_fps[i], train_fps
            ),
            dtype=float
        )

        # Exclude self
        sims[i] = -1.0

        candidates = np.flatnonzero(sims > 0)
        if len(candidates) == 0:
            continue

        neighbours = candidates[
            np.argsort(sims[candidates])[::-1][:k]
        ]

        weights = sims[neighbours]
        agreements = (
            train_y[neighbours] == train_y[i]
        ).astype(float)

        scores[i] = np.average(agreements, weights=weights)

    # If a molecule has no similar training neighbours,
    # use neutral consistency rather than dropping it.
    scores = np.where(np.isfinite(scores), scores, 0.5)
    return scores


# --------------------------------------------------
# 3. Compare ordinary and weighted Random Forests
# --------------------------------------------------
results = []

for fold_id in sorted(np.unique(folds)):
    train_idx = np.flatnonzero(folds != fold_id)
    test_idx = np.flatnonzero(folds == fold_id)

    y_train, y_test = y[train_idx], y[test_idx]

    if len(np.unique(y_test)) < 2:
        print(f"Skipping fold {fold_id}: test fold has one class.")
        continue

    # A. Ordinary baseline
    rf_plain = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    )
    rf_plain.fit(X[train_idx], y_train)
    p_plain = rf_plain.predict_proba(X[test_idx])[:, 1]

    # B. Training-only neighbour consistency weights
    S_train = training_consistency_scores(train_idx, k=10)

    # Simple candidate weighting rule; not yet validated.
    sample_weights = 0.1 + 0.9 * S_train

    # Normalize to mean 1 to keep weight scale interpretable.
    sample_weights = sample_weights / sample_weights.mean()

    rf_weighted = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    )
    rf_weighted.fit(
        X[train_idx],
        y_train,
        sample_weight=sample_weights
    )
    p_weighted = rf_weighted.predict_proba(X[test_idx])[:, 1]

    results.append({
        "fold": int(fold_id),
        "n_test": len(test_idx),
        "plain_ROC_AUC": roc_auc_score(y_test, p_plain),
        "weighted_ROC_AUC": roc_auc_score(y_test, p_weighted),
        "plain_PR_AUC": average_precision_score(y_test, p_plain),
        "weighted_PR_AUC": average_precision_score(y_test, p_weighted),
        "mean_training_weight": sample_weights.mean(),
        "min_training_weight": sample_weights.min(),
        "max_training_weight": sample_weights.max()
    })

results_df = pd.DataFrame(results)

print("=== FOLD-WISE COMPARISON ===")
display(results_df.round(4))

metric_cols = [
    "plain_ROC_AUC",
    "weighted_ROC_AUC",
    "plain_PR_AUC",
    "weighted_PR_AUC"
]

print("\n=== MEAN ± STANDARD DEVIATION ===")
for col in metric_cols:
    print(
        f"{col}: {results_df[col].mean():.4f} "
        f"± {results_df[col].std(ddof=1):.4f}"
    )

print("\n=== WEIGHTED MINUS PLAIN ===")
print(
    "Mean ROC-AUC difference:",
    round(
        (results_df["weighted_ROC_AUC"]
         - results_df["plain_ROC_AUC"]).mean(), 4
    )
)
print(
    "Mean PR-AUC difference:",
    round(
        (results_df["weighted_PR_AUC"]
         - results_df["plain_PR_AUC"]).mean(), 4
    )
)

results_df.to_csv(
    "unitox_reliability_weighted_comparison.csv",
    index=False
)
print("\nSaved: unitox_reliability_weighted_comparison.csv")


[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:29] WARNING: not removing hydrogen atom without neighbors
[19:15:30] WARNING: not removing hydrogen atom without neighbors
[19:15:30] WARNING: not removing hydrogen atom without neighbors


=== FOLD-WISE COMPARISON ===


,fold,n_test,plain_ROC_AUC,weighted_ROC_AUC,plain_PR_AUC,weighted_PR_AUC,mean_training_weight,min_training_weight,max_training_weight
0,0,163,0.7161,0.7435,0.7364,0.7773,1.0,0.1526,1.5263
1,1,132,0.7096,0.7104,0.7638,0.7574,1.0,0.1503,1.5029
2,2,132,0.7382,0.7318,0.7780,0.7623,1.0,0.1517,1.5175
3,3,132,0.7218,0.7184,0.7584,0.7654,1.0,0.1511,1.5114
4,4,132,0.7330,0.7279,0.7643,0.7855,1.0,0.1517,1.5173
5,5,132,0.5920,0.5768,0.6404,0.6083,1.0,0.1499,1.4985
6,6,132,0.6810,0.6841,0.6947,0.7156,1.0,0.1511,1.5110
7,7,132,0.8034,0.8144,0.8358,0.8544,1.0,0.1525,1.5254
8,8,131,0.7518,0.7805,0.7597,0.7841,1.0,0.1524,1.5236
9,9,131,0.6812,0.6776,0.7025,0.6999,1.0,0.1508,1.5081



=== MEAN ± STANDARD DEVIATION ===
plain_ROC_AUC: 0.7128 ± 0.0553
weighted_ROC_AUC: 0.7165 ± 0.0640
plain_PR_AUC: 0.7434 ± 0.0535
weighted_PR_AUC: 0.7510 ± 0.0652

=== WEIGHTED MINUS PLAIN ===
Mean ROC-AUC difference: 0.0037
Mean PR-AUC difference: 0.0076

Saved: unitox_reliability_weighted_comparison.csv


In [74]:

# STEP 1: Inspect the existing fold-wise comparison
# This cell only inspects results; it does not retrain models.

if "results_df" not in globals():
    print("results_df was not found.")
    print("Available variables containing 'result' or 'fold':")
    print([
        name for name in globals()
        if "result" in name.lower() or "fold" in name.lower()
    ])
else:
    print("Columns:", results_df.columns.tolist())
    print("\nShape:", results_df.shape)
    display(results_df)


Columns: ['fold', 'n_test', 'plain_ROC_AUC', 'weighted_ROC_AUC', 'plain_PR_AUC', 'weighted_PR_AUC', 'mean_training_weight', 'min_training_weight', 'max_training_weight']

Shape: (10, 9)


,fold,n_test,plain_ROC_AUC,weighted_ROC_AUC,plain_PR_AUC,weighted_PR_AUC,mean_training_weight,min_training_weight,max_training_weight
0,0,163,0.716134,0.743531,0.736434,0.777327,1.0,0.152625,1.526253
1,1,132,0.709576,0.710391,0.763755,0.757407,1.0,0.150285,1.502855
2,2,132,0.738234,0.731827,0.778038,0.762309,1.0,0.151746,1.517462
3,3,132,0.721808,0.718430,0.758360,0.765398,1.0,0.151141,1.511415
4,4,132,0.732992,0.727866,0.764278,0.785488,1.0,0.151734,1.517335
5,5,132,0.592032,0.576771,0.640385,0.608296,1.0,0.149854,1.498542
6,6,132,0.680984,0.684119,0.694716,0.715580,1.0,0.151096,1.510962
7,7,132,0.803355,0.814422,0.835804,0.854395,1.0,0.152541,1.525406
8,8,131,0.751771,0.780468,0.759681,0.784100,1.0,0.152358,1.523576
9,9,131,0.681153,0.677610,0.702526,0.699904,1.0,0.150806,1.508057


In [76]:
import numpy as np
import pandas as pd

# Extract the actual binary target values
target_col = "liver_toxicity_binary_rating_0_1"

if target_col in work_df.columns:
    y_array = work_df[target_col].to_numpy().astype(int)
else:
    raise ValueError(
        f"Target column '{target_col}' not found. "
        f"Available columns: {work_df.columns.tolist()}"
    )

# Check alignment
X_array = np.asarray(X)
outer_fold_ids = work_df["fold"].to_numpy()
scaffold_groups = work_df["scaffold"].astype(str).to_numpy()

print("X shape:", X_array.shape)
print("y shape:", y_array.shape)
print("Target distribution:", pd.Series(y_array).value_counts().sort_index().to_dict())
print("Fold count:", len(np.unique(outer_fold_ids)))
print("All lengths aligned:",
      len(X_array) == len(y_array) == len(outer_fold_ids) == len(scaffold_groups))

X shape: (1349, 2048)
y shape: (1349,)
Target distribution: {0: 613, 1: 736}
Fold count: 10
All lengths aligned: True


In [78]:

import numpy as np
import pandas as pd
from rdkit import DataStructs
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ttest_rel, wilcoxon

# 1. Validate data
X_array = np.asarray(X)
y = np.asarray(y_array).astype(int)
fold_ids = work_df["fold"].to_numpy()
scaffolds = work_df["scaffold"].astype(str).to_numpy()

assert len(X_array) == len(y) == len(fold_ids) == len(scaffolds)
assert len(fps) == len(y)

print("Dataset:", X_array.shape)
print("Target distribution:", dict(pd.Series(y).value_counts().sort_index()))
print("Outer folds:", len(np.unique(fold_ids)))


# 2. Similarity-weighted neighbour consistency
def neighbour_consistency(query_idx, reference_idx, k=10):
    if len(reference_idx) == 0:
        return 0.5

    similarities = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            fps[int(query_idx)],
            [fps[int(j)] for j in reference_idx]
        ),
        dtype=float
    )

    valid = np.flatnonzero(similarities > 0)
    if len(valid) == 0:
        return 0.5

    top = valid[np.argsort(similarities[valid])[-k:]]
    neighbour_idx = np.asarray(reference_idx)[top]
    weights = similarities[top]

    agreement = np.where(
        y[neighbour_idx] == y[int(query_idx)], 1.0, 0.0
    )
    return float(np.average(agreement, weights=weights))


# 3. Cross-fitted scores for outer-training samples
def cross_fitted_training_scores(outer_train_idx, k=10, n_splits=5):
    scores = np.full(len(outer_train_idx), np.nan)
    groups = scaffolds[outer_train_idx]

    n_groups = len(np.unique(groups))
    if n_groups < 2:
        raise ValueError("Not enough distinct scaffolds for cross-fitting.")

    cv = GroupKFold(n_splits=min(n_splits, n_groups))

    for inner_train_pos, inner_valid_pos in cv.split(
        np.zeros(len(outer_train_idx)),
        y[outer_train_idx],
        groups=groups
    ):
        reference_idx = outer_train_idx[inner_train_pos]
        query_idx = outer_train_idx[inner_valid_pos]

        for pos, idx in zip(inner_valid_pos, query_idx):
            scores[pos] = neighbour_consistency(
                int(idx), reference_idx, k=k
            )

    if np.isnan(scores).any():
        raise RuntimeError("Some reliability scores are missing.")

    return scores


# 4. Outer scaffold-CV experiment
rows = []
unique_folds = np.sort(np.unique(fold_ids))

for fold_number, fold in enumerate(unique_folds, start=1):
    print(f"\nProcessing outer fold {fold_number}/{len(unique_folds)}...")

    test_idx = np.flatnonzero(fold_ids == fold)
    train_idx = np.flatnonzero(fold_ids != fold)

    consistency = cross_fitted_training_scores(
        train_idx, k=10, n_splits=5
    )

    # Same weighting formula as the earlier experiment
    sample_weights = 0.1 + 0.9 * consistency
    sample_weights /= sample_weights.mean()

    plain_model = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1
    )
    plain_model.fit(X_array[train_idx], y[train_idx])
    plain_prob = plain_model.predict_proba(X_array[test_idx])[:, 1]

    weighted_model = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1
    )
    weighted_model.fit(
        X_array[train_idx],
        y[train_idx],
        sample_weight=sample_weights
    )
    weighted_prob = weighted_model.predict_proba(X_array[test_idx])[:, 1]

    y_test = y[test_idx]

    rows.append({
        "fold": int(fold),
        "n_test": len(test_idx),
        "plain_ROC_AUC": roc_auc_score(y_test, plain_prob),
        "weighted_ROC_AUC": roc_auc_score(y_test, weighted_prob),
        "plain_PR_AUC": average_precision_score(y_test, plain_prob),
        "weighted_PR_AUC": average_precision_score(y_test, weighted_prob),
        "mean_training_weight": sample_weights.mean(),
        "min_training_weight": sample_weights.min(),
        "max_training_weight": sample_weights.max()
    })

crossfit_results_df = pd.DataFrame(rows)
crossfit_results_df["delta_ROC_AUC"] = (
    crossfit_results_df["weighted_ROC_AUC"]
    - crossfit_results_df["plain_ROC_AUC"]
)
crossfit_results_df["delta_PR_AUC"] = (
    crossfit_results_df["weighted_PR_AUC"]
    - crossfit_results_df["plain_PR_AUC"]
)

print("\n=== Fold-wise results ===")
display(crossfit_results_df.round(4))

print("\n=== Mean ± SD across folds ===")
for metric in [
    "plain_ROC_AUC", "weighted_ROC_AUC",
    "plain_PR_AUC", "weighted_PR_AUC"
]:
    print(
        f"{metric}: {crossfit_results_df[metric].mean():.4f} "
        f"± {crossfit_results_df[metric].std(ddof=1):.4f}"
    )

print("\n=== Paired differences: weighted minus plain ===")
for delta_metric, plain_metric, weighted_metric in [
    ("delta_ROC_AUC", "plain_ROC_AUC", "weighted_ROC_AUC"),
    ("delta_PR_AUC", "plain_PR_AUC", "weighted_PR_AUC")
]:
    differences = crossfit_results_df[delta_metric].to_numpy()

    print(
        f"{delta_metric}: mean={differences.mean():+.4f}, "
        f"SD={differences.std(ddof=1):.4f}, "
        f"improved folds={(differences > 0).sum()}/10"
    )

    print(
        "Paired t-test p-value:",
        f"{ttest_rel(crossfit_results_df[weighted_metric], crossfit_results_df[plain_metric]).pvalue:.4f}"
    )

    try:
        print(
            "Wilcoxon p-value:",
            f"{wilcoxon(differences).pvalue:.4f}"
        )
    except ValueError:
        print("Wilcoxon test unavailable.")

output_file = "unitox_inner_crossfit_weighted_comparison.csv"
crossfit_results_df.to_csv(output_file, index=False)
print("\nSaved:", output_file)


Dataset: (1349, 2048)
Target distribution: {0: np.int64(613), 1: np.int64(736)}
Outer folds: 10

Processing outer fold 1/10...

Processing outer fold 2/10...

Processing outer fold 3/10...

Processing outer fold 4/10...

Processing outer fold 5/10...

Processing outer fold 6/10...

Processing outer fold 7/10...

Processing outer fold 8/10...

Processing outer fold 9/10...

Processing outer fold 10/10...

=== Fold-wise results ===


,fold,n_test,plain_ROC_AUC,weighted_ROC_AUC,plain_PR_AUC,weighted_PR_AUC,mean_training_weight,min_training_weight,max_training_weight,delta_ROC_AUC,delta_PR_AUC
0,0,163,0.7161,0.7212,0.7364,0.7638,1.0,0.1599,1.5992,0.0051,0.0274
1,1,132,0.7096,0.6934,0.7638,0.7355,1.0,0.1603,1.6032,-0.0162,-0.0282
2,2,132,0.7382,0.7323,0.7780,0.7786,1.0,0.1625,1.6252,-0.0059,0.0005
3,3,132,0.7218,0.7281,0.7584,0.7783,1.0,0.1618,1.6179,0.0063,0.0199
4,4,132,0.7330,0.7010,0.7643,0.7477,1.0,0.1618,1.6176,-0.0320,-0.0166
5,5,132,0.5920,0.5772,0.6404,0.6200,1.0,0.1608,1.6082,-0.0148,-0.0204
6,6,132,0.6810,0.6828,0.6947,0.7131,1.0,0.1604,1.6045,0.0019,0.0184
7,7,132,0.8034,0.7960,0.8358,0.8302,1.0,0.1632,1.6318,-0.0073,-0.0056
8,8,131,0.7518,0.7592,0.7597,0.7635,1.0,0.1617,1.6171,0.0074,0.0038
9,9,131,0.6812,0.6783,0.7025,0.7125,1.0,0.1614,1.6139,-0.0028,0.0099



=== Mean ± SD across folds ===
plain_ROC_AUC: 0.7128 ± 0.0553
weighted_ROC_AUC: 0.7070 ± 0.0581
plain_PR_AUC: 0.7434 ± 0.0535
weighted_PR_AUC: 0.7443 ± 0.0558

=== Paired differences: weighted minus plain ===
delta_ROC_AUC: mean=-0.0058, SD=0.0124, improved folds=4/10
Paired t-test p-value: 0.1707
Wilcoxon p-value: 0.2754
delta_PR_AUC: mean=+0.0009, SD=0.0186, improved folds=6/10
Paired t-test p-value: 0.8787
Wilcoxon p-value: 0.9219

Saved: unitox_inner_crossfit_weighted_comparison.csv


In [79]:

import pandas as pd

print("=== DataFrames available in the notebook ===")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)
        relevant = [
            c for c in cols
            if any(term in str(c).lower() for term in [
                "dili", "inchikey", "toxicity", "rating",
                "reliab", "consisten", "label", "drug"
            ])
        ]

        if relevant:
            print(f"\nDataFrame: {name}")
            print("Shape:", obj.shape)
            print("Relevant columns:", relevant[:30])


=== DataFrames available in the notebook ===

DataFrame: __
Shape: (8, 10)
Relevant columns: ['toxicity']

DataFrame: df
Shape: (1349, 44)
Relevant columns: ['cardio_toxicity_reasoning', 'cardio_toxicity_ternary_rating', 'cardio_toxicity_binary_rating', 'cardio_toxicity_confident_ternary_rating_0_1', 'cardio_toxicity_binary_rating_0_1', 'dermatological_toxicity_reasoning', 'dermatological_toxicity_ternary_rating', 'dermatological_toxicity_binary_rating', 'dermatological_toxicity_confident_ternary_rating_0_1', 'dermatological_toxicity_binary_rating_0_1', 'hematological_ternary_rating', 'hematological_binary_rating', 'hematological_confident_ternary_rating_0_1', 'hematological_binary_rating_0_1', 'infertility_ternary_rating', 'infertility_binary_rating', 'infertility_confident_ternary_rating_0_1', 'infertility_binary_rating_0_1', 'liver_toxicity_reasoning', 'liver_toxicity_ternary_rating', 'liver_toxicity_binary_rating', 'liver_toxicity_confident_ternary_rating_0_1', 'liver_toxicity_bina

In [80]:

import numpy as np
import pandas as pd
from rdkit import DataStructs
from scipy.stats import spearmanr, mannwhitneyu, fisher_exact
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# 1. Confirm required notebook objects
# ------------------------------------------------------------
required = ["fps", "work_df", "external_eval", "oof_df"]

missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Missing notebook objects: {missing}")

target_col = "liver_toxicity_binary_rating_0_1"

assert len(fps) == len(work_df) == 1349
assert target_col in work_df.columns
assert "fold" in work_df.columns
assert "scaffold" in work_df.columns

y = work_df[target_col].to_numpy().astype(int)
fold_ids = work_df["fold"].to_numpy()
scaffolds = work_df["scaffold"].astype(str).to_numpy()

# ------------------------------------------------------------
# 2. Compute held-out neighbour consistency for every molecule.
#    Reference neighbours come only from its outer training fold.
# ------------------------------------------------------------
def heldout_neighbour_consistency(query_idx, reference_idx, k=10):
    if len(reference_idx) == 0:
        return 0.5

    sims = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            fps[int(query_idx)],
            [fps[int(j)] for j in reference_idx]
        ),
        dtype=float
    )

    valid = np.flatnonzero(sims > 0)
    if len(valid) == 0:
        return 0.5

    top = valid[np.argsort(sims[valid])[-k:]]
    neighbours = np.asarray(reference_idx)[top]
    weights = sims[top]

    agreement = (y[neighbours] == y[int(query_idx)]).astype(float)
    return float(np.average(agreement, weights=weights))


reliability = np.full(len(y), np.nan)

for fold in np.sort(np.unique(fold_ids)):
    test_idx = np.flatnonzero(fold_ids == fold)
    train_idx = np.flatnonzero(fold_ids != fold)

    for idx in test_idx:
        reliability[idx] = heldout_neighbour_consistency(
            idx, train_idx, k=10
        )

assert not np.isnan(reliability).any()

reliability_df = pd.DataFrame({
    "unitox_inchikey": work_df["inchikey"].astype(str).to_numpy(),
    "unitox_label": y,
    "outer_fold": fold_ids,
    "neighbour_reliability": reliability
})

print("Reliability scores calculated:", len(reliability_df))
print(reliability_df["neighbour_reliability"].describe().round(4))

# ------------------------------------------------------------
# 3. Inspect external evaluation schema before merging
# ------------------------------------------------------------
print("\nExternal evaluation columns:")
print(external_eval.columns.tolist())

print("\nOOF dataframe columns:")
print(oof_df.columns.tolist())

# Match using exact InChIKey where available.
if "unitox_inchikey" not in external_eval.columns:
    raise KeyError(
        "external_eval lacks unitox_inchikey. "
        "Inspect the columns printed above before proceeding."
    )

if "external_agreement_label" not in external_eval.columns:
    raise KeyError(
        "external_eval lacks external_agreement_label. "
        "Inspect the columns printed above before proceeding."
    )

evaluation = external_eval.copy()
evaluation["unitox_inchikey"] = evaluation["unitox_inchikey"].astype(str)

# Remove duplicate keys before joining, to prevent accidental row multiplication.
reliability_unique = reliability_df.drop_duplicates(
    subset="unitox_inchikey"
)

evaluation = evaluation.merge(
    reliability_unique[
        ["unitox_inchikey", "neighbour_reliability"]
    ],
    on="unitox_inchikey",
    how="inner",
    validate="many_to_one"
)

evaluation["external_agreement_label"] = pd.to_numeric(
    evaluation["external_agreement_label"], errors="coerce"
)

evaluation = evaluation.dropna(
    subset=["neighbour_reliability", "external_agreement_label"]
).copy()

evaluation["external_agreement_label"] = (
    evaluation["external_agreement_label"].astype(int)
)

print("\nMatched evaluation rows:", len(evaluation))
print(
    "External agreement distribution:",
    evaluation["external_agreement_label"].value_counts().to_dict()
)

if evaluation["external_agreement_label"].nunique() != 2:
    raise ValueError(
        "External agreement target does not contain both 0 and 1. "
        "Inspect its meaning and values before calculating AUC."
    )

# ------------------------------------------------------------
# 4. Does neighbour reliability predict external agreement?
# ------------------------------------------------------------
auc = roc_auc_score(
    evaluation["external_agreement_label"],
    evaluation["neighbour_reliability"]
)

rho, rho_p = spearmanr(
    evaluation["neighbour_reliability"],
    evaluation["external_agreement_label"]
)

agree = evaluation.loc[
    evaluation["external_agreement_label"] == 1,
    "neighbour_reliability"
]
disagree = evaluation.loc[
    evaluation["external_agreement_label"] == 0,
    "neighbour_reliability"
]

mw = mannwhitneyu(
    agree, disagree, alternative="two-sided"
)

print("\n=== Reliability validity results ===")
print(f"External agreement ROC-AUC: {auc:.4f}")
print(f"Spearman correlation: {rho:.4f}, p={rho_p:.4g}")
print(f"Median reliability when labels agree: {agree.median():.4f}")
print(f"Median reliability when labels disagree: {disagree.median():.4f}")
print(f"Mann–Whitney p-value: {mw.pvalue:.4g}")

print("\nInterpretation guide:")
if auc > 0.5 and rho_p < 0.05:
    print("Positive association detected; further validation is needed.")
elif auc < 0.5 and rho_p < 0.05:
    print("A statistically significant association in the opposite direction was detected.")
else:
    print("No statistically significant monotonic association detected.")

# Save diagnostics
evaluation.to_csv(
    "unitox_external_reliability_validation.csv",
    index=False
)
reliability_df.to_csv(
    "unitox_heldout_neighbour_reliability.csv",
    index=False
)

print("\nSaved reliability validation datasets.")


Reliability scores calculated: 1349
count    1349.0000
mean        0.5867
std         0.2276
min         0.0000
25%         0.4115
50%         0.6036
75%         0.7695
max         1.0000
Name: neighbour_reliability, dtype: float64

External evaluation columns:
['generic_name', 'smiles', 'liver_toxicity_binary_rating_0_1', 'scaffold', 'fold', 'oof_probability', 'oof_prediction', 'oof_agrees_with_label', 'oof_model_confidence', 'inchikey', 'unitox_inchikey', 'DILIConcern_clean', 'external_reference_label', 'A_external_agreement', 'external_agreement_label']

OOF dataframe columns:
['generic_name', 'smiles', 'liver_toxicity_binary_rating_0_1', 'scaffold', 'fold', 'oof_probability', 'oof_prediction', 'oof_agrees_with_label', 'oof_model_confidence', 'inchikey', 'unitox_inchikey', 'DILIConcern_clean', 'external_reference_label', 'A_external_agreement']

Matched evaluation rows: 116
External agreement distribution: {1: 100, 0: 16}

=== Reliability validity results ===
External agreement ROC-

In [81]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# Use the 116 externally evaluated compounds
ev = evaluation.copy()

# Ensure the outcome is coded 1 = agreement, 0 = disagreement
y_true = ev["external_agreement_label"].astype(int).to_numpy()

# Candidate reliability signals
signals = {
    "Neighbour reliability": "neighbour_reliability",
    "OOF model confidence": "oof_model_confidence",
    "OOF model-label agreement": "oof_agrees_with_label"
}

for col in signals.values():
    if col not in ev.columns:
        raise KeyError(
            f"Missing column: {col}. Available columns: {ev.columns.tolist()}"
        )

# Convert signals to numeric
for col in signals.values():
    ev[col] = pd.to_numeric(ev[col], errors="coerce")

ev = ev.dropna(subset=list(signals.values()) + ["external_agreement_label"])
y_true = ev["external_agreement_label"].astype(int).to_numpy()

if len(np.unique(y_true)) != 2:
    raise ValueError("Need both agreement and disagreement cases.")

# ------------------------------------------------------------
# 1. Compare discrimination
# ------------------------------------------------------------
rows = []

for name, col in signals.items():
    scores = ev[col].to_numpy()

    # AUC assumes larger score means more likely to agree.
    auc = roc_auc_score(y_true, scores)
    rho, rho_p = spearmanr(scores, y_true)

    rows.append({
        "signal": name,
        "n": len(ev),
        "ROC_AUC": auc,
        "Spearman_rho": rho,
        "Spearman_p": rho_p
    })

comparison_df = pd.DataFrame(rows)

print("=== Reliability signal comparison ===")
display(comparison_df.round(4))

# ------------------------------------------------------------
# 2. Bootstrap confidence interval for each ROC-AUC
#    Resample cases within agreement/disagreement groups.
# ------------------------------------------------------------
rng = np.random.default_rng(42)
B = 5000
boot_rows = []

agree_idx = np.flatnonzero(y_true == 1)
disagree_idx = np.flatnonzero(y_true == 0)

for name, col in signals.items():
    scores = ev[col].to_numpy()
    boot_aucs = []

    for _ in range(B):
        sampled_agree = rng.choice(
            agree_idx, size=len(agree_idx), replace=True
        )
        sampled_disagree = rng.choice(
            disagree_idx, size=len(disagree_idx), replace=True
        )
        sampled_idx = np.concatenate([sampled_agree, sampled_disagree])

        boot_aucs.append(
            roc_auc_score(y_true[sampled_idx], scores[sampled_idx])
        )

    lower, upper = np.percentile(boot_aucs, [2.5, 97.5])

    boot_rows.append({
        "signal": name,
        "ROC_AUC": roc_auc_score(y_true, scores),
        "bootstrap_95CI_lower": lower,
        "bootstrap_95CI_upper": upper
    })

bootstrap_df = pd.DataFrame(boot_rows)

print("\n=== Stratified bootstrap 95% confidence intervals ===")
display(bootstrap_df.round(4))

# ------------------------------------------------------------
# 3. Reliability by score quartile
# ------------------------------------------------------------
ev["reliability_quartile"] = pd.qcut(
    ev["neighbour_reliability"],
    q=4,
    labels=False,
    duplicates="drop"
) + 1

quartile_df = (
    ev.groupby("reliability_quartile", observed=True)
      .agg(
          n=("external_agreement_label", "size"),
          agreement_rate=("external_agreement_label", "mean"),
          median_score=("neighbour_reliability", "median")
      )
      .reset_index()
)

print("\n=== External agreement rate by neighbour-reliability quartile ===")
display(quartile_df.round(4))

# Save outputs
comparison_df.to_csv("unitox_reliability_signal_comparison.csv", index=False)
bootstrap_df.to_csv("unitox_reliability_auc_bootstrap.csv", index=False)
quartile_df.to_csv("unitox_reliability_quartile_analysis.csv", index=False)

print("\nSaved comparison, bootstrap, and quartile results.")


=== Reliability signal comparison ===


,signal,n,ROC_AUC,Spearman_rho,Spearman_p
0,Neighbour reliability,116,0.6200,0.1433,0.1248
1,OOF model confidence,116,0.5250,0.0299,0.7503
2,OOF model-label agreement,116,0.5325,0.0481,0.6082



=== Stratified bootstrap 95% confidence intervals ===


,signal,ROC_AUC,bootstrap_95CI_lower,bootstrap_95CI_upper
0,Neighbour reliability,0.6200,0.4712,0.7694
1,OOF model confidence,0.5250,0.3969,0.6541
2,OOF model-label agreement,0.5325,0.4087,0.6612



=== External agreement rate by neighbour-reliability quartile ===


,reliability_quartile,n,agreement_rate,median_score
0,1,29,0.7586,0.3301
1,2,29,0.8966,0.5065
2,3,29,0.8621,0.6335
3,4,29,0.9310,0.8240



Saved comparison, bootstrap, and quartile results.


In [82]:

import pandas as pd
import numpy as np

# Inspect the exact InChIKey match dataset
print("analysis_df shape:", analysis_df.shape)
print("\nAvailable columns:")
print(analysis_df.columns.tolist())

print("\nDILIrank category distribution:")
print(analysis_df["DILIConcern_clean"].value_counts(dropna=False))

print("\nUniTox label distribution:")
print(analysis_df["unitox_label"].value_counts(dropna=False))

print("\nDILIrank category vs UniTox label:")
display(pd.crosstab(
    analysis_df["DILIConcern_clean"],
    analysis_df["unitox_label"],
    margins=True,
    dropna=False
))

print("\nOrdinal mapping already used:")
if "dili_ordinal" in analysis_df.columns:
    display(pd.crosstab(
        analysis_df["DILIConcern_clean"],
        analysis_df["dili_ordinal"],
        margins=True,
        dropna=False
    ))
else:
    print("No dili_ordinal column found.")


analysis_df shape: (253, 16)

Available columns:
['generic_name', 'smiles', 'liver_toxicity_binary_rating_0_1', 'normalized_name', 'LabelCompoundName', 'SMILES', 'DILIConcern', 'SeverityClass', 'AdjudicatedDILI', 'unitox_inchikey', 'dilirank_inchikey', 'structure_status', 'identity_class', 'DILIConcern_clean', 'dili_ordinal', 'unitox_label']

DILIrank category distribution:
DILIConcern_clean
Less    137
No       61
Most     55
Name: count, dtype: Int64

UniTox label distribution:
unitox_label
1    167
0     86
Name: count, dtype: int64

DILIrank category vs UniTox label:


unitox_label,0,1,All
DILIConcern_clean,,,
Less,37,100,137
Most,2,53,55
No,47,14,61
All,86,167,253



Ordinal mapping already used:


dili_ordinal,0,1,2,All
DILIConcern_clean,,,,
Less,0,137,0,137
Most,0,0,55,55
No,61,0,0,61
All,61,137,55,253


In [84]:

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kruskal, mannwhitneyu
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# 1. Prepare reliability scores and diagnose duplicate keys
# ------------------------------------------------------------
rel = reliability_df.copy()
external_253 = analysis_df.copy()

rel["unitox_inchikey"] = rel["unitox_inchikey"].astype(str).str.strip()
external_253["unitox_inchikey"] = (
    external_253["unitox_inchikey"].astype(str).str.strip()
)

print("Reliability rows:", len(rel))
print("Unique reliability InChIKeys:", rel["unitox_inchikey"].nunique())
print("Duplicate-key rows:", rel["unitox_inchikey"].duplicated().sum())

# Aggregate duplicate records by mean reliability score.
# This avoids multiplying external rows during the merge.
rel_unique = (
    rel.groupby("unitox_inchikey", as_index=False)
       .agg(
           neighbour_reliability=("neighbour_reliability", "mean"),
           n_unitox_records=("neighbour_reliability", "size")
       )
)

print("Unique aggregated keys:", len(rel_unique))
print("Keys representing multiple UniTox records:",
      int((rel_unique["n_unitox_records"] > 1).sum()))

# ------------------------------------------------------------
# 2. Merge the exact-match reference dataset
# ------------------------------------------------------------
external_253 = external_253.merge(
    rel_unique,
    on="unitox_inchikey",
    how="inner",
    validate="one_to_one"
)

print("\nMatched exact-key rows:", len(external_253))
print(external_253["DILIConcern_clean"].value_counts())

if len(external_253) == 0:
    raise ValueError("No exact-key matches found after merge.")

# ------------------------------------------------------------
# 3. Ordinal association: No < Less < Most
# ------------------------------------------------------------
category_order = {"No": 0, "Less": 1, "Most": 2}
external_253["dili_order"] = (
    external_253["DILIConcern_clean"].map(category_order)
)

if external_253["dili_order"].isna().any():
    raise ValueError("Unexpected DILIrank categories found.")

rho, rho_p = spearmanr(
    external_253["neighbour_reliability"],
    external_253["dili_order"]
)

print("\n=== Ordinal association ===")
print(f"Spearman rho: {rho:.4f}")
print(f"Spearman p-value: {rho_p:.6f}")

category_summary = (
    external_253.groupby("DILIConcern_clean", observed=True)
    ["neighbour_reliability"]
    .agg(["count", "mean", "std", "median"])
    .reindex(["No", "Less", "Most"])
)

print("\n=== Reliability by DILIrank category ===")
display(category_summary.round(4))

groups = [
    external_253.loc[
        external_253["DILIConcern_clean"] == cat,
        "neighbour_reliability"
    ].to_numpy()
    for cat in ["No", "Less", "Most"]
]

kw = kruskal(*groups)
print(f"Kruskal-Wallis statistic: {kw.statistic:.4f}")
print(f"Kruskal-Wallis p-value: {kw.pvalue:.6f}")

# ------------------------------------------------------------
# 4. Extreme categories: No versus Most only
# ------------------------------------------------------------
extremes_253 = external_253[
    external_253["DILIConcern_clean"].isin(["No", "Most"])
].copy()

extremes_253["most_category"] = (
    extremes_253["DILIConcern_clean"] == "Most"
).astype(int)

auc_extreme = roc_auc_score(
    extremes_253["most_category"],
    extremes_253["neighbour_reliability"]
)

no_scores = extremes_253.loc[
    extremes_253["DILIConcern_clean"] == "No",
    "neighbour_reliability"
]
most_scores = extremes_253.loc[
    extremes_253["DILIConcern_clean"] == "Most",
    "neighbour_reliability"
]

mw = mannwhitneyu(
    most_scores, no_scores, alternative="two-sided"
)

print("\n=== Extreme-category analysis ===")
print("Number of compounds:", len(extremes_253))
print(f"ROC-AUC: {auc_extreme:.4f}")
print(f"Median reliability, No: {no_scores.median():.4f}")
print(f"Median reliability, Most: {most_scores.median():.4f}")
print(f"Mann-Whitney p-value: {mw.pvalue:.6f}")

# Save outputs
external_253.to_csv(
    "unitox_253_exact_match_reliability_validation.csv",
    index=False
)
category_summary.to_csv(
    "unitox_reliability_by_dilirank_category.csv"
)

print("\nSaved updated validation results.")


Reliability rows: 1349
Unique reliability InChIKeys: 1348
Duplicate-key rows: 1
Unique aggregated keys: 1348
Keys representing multiple UniTox records: 1

Matched exact-key rows: 253
DILIConcern_clean
Less    137
No       61
Most     55
Name: count, dtype: Int64

=== Ordinal association ===
Spearman rho: 0.2303
Spearman p-value: 0.000219

=== Reliability by DILIrank category ===


,count,mean,std,median
DILIConcern_clean,,,,
No,61,0.5141,0.1857,0.5072
Less,137,0.5788,0.2044,0.5964
Most,55,0.6535,0.1946,0.6402


Kruskal-Wallis statistic: 13.3734
Kruskal-Wallis p-value: 0.001247

=== Extreme-category analysis ===
Number of compounds: 116
ROC-AUC: 0.6987
Median reliability, No: 0.5072
Median reliability, Most: 0.6402
Mann-Whitney p-value: 0.000231

Saved updated validation results.


In [85]:

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kruskal

# Uses external_253 from the previous successful cell
data = external_253.copy()

data["dili_order"] = data["DILIConcern_clean"].map(
    {"No": 0, "Less": 1, "Most": 2}
).astype(int)

data["unitox_label"] = data["unitox_label"].astype(int)

print("=== Category distribution within UniTox label ===")
display(pd.crosstab(
    data["unitox_label"],
    data["DILIConcern_clean"],
    margins=True
))

# ------------------------------------------------------------
# 1. Within-label ordinal association
# ------------------------------------------------------------
rows = []

for label in sorted(data["unitox_label"].unique()):
    group = data[data["unitox_label"] == label]

    rho, p = spearmanr(
        group["neighbour_reliability"],
        group["dili_order"]
    )

    rows.append({
        "unitox_label": label,
        "n": len(group),
        "spearman_rho": rho,
        "p_value": p,
        "n_categories": group["DILIConcern_clean"].nunique()
    })

within_label_results = pd.DataFrame(rows)

print("\n=== Reliability vs DILIrank category within each UniTox label ===")
display(within_label_results.round(4))

# ------------------------------------------------------------
# 2. Reliability by category separately for each UniTox label
# ------------------------------------------------------------
stratified_summary = (
    data.groupby(
        ["unitox_label", "DILIConcern_clean"],
        observed=True
    )["neighbour_reliability"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

print("\n=== Stratified reliability summary ===")
display(stratified_summary.round(4))

# ------------------------------------------------------------
# 3. Within-label Kruskal-Wallis tests where possible
# ------------------------------------------------------------
print("\n=== Within-label category tests ===")

for label in sorted(data["unitox_label"].unique()):
    group = data[data["unitox_label"] == label]

    category_scores = [
        group.loc[
            group["DILIConcern_clean"] == category,
            "neighbour_reliability"
        ].to_numpy()
        for category in ["No", "Less", "Most"]
    ]

    category_scores = [scores for scores in category_scores if len(scores) > 0]

    if len(category_scores) >= 2:
        result = kruskal(*category_scores)
        print(
            f"UniTox label {label}: "
            f"H={result.statistic:.4f}, p={result.pvalue:.6f}"
        )
    else:
        print(
            f"UniTox label {label}: insufficient category variation."
        )

within_label_results.to_csv(
    "unitox_within_label_reliability_validation.csv",
    index=False
)

stratified_summary.to_csv(
    "unitox_stratified_reliability_summary.csv",
    index=False
)

print("\nSaved within-label validation results.")


=== Category distribution within UniTox label ===


DILIConcern_clean,Less,Most,No,All
unitox_label,,,,
0,37,2,47,86
1,100,53,14,167
All,137,55,61,253



=== Reliability vs DILIrank category within each UniTox label ===


,unitox_label,n,spearman_rho,p_value,n_categories
0,0,86,-0.2185,0.0433,3
1,1,167,0.1432,0.0648,3



=== Stratified reliability summary ===


,unitox_label,DILIConcern_clean,count,mean,median,std
0,0,Less,37,0.4356,0.4528,0.1954
1,0,Most,2,0.2742,0.2742,0.0291
2,0,No,47,0.5077,0.5072,0.1853
3,1,Less,100,0.6318,0.6453,0.1817
4,1,Most,53,0.6678,0.6601,0.1832
5,1,No,14,0.5356,0.5386,0.1921



=== Within-label category tests ===
UniTox label 0: H=5.3538, p=0.068776
UniTox label 1: H=4.4749, p=0.106732

Saved within-label validation results.


In [87]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
from scipy.stats import chi2

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------
reg_df = external_253.copy()

reg_df["dili_order"] = reg_df["DILIConcern_clean"].map(
    {"No": 0, "Less": 1, "Most": 2}
)

reg_df["unitox_label"] = reg_df["unitox_label"].astype(int)

reg_df = reg_df.dropna(
    subset=["dili_order", "unitox_label", "neighbour_reliability"]
).copy()

scaler = StandardScaler()
reg_df["reliability_z"] = scaler.fit_transform(
    reg_df[["neighbour_reliability"]]
).ravel()

y_cat = reg_df["dili_order"].astype(int).to_numpy()

# Model 1: UniTox binary label only
X_base = reg_df[["unitox_label"]].to_numpy()

# Model 2: UniTox label plus neighbour reliability
X_extended = reg_df[
    ["unitox_label", "reliability_z"]
].to_numpy()

# ------------------------------------------------------------
# 2. Fit multinomial logistic regression
# ------------------------------------------------------------
base_model = LogisticRegression(
    C=1e6, max_iter=5000, solver="lbfgs"
)
extended_model = LogisticRegression(
    C=1e6, max_iter=5000, solver="lbfgs"
)

base_model.fit(X_base, y_cat)
extended_model.fit(X_extended, y_cat)

base_prob = base_model.predict_proba(X_base)
extended_prob = extended_model.predict_proba(X_extended)

base_loss = log_loss(y_cat, base_prob, labels=[0, 1, 2])
extended_loss = log_loss(y_cat, extended_prob, labels=[0, 1, 2])

print("Observations:", len(reg_df))
print("\n=== In-sample fit comparison ===")
print(f"UniTox label only log-loss: {base_loss:.4f}")
print(f"Label + reliability log-loss: {extended_loss:.4f}")
print(f"Log-loss change: {extended_loss - base_loss:+.4f}")

# ------------------------------------------------------------
# 3. Inspect reliability coefficient by category
# ------------------------------------------------------------
coef_df = pd.DataFrame(
    extended_model.coef_,
    columns=["unitox_label", "reliability_z"],
    index=extended_model.classes_
)

coef_df.index.name = "DILIrank_category_code"

print("\n=== Multinomial coefficients ===")
display(coef_df.round(4))

print("\nCategory codes: 0=No, 1=Less, 2=Most")
print(
    "Important: these are in-sample associations, not validated "
    "predictive improvements."
)

# Save descriptive results
coef_df.to_csv("unitox_multinomial_reliability_coefficients.csv")

pd.DataFrame({
    "model": ["UniTox label only", "Label + reliability"],
    "in_sample_log_loss": [base_loss, extended_loss]
}).to_csv("unitox_multinomial_model_comparison.csv", index=False)

print("\nSaved regression outputs.")


Observations: 253

=== In-sample fit comparison ===
UniTox label only log-loss: 0.8456
Label + reliability log-loss: 0.8424
Log-loss change: -0.0032

=== Multinomial coefficients ===


,unitox_label,reliability_z
DILIrank_category_code,,
0,-2.2191,-0.0156
1,0.0561,-0.1038
2,2.1630,0.1195



Category codes: 0=No, 1=Less, 2=Most
Important: these are in-sample associations, not validated predictive improvements.

Saved regression outputs.


In [89]:

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score, balanced_accuracy_score

# ------------------------------------------------------------
# 1. Recover scaffold information from the original UniTox data
# ------------------------------------------------------------
cv_df = external_253.copy()

cv_df["unitox_inchikey"] = cv_df["unitox_inchikey"].astype(str)
work_keys = work_df[["inchikey", "scaffold"]].copy()
work_keys["inchikey"] = work_keys["inchikey"].astype(str)

# Resolve duplicate InChIKeys by assigning one scaffold per key.
# If duplicates disagree, stop rather than silently choose one.
scaffold_counts = work_keys.groupby("inchikey")["scaffold"].nunique()
conflicting_keys = scaffold_counts[scaffold_counts > 1]

if len(conflicting_keys):
    raise ValueError(
        "Some duplicate InChIKeys have conflicting scaffolds: "
        f"{conflicting_keys.index.tolist()[:10]}"
    )

work_keys = work_keys.drop_duplicates("inchikey")

cv_df = cv_df.merge(
    work_keys,
    left_on="unitox_inchikey",
    right_on="inchikey",
    how="left",
    validate="many_to_one"
)

# Prepare target and predictors
cv_df["dili_order"] = cv_df["DILIConcern_clean"].map(
    {"No": 0, "Less": 1, "Most": 2}
)

cv_df = cv_df.dropna(
    subset=[
        "dili_order",
        "unitox_label",
        "neighbour_reliability",
        "scaffold"
    ]
).reset_index(drop=True)

y_cv = cv_df["dili_order"].astype(int).to_numpy()
label_feature = cv_df["unitox_label"].astype(int).to_numpy()
reliability_feature = cv_df["neighbour_reliability"].to_numpy()
groups = cv_df["scaffold"].astype(str).to_numpy()

classes = np.array([0, 1, 2])

print("Samples:", len(cv_df))
print("Unique scaffolds:", len(np.unique(groups)))
print("Category counts:", dict(pd.Series(y_cv).value_counts().sort_index()))

# ------------------------------------------------------------
# 2. Scaffold-grouped cross-validation
# ------------------------------------------------------------
cv = GroupKFold(n_splits=5)
oof_base = np.full((len(cv_df), 3), np.nan)
oof_extended = np.full((len(cv_df), 3), np.nan)
fold_rows = []

def aligned_probabilities(model, X_test):
    raw = model.predict_proba(X_test)
    aligned = np.full((len(X_test), 3), 1e-15)

    for j, category in enumerate(model.classes_):
        aligned[:, np.where(classes == category)[0][0]] = raw[:, j]

    return aligned / aligned.sum(axis=1, keepdims=True)

for fold, (train_idx, test_idx) in enumerate(
    cv.split(np.zeros(len(cv_df)), y_cv, groups=groups),
    start=1
):
    y_train, y_test = y_cv[train_idx], y_cv[test_idx]

    # Baseline: original UniTox binary label only
    X_base_train = label_feature[train_idx].reshape(-1, 1)
    X_base_test = label_feature[test_idx].reshape(-1, 1)

    base_model = LogisticRegression(max_iter=3000, C=1.0)
    base_model.fit(X_base_train, y_train)

    # Extended: UniTox label plus neighbour reliability
    scaler = StandardScaler()
    rel_train = scaler.fit_transform(
        reliability_feature[train_idx].reshape(-1, 1)
    ).ravel()
    rel_test = scaler.transform(
        reliability_feature[test_idx].reshape(-1, 1)
    ).ravel()

    X_ext_train = np.column_stack([
        label_feature[train_idx], rel_train
    ])
    X_ext_test = np.column_stack([
        label_feature[test_idx], rel_test
    ])

    extended_model = LogisticRegression(max_iter=3000, C=1.0)
    extended_model.fit(X_ext_train, y_train)

    p_base = aligned_probabilities(base_model, X_base_test)
    p_ext = aligned_probabilities(extended_model, X_ext_test)

    oof_base[test_idx] = p_base
    oof_extended[test_idx] = p_ext

    fold_rows.append({
        "fold": fold,
        "n_test": len(test_idx),
        "baseline_log_loss": log_loss(y_test, p_base, labels=classes),
        "extended_log_loss": log_loss(y_test, p_ext, labels=classes),
        "baseline_accuracy": accuracy_score(y_test, p_base.argmax(axis=1)),
        "extended_accuracy": accuracy_score(y_test, p_ext.argmax(axis=1)),
        "baseline_balanced_accuracy": balanced_accuracy_score(
            y_test, p_base.argmax(axis=1)
        ),
        "extended_balanced_accuracy": balanced_accuracy_score(
            y_test, p_ext.argmax(axis=1)
        )
    })

fold_results = pd.DataFrame(fold_rows)
fold_results["log_loss_change"] = (
    fold_results["extended_log_loss"] - fold_results["baseline_log_loss"]
)

print("\n=== Fold-wise results ===")
display(fold_results.round(4))

print("\n=== Pooled out-of-fold results ===")
base_loss = log_loss(y_cv, oof_base, labels=classes)
ext_loss = log_loss(y_cv, oof_extended, labels=classes)

print(f"Baseline log-loss: {base_loss:.4f}")
print(f"Extended log-loss: {ext_loss:.4f}")
print(f"Change (extended - baseline): {ext_loss - base_loss:+.4f}")

print(f"Baseline accuracy: {accuracy_score(y_cv, oof_base.argmax(axis=1)):.4f}")
print(f"Extended accuracy: {accuracy_score(y_cv, oof_extended.argmax(axis=1)):.4f}")
print(
    "Baseline balanced accuracy:",
    f"{balanced_accuracy_score(y_cv, oof_base.argmax(axis=1)):.4f}"
)
print(
    "Extended balanced accuracy:",
    f"{balanced_accuracy_score(y_cv, oof_extended.argmax(axis=1)):.4f}"
)

fold_results.to_csv(
    "unitox_scaffold_cv_reliability_incremental_value.csv", index=False
)

print("\nSaved fold results.")


Samples: 253
Unique scaffolds: 177
Category counts: {0: np.int64(61), 1: np.int64(137), 2: np.int64(55)}

=== Fold-wise results ===


,fold,n_test,baseline_log_loss,extended_log_loss,baseline_accuracy,extended_accuracy,baseline_balanced_accuracy,extended_balanced_accuracy,log_loss_change
0,1,51,1.0158,1.0370,0.3333,0.3922,0.3333,0.3839,0.0213
1,2,51,0.8226,0.8518,0.5490,0.5098,0.4847,0.4128,0.0292
2,3,51,0.7945,0.7848,0.6078,0.6078,0.5111,0.5111,-0.0097
3,4,50,0.9282,0.9506,0.6000,0.6000,0.5071,0.5071,0.0224
4,5,50,0.7841,0.7903,0.6400,0.6600,0.5126,0.5227,0.0061



=== Pooled out-of-fold results ===
Baseline log-loss: 0.8691
Extended log-loss: 0.8830
Change (extended - baseline): +0.0138
Baseline accuracy: 0.5455
Extended accuracy: 0.5534
Baseline balanced accuracy: 0.4358
Extended balanced accuracy: 0.4437

Saved fold results.


In [90]:

# STEP 1: AUDIT CURRENT NOTEBOOK OBJECTS
# Read-only: does not modify datasets or train models.

import numpy as np
import pandas as pd

print("=" * 75)
print("RELIABILITY-AWARE UNITox PROJECT — NOTEBOOK AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# A. Check the main DataFrames used in our previous experiments
# ------------------------------------------------------------
key_names = [
    "df",
    "work_df",
    "reliability_df",
    "analysis_df",
    "external_eval",
    "external_253",
    "cv_df",
    "oof_df",
    "dilirank",
    "fold_results",
]

relevant_terms = [
    "inchikey",
    "smiles",
    "generic_name",
    "scaffold",
    "fold",
    "liver_toxicity",
    "unitox_label",
    "dili",
    "reliability",
    "oof",
    "probability",
    "prediction",
]

print("\n[A] Key DataFrames")
found_dataframes = []

for name in key_names:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        found_dataframes.append(name)

        relevant_columns = [
            col for col in obj.columns
            if any(term in str(col).lower() for term in relevant_terms)
        ]

        print(f"\n{name}: shape={obj.shape}")
        print("Relevant columns:", relevant_columns)

        # Report relevant category distributions, without printing rows.
        for col in [
            "liver_toxicity_binary_rating_0_1",
            "unitox_label",
            "DILIConcern_clean",
            "outer_fold",
            "fold",
        ]:
            if col in obj.columns:
                print(f"  {col} counts:")
                print(obj[col].value_counts(dropna=False).head(10).to_dict())

    elif obj is not None:
        print(f"\n{name}: exists, but type is {type(obj).__name__}")

    else:
        print(f"\n{name}: not found")

# ------------------------------------------------------------
# B. Locate available molecular feature matrices and fold arrays
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("[B] Candidate feature matrices, labels and fold objects")
print("=" * 75)

tokens = (
    "morgan",
    "fingerprint",
    "fingerprints",
    "features",
    "fold",
    "liver_y",
    "y_liver",
    "y_binary",
    "y_target",
    "x_train",
    "x_morgan",
    "morgan_fp",
)

seen = set()

for name, obj in list(globals().items()):
    if name.startswith("_") or name in seen:
        continue

    name_lower = name.lower()

    if not any(token in name_lower for token in tokens):
        continue

    if isinstance(obj, np.ndarray):
        print(
            f"{name}: NumPy array | shape={obj.shape} | "
            f"dtype={obj.dtype}"
        )
        seen.add(name)

    elif isinstance(obj, pd.DataFrame):
        if name not in key_names:
            print(f"{name}: DataFrame | shape={obj.shape}")
            print("  Columns:", list(obj.columns)[:15])
            seen.add(name)

    elif isinstance(obj, (list, tuple)):
        if any(token in name_lower for token in ("fold", "morgan", "fingerprint")):
            print(f"{name}: {type(obj).__name__} | length={len(obj)}")
            seen.add(name)

# ------------------------------------------------------------
# C. Verify basic structure of the main UniTox table
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("[C] Main UniTox table checks")
print("=" * 75)

if "work_df" in globals() and isinstance(work_df, pd.DataFrame):
    print("work_df rows:", len(work_df))
    print("work_df columns:", list(work_df.columns))

    if "scaffold" in work_df.columns:
        print("Unique scaffolds:", work_df["scaffold"].nunique(dropna=True))
        print("Missing scaffold values:", work_df["scaffold"].isna().sum())
    else:
        print("WARNING: work_df does not contain 'scaffold'.")

    if "liver_toxicity_binary_rating_0_1" in work_df.columns:
        print(
            "Liver target distribution:",
            work_df["liver_toxicity_binary_rating_0_1"]
            .value_counts(dropna=False)
            .to_dict()
        )

    if "inchikey" in work_df.columns:
        print("Missing InChIKeys:", work_df["inchikey"].isna().sum())
        print(
            "Unique non-missing InChIKeys:",
            work_df["inchikey"].nunique(dropna=True)
        )
else:
    print("WARNING: work_df is unavailable or is not a DataFrame.")

# ------------------------------------------------------------
# D. Verify the external analysis table
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("[D] External-validation data checks")
print("=" * 75)

for name in ["analysis_df", "external_253", "cv_df"]:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}: {len(obj)} rows")

        for col in [
            "unitox_inchikey",
            "inchikey",
            "DILIConcern_clean",
            "unitox_label",
            "neighbour_reliability",
            "scaffold",
        ]:
            if col in obj.columns:
                print(
                    f"  {col}: missing={obj[col].isna().sum()}, "
                    f"unique={obj[col].nunique(dropna=True)}"
                )

print("\n" + "=" * 75)
print("AUDIT COMPLETE — no datasets were modified.")
print("=" * 75)


RELIABILITY-AWARE UNITox PROJECT — NOTEBOOK AUDIT

[A] Key DataFrames

df: shape=(1349, 44)
Relevant columns: ['generic_name', 'liver_toxicity_reasoning', 'liver_toxicity_ternary_rating', 'liver_toxicity_binary_rating', 'liver_toxicity_confident_ternary_rating_0_1', 'liver_toxicity_binary_rating_0_1', 'smiles', 'all_smiles', 'scaffold']
  liver_toxicity_binary_rating_0_1 counts:
{1: 736, 0: 613}

work_df: shape=(1349, 49)
Relevant columns: ['generic_name', 'liver_toxicity_reasoning', 'liver_toxicity_ternary_rating', 'liver_toxicity_binary_rating', 'liver_toxicity_confident_ternary_rating_0_1', 'liver_toxicity_binary_rating_0_1', 'smiles', 'all_smiles', 'scaffold', 'fold', 'smiles_valid', 'inchikey', 'S_neighbour_oof']
  liver_toxicity_binary_rating_0_1 counts:
{1: 736, 0: 613}
  fold counts:
{0: 163, 7: 132, 2: 132, 3: 132, 1: 132, 4: 132, 6: 132, 5: 132, 8: 131, 9: 131}

reliability_df: shape=(1349, 4)
Relevant columns: ['unitox_inchikey', 'unitox_label', 'outer_fold', 'neighbour_reli

In [91]:

# STEP 2A: FOLD, FEATURE AND RELIABILITY ALIGNMENT AUDIT
# Read-only diagnostic. No models are trained and no data are modified.

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 2A — DATA AND RELIABILITY INTEGRITY")
print("=" * 80)

# ------------------------------------------------------------
# A. Check fingerprint and target alignment
# ------------------------------------------------------------
print("\n[A] Fingerprint and target alignment")

N = len(work_df)

print("Number of samples:", N)
print("Morgan shape:", morgan.shape)
print("X_morgan shape:", X_morgan.shape)

if morgan.shape == X_morgan.shape:
    print("Morgan and X_morgan numerically identical:",
          np.array_equal(morgan, X_morgan))
else:
    print("WARNING: fingerprint shapes do not match.")

target = work_df["liver_toxicity_binary_rating_0_1"].to_numpy()

print("Target length:", len(target))
print("Target values:", np.unique(target))

if "df" in globals() and len(df) == N:
    print(
        "df and work_df liver labels aligned:",
        np.array_equal(
            df["liver_toxicity_binary_rating_0_1"].to_numpy(),
            target
        )
    )

# ------------------------------------------------------------
# B. Verify fold arrays against work_df
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[B] Fold assignment alignment")

canonical_fold = work_df["fold"].to_numpy()

fold_variables = [
    "outer_fold_ids",
    "fold_ids",
    "folds"
]

for name in fold_variables:
    obj = globals().get(name)

    if obj is None:
        print(f"{name}: unavailable")
        continue

    arr = np.asarray(obj)

    if arr.ndim == 1 and len(arr) == N:
        print(
            f"{name}: shape={arr.shape}, "
            f"matches work_df['fold'] = "
            f"{np.array_equal(arr, canonical_fold)}"
        )
    else:
        print(f"{name}: shape={arr.shape}, cannot compare directly")

print("\nFold sizes:")
print(work_df["fold"].value_counts().sort_index().to_dict())

print("\nScaffold separation check:")

scaffold_fold_counts = (
    work_df.groupby("scaffold")["fold"].nunique()
)

print(
    "Scaffolds assigned to more than one fold:",
    int((scaffold_fold_counts > 1).sum())
)

print(
    "Maximum number of folds occupied by one scaffold:",
    int(scaffold_fold_counts.max())
)

print("\nPer-fold scaffold counts:")
print(
    work_df.groupby("fold")["scaffold"]
    .nunique()
    .sort_index()
    .to_dict()
)

# ------------------------------------------------------------
# C. Inspect the structure of fold_indices
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[C] Existing fold_indices structure")

if "fold_indices" in globals():
    print("Number of entries:", len(fold_indices))

    if len(fold_indices) > 0:
        first = fold_indices[0]
        print("First entry type:", type(first).__name__)

        if isinstance(first, (tuple, list)):
            print("Number of parts in first entry:", len(first))

            for i, part in enumerate(first):
                part_array = np.asarray(part)
                print(
                    f"  Part {i}: type={type(part).__name__}, "
                    f"shape={part_array.shape}, "
                    f"dtype={part_array.dtype}"
                )
        else:
            print("First entry shape:", np.asarray(first).shape)
else:
    print("fold_indices unavailable")

# ------------------------------------------------------------
# D. Investigate duplicate chemical identifiers
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[D] Duplicate InChIKey investigation")

duplicate_mask = work_df["inchikey"].duplicated(keep=False)

duplicate_cols = [
    "generic_name",
    "inchikey",
    "liver_toxicity_binary_rating_0_1",
    "scaffold",
    "fold",
    "S_neighbour_consistency",
    "S_neighbour_oof"
]

duplicate_cols = [
    col for col in duplicate_cols if col in work_df.columns
]

duplicates = work_df.loc[
    duplicate_mask, duplicate_cols
].sort_values("inchikey")

print("Rows involved in duplicate InChIKeys:", len(duplicates))
print(duplicates.to_string(index=False))

# ------------------------------------------------------------
# E. Compare the two neighbour reliability columns
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[E] Neighbour score comparison")

score_columns = [
    col for col in [
        "S_neighbour_consistency",
        "S_neighbour_oof"
    ]
    if col in work_df.columns
]

for col in score_columns:
    s = pd.to_numeric(work_df[col], errors="coerce")

    print(f"\n{col}")
    print(s.describe().round(5).to_string())
    print("Missing:", int(s.isna().sum()))

if (
    "S_neighbour_consistency" in work_df.columns
    and "S_neighbour_oof" in work_df.columns
):
    print(
        "\nCorrelation between the two scores:",
        work_df[
            ["S_neighbour_consistency", "S_neighbour_oof"]
        ].corr().iloc[0, 1]
    )

    print(
        "Exact equality between the columns:",
        np.allclose(
            work_df["S_neighbour_consistency"].to_numpy(),
            work_df["S_neighbour_oof"].to_numpy(),
            equal_nan=True
        )
    )

# ------------------------------------------------------------
# F. Compare reliability_df against work_df by chemical key
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[F] Reliability table alignment")

rel = reliability_df.copy()

rel["unitox_inchikey"] = rel["unitox_inchikey"].astype(str)

rel_stats = (
    rel.groupby("unitox_inchikey")["neighbour_reliability"]
    .agg(["count", "nunique", "mean", "std"])
)

print("Reliability rows:", len(rel))
print("Unique reliability keys:", len(rel_stats))
print(
    "Keys with inconsistent reliability scores:",
    int((rel_stats["nunique"] > 1).sum())
)

# Compare the working table's OOF score against the reliability table.
if "S_neighbour_oof" in work_df.columns:
    work_score_stats = (
        work_df.assign(
            inchikey=work_df["inchikey"].astype(str)
        )
        .groupby("inchikey")["S_neighbour_oof"]
        .agg(["count", "nunique", "mean", "std"])
    )

    common_keys = work_score_stats.index.intersection(
        rel_stats.index
    )

    left = work_score_stats.loc[common_keys, "mean"].to_numpy()
    right = rel_stats.loc[common_keys, "mean"].to_numpy()

    print("Common chemical keys:", len(common_keys))
    print(
        "Work_df OOF score vs reliability_df score, "
        "maximum absolute difference:",
        float(np.max(np.abs(left - right))) if len(left) else np.nan
    )

    print(
        "Scores agree within 1e-10:",
        bool(np.allclose(left, right, atol=1e-10))
    )

# ------------------------------------------------------------
# G. Verify the 253-compound external reliability values
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[G] External 253-compound reliability alignment")

external_check = external_253.copy()
external_check["unitox_inchikey"] = (
    external_check["unitox_inchikey"].astype(str)
)

rel_key_mean = rel_stats["mean"]

expected_external_score = (
    external_check["unitox_inchikey"].map(rel_key_mean)
)

actual_external_score = pd.to_numeric(
    external_check["neighbour_reliability"],
    errors="coerce"
)

valid = expected_external_score.notna() & actual_external_score.notna()

print("External compounds:", len(external_check))
print("Keys found in reliability table:", int(valid.sum()))
print(
    "Missing reliability matches:",
    int((~valid).sum())
)

if valid.sum() > 0:
    differences = np.abs(
        expected_external_score[valid].to_numpy()
        - actual_external_score[valid].to_numpy()
    )

    print(
        "Maximum absolute score difference:",
        float(differences.max())
    )

    print(
        "External reliability values match:",
        bool(np.allclose(differences, 0, atol=1e-10))
    )

# ------------------------------------------------------------
# H. Summary
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2A COMPLETE")
print("No dataframes, folds, features or models were modified.")
print("=" * 80)


STEP 2A — DATA AND RELIABILITY INTEGRITY

[A] Fingerprint and target alignment
Number of samples: 1349
Morgan shape: (1349, 2048)
X_morgan shape: (1349, 2048)
Morgan and X_morgan numerically identical: True
Target length: 1349
Target values: [0 1]
df and work_df liver labels aligned: True

[B] Fold assignment alignment
outer_fold_ids: shape=(1349,), matches work_df['fold'] = True
fold_ids: shape=(1349,), matches work_df['fold'] = True
folds: shape=(1349,), matches work_df['fold'] = True

Fold sizes:
{0: 163, 1: 132, 2: 132, 3: 132, 4: 132, 5: 132, 6: 132, 7: 132, 8: 131, 9: 131}

Scaffold separation check:
Scaffolds assigned to more than one fold: 0
Maximum number of folds occupied by one scaffold: 1

Per-fold scaffold counts:
{0: 40, 1: 56, 2: 87, 3: 88, 4: 89, 5: 88, 6: 89, 7: 89, 8: 88, 9: 88}

[C] Existing fold_indices structure
Number of entries: 10
First entry type: ndarray
First entry shape: (135,)

[D] Duplicate InChIKey investigation
Rows involved in duplicate InChIKeys: 2
   

In [92]:

# STEP 2A.1: DUPLICATE MOLECULE AND FOLD-INDEX VERIFICATION
# Read-only diagnostic.

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 2A.1 — DUPLICATE IDENTITY AND FOLD INDEX AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# A. Investigate the duplicate InChIKey
# ------------------------------------------------------------
print("\n[A] Duplicate chemical identity")

dup_key = work_df.loc[
    work_df["inchikey"].duplicated(keep=False),
    "inchikey"
].iloc[0]

dup_idx = np.where(
    work_df["inchikey"].astype(str).to_numpy() == str(dup_key)
)[0]

print("Duplicate key:", dup_key)
print("Row indices:", dup_idx.tolist())

display_cols = [
    "generic_name",
    "inchikey",
    "smiles",
    "all_smiles",
    "liver_toxicity_binary_rating_0_1",
    "scaffold",
    "fold",
    "S_neighbour_consistency",
    "S_neighbour_oof"
]

display_cols = [
    c for c in display_cols if c in work_df.columns
]

print("\nDuplicate records:")
display(work_df.iloc[dup_idx][display_cols])

# Compare fingerprints
fp_a = np.asarray(morgan[dup_idx[0]])
fp_b = np.asarray(morgan[dup_idx[1]])

intersection = np.logical_and(fp_a > 0, fp_b > 0).sum()
union = np.logical_or(fp_a > 0, fp_b > 0).sum()

tanimoto = intersection / union if union else 0.0

print("\nFingerprint comparison:")
print("Fingerprint A nonzero bits:", int((fp_a > 0).sum()))
print("Fingerprint B nonzero bits:", int((fp_b > 0).sum()))
print("Exactly identical:", bool(np.array_equal(fp_a, fp_b)))
print("Different bit positions:", int(np.sum(fp_a != fp_b)))
print("Tanimoto similarity:", float(tanimoto))

if "X_morgan" in globals():
    print(
        "X_morgan rows identical:",
        bool(np.array_equal(
            X_morgan[dup_idx[0]],
            X_morgan[dup_idx[1]]
        ))
    )

# ------------------------------------------------------------
# B. Inspect duplicate records in the reliability table
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[B] Duplicate records in reliability_df")

duplicate_reliability = reliability_df.loc[
    reliability_df["unitox_inchikey"].astype(str) == str(dup_key)
].copy()

display(duplicate_reliability)

# ------------------------------------------------------------
# C. Inspect fold_indices
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[C] fold_indices comparison with canonical outer folds")

canonical = work_df["fold"].to_numpy()
n = len(work_df)

if "fold_indices" in globals():
    fold_index_arrays = [
        np.asarray(x, dtype=int).ravel()
        for x in fold_indices
    ]

    all_valid = all(
        len(x) == 0 or (x.min() >= 0 and x.max() < n)
        for x in fold_index_arrays
    )

    print("Number of arrays:", len(fold_index_arrays))
    print("All indices within valid range:", all_valid)

    if all_valid:
        for k, arr in enumerate(fold_index_arrays):
            expected = np.where(canonical == k)[0]

            print(
                f"Fold {k}: length={len(arr)}, "
                f"expected test size={len(expected)}, "
                f"exactly matches canonical fold={np.array_equal(np.sort(arr), expected)}"
            )

        concatenated = np.concatenate(fold_index_arrays)

        print("\nCombined fold-index diagnostics:")
        print("Total entries:", len(concatenated))
        print("Unique entries:", len(np.unique(concatenated)))
        print("Covers all samples exactly once:",
              len(concatenated) == n
              and len(np.unique(concatenated)) == n
              and set(concatenated) == set(range(n)))

        counts = pd.Series(concatenated).value_counts()
        print("Indices occurring more than once:", int((counts > 1).sum()))
        print("Indices never included:", int(n - len(counts)))
else:
    print("fold_indices unavailable.")

# ------------------------------------------------------------
# D. Verify DILIrank evidence categories
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[D] External evidence coverage")

for col in ["DILIConcern_clean", "AdjudicatedDILI"]:
    if col in external_253.columns:
        print(f"\n{col}:")
        print(
            external_253[col]
            .value_counts(dropna=False)
            .to_string()
        )

print("\n" + "=" * 80)
print("STEP 2A.1 COMPLETE")
print("No dataframes or models were modified.")
print("=" * 80)


STEP 2A.1 — DUPLICATE IDENTITY AND FOLD INDEX AUDIT

[A] Duplicate chemical identity
Duplicate key: DQSGVVGOPRWTKI-UHFFFAOYSA-N
Row indices: [986, 1276]

Duplicate records:


,generic_name,inchikey,smiles,all_smiles,liver_toxicity_binary_rating_0_1,scaffold,fold,S_neighbour_consistency,S_neighbour_oof
986,ATAZANAVIR SULFATE,DQSGVVGOPRWTKI-UHFFFAOYSA-N,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,1,c1ccc(CCCCNCc2ccc(-c3ccccn3)cc2)cc1,7,0.933653,0.804975
1276,REYATAZ,DQSGVVGOPRWTKI-UHFFFAOYSA-N,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,1,c1ccc(CCCCNCc2ccc(-c3ccccn3)cc2)cc1,7,1.000000,0.727854



Fingerprint comparison:
Fingerprint A nonzero bits: 67
Fingerprint B nonzero bits: 69
Exactly identical: False
Different bit positions: 6
Tanimoto similarity: 0.9154929577464789
X_morgan rows identical: False

[B] Duplicate records in reliability_df


,unitox_inchikey,unitox_label,outer_fold,neighbour_reliability
986,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7,0.804975
1276,DQSGVVGOPRWTKI-UHFFFAOYSA-N,1,7,0.727854



[C] fold_indices comparison with canonical outer folds
Number of arrays: 10
All indices within valid range: True
Fold 0: length=135, expected test size=163, exactly matches canonical fold=False
Fold 1: length=135, expected test size=132, exactly matches canonical fold=False
Fold 2: length=135, expected test size=132, exactly matches canonical fold=False
Fold 3: length=135, expected test size=132, exactly matches canonical fold=False
Fold 4: length=135, expected test size=132, exactly matches canonical fold=False
Fold 5: length=135, expected test size=132, exactly matches canonical fold=False
Fold 6: length=135, expected test size=132, exactly matches canonical fold=False
Fold 7: length=135, expected test size=132, exactly matches canonical fold=False
Fold 8: length=135, expected test size=131, exactly matches canonical fold=False
Fold 9: length=134, expected test size=131, exactly matches canonical fold=False

Combined fold-index diagnostics:
Total entries: 1349
Unique entries: 1349
C

In [93]:

# STEP 2A.2: CHEMICAL STANDARDIZATION + FOLD INDICES AUDIT
# Read-only diagnostic. Does not overwrite existing fingerprints or scores.

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize

print("=" * 80)
print("STEP 2A.2 — CHEMICAL IDENTITY AND FOLD AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# A. Inspect duplicate SMILES and chemical representations
# ------------------------------------------------------------
dup_key = work_df.loc[
    work_df["inchikey"].duplicated(keep=False),
    "inchikey"
].iloc[0]

dup_idx = np.where(
    work_df["inchikey"].astype(str).to_numpy() == str(dup_key)
)[0]

print("\n[A] Original chemical representations")

for idx in dup_idx:
    row = work_df.iloc[idx]

    print(f"\nDataset row index: {idx}")
    print("Name:", row["generic_name"])
    print("InChIKey:", row["inchikey"])
    print("SMILES:", row["smiles"])
    print("All SMILES:", row.get("all_smiles", None))

# ------------------------------------------------------------
# B. Compare original and standardized Morgan fingerprints
# ------------------------------------------------------------
def make_morgan(mol, radius=2, n_bits=2048):
    return AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=n_bits
    )

def compare_molecules(mol_a, mol_b):
    fp_a = make_morgan(mol_a)
    fp_b = make_morgan(mol_b)
    return DataStructs.TanimotoSimilarity(fp_a, fp_b), (
        fp_a.ToBitString() == fp_b.ToBitString()
    )

print("\n" + "=" * 80)
print("[B] Original vs standardized chemical structures")

parent_molecules = []

for idx in dup_idx:
    smiles = str(work_df.iloc[idx]["smiles"])
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Could not parse SMILES for row {idx}")
        parent_molecules.append(None)
        continue

    # Standardization is performed only for diagnosis.
    cleaned = rdMolStandardize.Cleanup(mol)
    parent = rdMolStandardize.FragmentParent(cleaned)
    parent = rdMolStandardize.Uncharger().uncharge(parent)

    parent_molecules.append(parent)

    print(f"\nRow {idx}")
    print("Original canonical SMILES:",
          Chem.MolToSmiles(mol, canonical=True))
    print("Original fragments:",
          len(Chem.GetMolFrags(mol)))
    print("Original InChIKey:",
          Chem.MolToInchiKey(mol))
    print("Standardized parent SMILES:",
          Chem.MolToSmiles(parent, canonical=True))
    print("Standardized parent InChIKey:",
          Chem.MolToInchiKey(parent))
    print("Original -> Parent Tanimoto:",
          compare_molecules(mol, parent)[0])

if all(m is not None for m in parent_molecules):
    similarity, identical = compare_molecules(
        parent_molecules[0], parent_molecules[1]
    )

    print("\nStandardized pair comparison:")
    print("Parent Tanimoto similarity:", similarity)
    print("Parent fingerprints identical:", identical)

# ------------------------------------------------------------
# C. Test whether fold_indices is scaffold-separated
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[C] Scaffold leakage in fold_indices")

canonical_scaffolds = work_df["scaffold"].astype(str).to_numpy()

fold_indices_scaffold_report = []

for k, indices in enumerate(fold_indices):
    indices = np.asarray(indices, dtype=int)
    fold_indices_scaffold_report.append({
        "fold": k,
        "n_samples": len(indices),
        "n_scaffolds": len(np.unique(canonical_scaffolds[indices]))
    })

index_to_fold = np.full(len(work_df), -1, dtype=int)

for k, indices in enumerate(fold_indices):
    index_to_fold[np.asarray(indices, dtype=int)] = k

scaffold_fold_counts = (
    pd.DataFrame({
        "scaffold": canonical_scaffolds,
        "fold_indices_group": index_to_fold
    })
    .groupby("scaffold")["fold_indices_group"]
    .nunique()
)

print(pd.DataFrame(fold_indices_scaffold_report).to_string(index=False))

print(
    "\nScaffolds split across multiple fold_indices groups:",
    int((scaffold_fold_counts > 1).sum())
)

# ------------------------------------------------------------
# D. Compare class proportions across fold_indices groups
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[D] Class distribution in fold_indices")

fold_index_distribution = pd.DataFrame({
    "fold_indices_group": index_to_fold,
    "liver_label": work_df[
        "liver_toxicity_binary_rating_0_1"
    ].to_numpy()
})

print(
    pd.crosstab(
        fold_index_distribution["fold_indices_group"],
        fold_index_distribution["liver_label"],
        margins=True
    )
)

# ------------------------------------------------------------
# E. Find potentially relevant existing helper functions
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[E] Potential reliability calculation functions")

import inspect

function_candidates = []

for name, obj in globals().copy().items():
    if not callable(obj) or name.startswith("_"):
        continue

    if any(term in name.lower() for term in [
        "neighbour", "neighbor", "reliability",
        "tanimoto", "similarity"
    ]):
        function_candidates.append((name, obj))

if function_candidates:
    for name, func in function_candidates:
        print("\nFunction:", name)

        try:
            print("Signature:", inspect.signature(func))
            source = inspect.getsource(func)
            print("Source preview:")
            print("\n".join(source.splitlines()[:35]))
        except Exception as error:
            print("Source unavailable:", str(error))
else:
    print("No matching user-defined helper functions found.")

print("\n" + "=" * 80)
print("STEP 2A.2 COMPLETE — NO EXISTING DATA WAS MODIFIED.")
print("=" * 80)


STEP 2A.2 — CHEMICAL IDENTITY AND FOLD AUDIT

[A] Original chemical representations

Dataset row index: 986
Name: ATAZANAVIR SULFATE
InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C.O=S(=O)(O)O
All SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C.O=S(=O)(O)O

Dataset row index: 1276
Name: REYATAZ
InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C.O=S(=O)([O-])[O-].[H+].[H+]
All SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C.O=S(=O)([O-])[O-].[H+].[H+]

[B] Original vs standardized chemical structures

Row 986
Original canonical SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C.O=S(=O)(O)O
Original fragments: 2
Original InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
Standardiz

[10:48:45] Initializing MetalDisconnector
[10:48:45] Running MetalDisconnector
[10:48:45] Initializing Normalizer
[10:48:45] Running Normalizer
[10:48:45] Initializing MetalDisconnector
[10:48:45] Running MetalDisconnector
[10:48:45] Initializing Normalizer
[10:48:45] Running Normalizer
[10:48:45] Running LargestFragmentChooser
[10:48:45] Fragment: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C
[10:48:45] New largest fragment: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C (103)
[10:48:45] Fragment: O=S(=O)(O)O
[10:48:45] Running Uncharger
[10:48:45] DEPRECATION WARNING: please use MorganGenerator
[10:48:45] DEPRECATION WARNING: please use MorganGenerator
[10:48:45] 

****
Pre-condition Violation
RingInfo not initialized
Violation occurred on line 52 in file C:\rdkit\build\temp.win-amd64-cpython-314\Release\rdkit\Code\GraphMol\RingInfo.cpp
Failed Expression: df_init
****



RuntimeError: Pre-condition Violation
	RingInfo not initialized
	Violation occurred on line 52 in file Code\GraphMol\RingInfo.cpp
	Failed Expression: df_init
	RDKIT: 2026.03.6
	BOOST: 1_85


In [94]:

# STEP 2A.2 — CORRECTED CHEMICAL STANDARDIZATION AND FOLD AUDIT
# Read-only: creates diagnostic objects but does not overwrite existing data.

import numpy as np
import pandas as pd
import inspect

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

print("=" * 85)
print("STEP 2A.2 — CORRECTED DIAGNOSTIC")
print("=" * 85)

# ------------------------------------------------------------
# A. Identify duplicate chemical records
# ------------------------------------------------------------
duplicate_mask = work_df["inchikey"].duplicated(keep=False)
duplicate_keys = work_df.loc[duplicate_mask, "inchikey"].unique()

print("\n[A] Duplicate chemical identities")
print("Number of duplicate InChIKeys:", len(duplicate_keys))

duplicate_indices = []

for key in duplicate_keys:
    indices = np.where(
        work_df["inchikey"].astype(str).to_numpy() == str(key)
    )[0]

    duplicate_indices.extend(indices.tolist())

    print("\nInChIKey:", key)
    print("Row positions:", indices.tolist())

    display(
        work_df.iloc[indices][[
            "generic_name",
            "inchikey",
            "smiles",
            "liver_toxicity_binary_rating_0_1",
            "scaffold",
            "fold"
        ]]
    )

# ------------------------------------------------------------
# B. Standardize both representations safely
# ------------------------------------------------------------
def standardize_parent_diagnostic(smiles):
    """
    Diagnostic chemical standardization:
    parse -> sanitize -> cleanup -> largest fragment -> uncharge.
    Original input data are not modified.
    """
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        raise ValueError("SMILES could not be parsed.")

    mol = Chem.Mol(mol)
    Chem.SanitizeMol(mol)

    cleaned = rdMolStandardize.Cleanup(mol)
    parent = rdMolStandardize.FragmentParent(cleaned)
    parent = rdMolStandardize.Uncharger().uncharge(parent)

    # Explicit sanitization and ring initialization.
    Chem.SanitizeMol(parent)
    parent.UpdatePropertyCache(strict=False)
    Chem.GetSymmSSSR(parent)

    return mol, parent


morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def get_morgan_fp(mol):
    mol_copy = Chem.Mol(mol)
    Chem.SanitizeMol(mol_copy)
    mol_copy.UpdatePropertyCache(strict=False)
    Chem.GetSymmSSSR(mol_copy)

    return morgan_generator.GetFingerprint(mol_copy)


print("\n" + "=" * 85)
print("[B] Standardized parent comparison")

standardized_records = []

for idx in duplicate_indices:
    row = work_df.iloc[idx]

    original, parent = standardize_parent_diagnostic(row["smiles"])
    parent_fp = get_morgan_fp(parent)

    standardized_records.append({
        "row_index": idx,
        "name": row["generic_name"],
        "original_inchikey": Chem.MolToInchiKey(original),
        "parent_inchikey": Chem.MolToInchiKey(parent),
        "parent_smiles": Chem.MolToSmiles(parent, canonical=True),
        "parent_fingerprint": parent_fp,
        "parent_nonzero_bits": parent_fp.GetNumOnBits()
    })

    print(f"\nRow {idx}: {row['generic_name']}")
    print("Original InChIKey:", Chem.MolToInchiKey(original))
    print("Parent InChIKey:", Chem.MolToInchiKey(parent))
    print("Parent SMILES:", Chem.MolToSmiles(parent, canonical=True))
    print("Parent fingerprint nonzero bits:", parent_fp.GetNumOnBits())

if len(standardized_records) == 2:
    fp1 = standardized_records[0]["parent_fingerprint"]
    fp2 = standardized_records[1]["parent_fingerprint"]

    print("\nStandardized pair comparison:")
    print(
        "Parent Tanimoto similarity:",
        DataStructs.TanimotoSimilarity(fp1, fp2)
    )

    print(
        "Parent fingerprints identical:",
        fp1.ToBitString() == fp2.ToBitString()
    )

    print(
        "Parent InChIKeys identical:",
        standardized_records[0]["parent_inchikey"]
        == standardized_records[1]["parent_inchikey"]
    )

# ------------------------------------------------------------
# C. Inspect fold_indices for scaffold leakage
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[C] fold_indices scaffold-separation audit")

canonical_scaffolds = work_df["scaffold"].astype(str).to_numpy()
target = work_df[
    "liver_toxicity_binary_rating_0_1"
].to_numpy()

index_to_fold = np.full(len(work_df), -1, dtype=int)

for k, indices in enumerate(fold_indices):
    indices = np.asarray(indices, dtype=int).ravel()

    if np.any(indices < 0) or np.any(indices >= len(work_df)):
        raise ValueError(f"Invalid indices found in fold_indices[{k}]")

    index_to_fold[indices] = k

    print(
        f"Fold {k}: samples={len(indices)}, "
        f"scaffolds={len(np.unique(canonical_scaffolds[indices]))}, "
        f"negative={(target[indices] == 0).sum()}, "
        f"positive={(target[indices] == 1).sum()}"
    )

print("\nComplete partition:")
print("Unassigned rows:", int((index_to_fold == -1).sum()))
print(
    "Number of uniquely assigned rows:",
    len(np.unique(np.concatenate(fold_indices)))
)

scaffold_group_check = pd.DataFrame({
    "scaffold": canonical_scaffolds,
    "fold_indices_group": index_to_fold
}).groupby("scaffold")["fold_indices_group"].nunique()

print(
    "Scaffolds split across multiple fold_indices groups:",
    int((scaffold_group_check > 1).sum())
)

# ------------------------------------------------------------
# D. Inspect external adjudication evidence
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[D] External adjudication evidence")

for col in ["DILIConcern_clean", "AdjudicatedDILI"]:
    if col in external_253.columns:
        print(f"\n{col}:")
        print(
            external_253[col]
            .value_counts(dropna=False)
            .to_string()
        )

# ------------------------------------------------------------
# E. Recover original reliability calculation cells
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[E] Notebook history: reliability calculation")

history = globals().get("In", [])

tokens = [
    "S_neighbour_oof",
    "neighbour_reliability",
    "S_neighbour_consistency",
    "TanimotoSimilarity",
    "GetMorganFingerprint",
    "GetMorganGenerator"
]

matches = []

for cell_number, source in enumerate(history):
    if not isinstance(source, str):
        continue

    if any(token in source for token in tokens):
        matches.append((cell_number, source))

print("Matching executed notebook cells:", len(matches))

for cell_number, source in matches[-8:]:
    print("\n" + "-" * 75)
    print("NOTEBOOK CELL:", cell_number)
    lines = source.splitlines()

    # Show up to 65 lines per matching cell.
    print("\n".join(lines[:65]))

    if len(lines) > 65:
        print("... [remaining lines omitted from diagnostic]")

print("\n" + "=" * 85)
print("STEP 2A.2 COMPLETE")
print("Original datasets and fingerprints remain unchanged.")
print("=" * 85)


STEP 2A.2 — CORRECTED DIAGNOSTIC

[A] Duplicate chemical identities
Number of duplicate InChIKeys: 1

InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
Row positions: [986, 1276]


,generic_name,inchikey,smiles,liver_toxicity_binary_rating_0_1,scaffold,fold
986,ATAZANAVIR SULFATE,DQSGVVGOPRWTKI-UHFFFAOYSA-N,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,1,c1ccc(CCCCNCc2ccc(-c3ccccn3)cc2)cc1,7
1276,REYATAZ,DQSGVVGOPRWTKI-UHFFFAOYSA-N,COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2c...,1,c1ccc(CCCCNCc2ccc(-c3ccccn3)cc2)cc1,7



[B] Standardized parent comparison

Row 986: ATAZANAVIR SULFATE
Original InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
Parent InChIKey: AXRYRYVKAWYZBR-UHFFFAOYSA-N
Parent SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C
Parent fingerprint nonzero bits: 63

Row 1276: REYATAZ
Original InChIKey: DQSGVVGOPRWTKI-UHFFFAOYSA-N
Parent InChIKey: AXRYRYVKAWYZBR-UHFFFAOYSA-N
Parent SMILES: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C
Parent fingerprint nonzero bits: 63

Standardized pair comparison:
Parent Tanimoto similarity: 1.0
Parent fingerprints identical: True
Parent InChIKeys identical: True

[C] fold_indices scaffold-separation audit
Fold 0: samples=135, scaffolds=14, negative=95, positive=40
Fold 1: samples=135, scaffolds=59, negative=64, positive=71
Fold 2: samples=135, scaffolds=90, negative=74, positive=61
Fold 3: samples=135, scaffolds=91, negative=56, positive=79
Fold 4: samples=135, scaffolds=9

[10:54:29] Initializing MetalDisconnector
[10:54:29] Running MetalDisconnector
[10:54:29] Initializing Normalizer
[10:54:29] Running Normalizer
[10:54:29] Initializing MetalDisconnector
[10:54:29] Running MetalDisconnector
[10:54:29] Initializing Normalizer
[10:54:29] Running Normalizer
[10:54:29] Running LargestFragmentChooser
[10:54:29] Fragment: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C
[10:54:29] New largest fragment: COC(=O)NC(C(=O)NC(Cc1ccccc1)C(O)CN(Cc1ccc(-c2ccccn2)cc1)NC(=O)C(NC(=O)OC)C(C)(C)C)C(C)(C)C (103)
[10:54:29] Fragment: O=S(=O)(O)O
[10:54:29] Running Uncharger
[10:54:29] WARNING: not removing hydrogen atom without neighbors
[10:54:29] WARNING: not removing hydrogen atom without neighbors
[10:54:29] WARNING: not removing hydrogen atom without neighbors
[10:54:29] WARNING: not removing hydrogen atom without neighbors
[10:54:29] Initializing MetalDisconnector
[10:54:29] Running MetalDisconnector
[10:54:29] Initializing Nor

In [95]:

# STEP 2B: FULL DATASET STANDARDIZATION + ORIGINAL NOTEBOOK HISTORY
#
# Read-only audit.
# Does not replace the original UniTox fingerprints or scores.

import numpy as np
import pandas as pd
from collections import defaultdict

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

print("=" * 85)
print("STEP 2B — FINAL IDENTITY AND FOLD PROTOCOL AUDIT")
print("=" * 85)

# ------------------------------------------------------------
# A. Standardize all molecules and generate parent identities
# ------------------------------------------------------------
print("\n[A] Standardizing all 1,349 chemical structures")

def get_standardized_parent(smiles):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None, None

    mol = Chem.Mol(mol)
    Chem.SanitizeMol(mol)

    cleaned = rdMolStandardize.Cleanup(mol)
    parent = rdMolStandardize.FragmentParent(cleaned)
    parent = rdMolStandardize.Uncharger().uncharge(parent)

    Chem.SanitizeMol(parent)
    parent.UpdatePropertyCache(strict=False)
    Chem.GetSymmSSSR(parent)

    return (
        Chem.MolToInchiKey(parent),
        Chem.MolToSmiles(parent, canonical=True)
    )

parent_keys = []
parent_smiles = []
standardization_errors = []

for i, smiles in enumerate(work_df["smiles"]):
    try:
        key, parent = get_standardized_parent(smiles)
        parent_keys.append(key)
        parent_smiles.append(parent)

        if key is None:
            standardization_errors.append((i, "Invalid SMILES"))

    except Exception as error:
        parent_keys.append(None)
        parent_smiles.append(None)
        standardization_errors.append((i, str(error)))

standardized_audit_df = pd.DataFrame({
    "row_index": np.arange(len(work_df)),
    "generic_name": work_df["generic_name"].to_numpy(),
    "original_inchikey": work_df["inchikey"].astype(str).to_numpy(),
    "parent_inchikey": parent_keys,
    "parent_smiles": parent_smiles,
    "liver_label": work_df[
        "liver_toxicity_binary_rating_0_1"
    ].to_numpy(),
    "scaffold": work_df["scaffold"].astype(str).to_numpy(),
    "outer_fold": work_df["fold"].to_numpy()
})

valid_parent = standardized_audit_df.dropna(
    subset=["parent_inchikey"]
).copy()

print("Total records:", len(standardized_audit_df))
print("Successfully standardized:", len(valid_parent))
print("Standardization failures:", len(standardization_errors))
print("Unique original InChIKeys:",
      standardized_audit_df["original_inchikey"].nunique())
print("Unique standardized parent keys:",
      valid_parent["parent_inchikey"].nunique())

# ------------------------------------------------------------
# B. Identify parent groups containing multiple records
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[B] Duplicate standardized parent groups")

parent_groups = (
    valid_parent.groupby("parent_inchikey")
    .agg(
        n_records=("row_index", "size"),
        n_original_keys=("original_inchikey", "nunique"),
        n_labels=("liver_label", "nunique"),
        n_scaffolds=("scaffold", "nunique"),
        n_outer_folds=("outer_fold", "nunique"),
        records=("generic_name", lambda x: list(x))
    )
)

duplicate_parent_groups = parent_groups[
    parent_groups["n_records"] > 1
].copy()

print("Duplicate parent groups:", len(duplicate_parent_groups))
print("Records in duplicate parent groups:",
      int(duplicate_parent_groups["n_records"].sum()))

print(
    "Groups with label disagreement:",
    int((duplicate_parent_groups["n_labels"] > 1).sum())
)

print(
    "Groups spanning outer folds:",
    int((duplicate_parent_groups["n_outer_folds"] > 1).sum())
)

print(
    "Groups with multiple original InChIKeys:",
    int((duplicate_parent_groups["n_original_keys"] > 1).sum())
)

if len(duplicate_parent_groups):
    display(duplicate_parent_groups.reset_index())

    duplicate_parent_details = valid_parent[
        valid_parent["parent_inchikey"].isin(
            duplicate_parent_groups.index
        )
    ].sort_values(["parent_inchikey", "generic_name"])

    display(duplicate_parent_details)

# ------------------------------------------------------------
# C. Classify whether original and parent identities differ
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[C] Original identity versus parent identity")

identity_comparison = valid_parent.groupby(
    "parent_inchikey"
).agg(
    original_keys=("original_inchikey", "nunique"),
    records=("row_index", "size"),
)

different_identity_groups = identity_comparison[
    identity_comparison["original_keys"] > 1
]

print(
    "Parent groups combining multiple original InChIKeys:",
    len(different_identity_groups)
)

# ------------------------------------------------------------
# D. Compare label balance in both valid scaffold partitions
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[D] Compare current scaffold partitions")

def summarize_partition(fold_array, partition_name):
    tmp = pd.DataFrame({
        "fold": np.asarray(fold_array),
        "label": work_df[
            "liver_toxicity_binary_rating_0_1"
        ].to_numpy(),
        "scaffold": work_df["scaffold"].astype(str).to_numpy()
    })

    result = tmp.groupby("fold").agg(
        n_samples=("label", "size"),
        n_negative=("label", lambda x: (x == 0).sum()),
        n_positive=("label", lambda x: (x == 1).sum()),
        n_scaffolds=("scaffold", "nunique")
    )

    result["positive_rate"] = (
        result["n_positive"] / result["n_samples"]
    )

    print(f"\n{partition_name}")
    display(result.round(4))

summarize_partition(
    work_df["fold"].to_numpy(),
    "Canonical work_df fold"
)

index_partition = np.full(len(work_df), -1, dtype=int)

for k, indices in enumerate(fold_indices):
    index_partition[np.asarray(indices, dtype=int)] = k

summarize_partition(
    index_partition,
    "Alternative fold_indices partition"
)

# ------------------------------------------------------------
# E. Search notebook history for original fold creation
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[E] Original fold creation cells")

history = globals().get("In", [])

fold_patterns = [
    "fold_indices",
    "outer_fold_ids",
    "scaffold_groups",
    "scaffold_results",
    "GroupKFold",
    "StratifiedGroupKFold",
]

for pattern in fold_patterns:
    matching_cells = [
        (i, source)
        for i, source in enumerate(history)
        if isinstance(source, str) and pattern in source
    ]

    print(f"\n{pattern}: {len(matching_cells)} matching cells")

    for cell_num, source in matching_cells[-5:]:
        lines = source.splitlines()

        relevant_lines = [
            j for j, line in enumerate(lines)
            if pattern in line
            or (
                pattern == "fold_indices"
                and ("split(" in line or "append(" in line)
            )
        ]

        print(f"\nCell {cell_num}:")

        if relevant_lines:
            shown = set()

            for j in relevant_lines[:4]:
                start = max(0, j - 2)
                end = min(len(lines), j + 4)

                for k in range(start, end):
                    if k not in shown:
                        print(f"{k+1}: {lines[k]}")
                        shown.add(k)
        else:
            print("\n".join(lines[:12]))

# ------------------------------------------------------------
# F. Search notebook history for neighbour reliability generation
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("[F] Original reliability generation cells")

reliability_patterns = [
    "S_neighbour_oof",
    "S_neighbour_consistency",
    "neighbour_reliability",
    "TanimotoSimilarity",
    "GetMorganGenerator"
]

for pattern in reliability_patterns:
    matching_cells = [
        (i, source)
        for i, source in enumerate(history)
        if isinstance(source, str) and pattern in source
    ]

    print(f"\n{pattern}: {len(matching_cells)} matching cells")

    for cell_num, source in matching_cells[-4:]:
        lines = source.splitlines()

        print(f"\nCell {cell_num}:")

        relevant_lines = [
            j for j, line in enumerate(lines)
            if pattern in line
        ]

        shown = set()

        for j in relevant_lines[:5]:
            start = max(0, j - 3)
            end = min(len(lines), j + 7)

            for k in range(start, end):
                if k not in shown:
                    print(f"{k+1}: {lines[k]}")
                    shown.add(k)

# ------------------------------------------------------------
# G. Summary
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("STEP 2B COMPLETE")
print("No original data or models were modified.")
print("=" * 85)


STEP 2B — FINAL IDENTITY AND FOLD PROTOCOL AUDIT

[A] Standardizing all 1,349 chemical structures


[11:00:10] Initializing MetalDisconnector
[11:00:10] Running MetalDisconnector
[11:00:10] Initializing Normalizer
[11:00:10] Running Normalizer
[11:00:10] Initializing MetalDisconnector
[11:00:10] Running MetalDisconnector
[11:00:10] Initializing Normalizer
[11:00:10] Running Normalizer
[11:00:10] Running LargestFragmentChooser
[11:00:10] Fragment: CC(O)C(NC(=O)C1CSSCC(NC(=O)C(Cc2ccccc2)NC(=O)CN2CCN(CC(=O)[O-])CCN(CC(=O)[O-])CCN(CC(=O)[O-])CC2)C(=O)NC(Cc2ccc(O)cc2)C(=O)NC(Cc2c[nH]c3ccccc23)C(=O)NC(CCCCN)C(=O)NC(C(C)O)C(=O)N1)C(=O)O
[11:00:10] New largest fragment: CC(O)C(NC(=O)C1CSSCC(NC(=O)C(Cc2ccccc2)NC(=O)CN2CCN(CC(=O)[O-])CCN(CC(=O)[O-])CCN(CC(=O)[O-])CC2)C(=O)NC(Cc2ccc(O)cc2)C(=O)NC(Cc2c[nH]c3ccccc23)C(=O)NC(CCCCN)C(=O)NC(C(C)O)C(=O)N1)C(=O)O (187)
[11:00:10] Fragment: [Ga+3]
[11:00:10] Running Uncharger
[11:00:10] Removed negative charge.
[11:00:10] Removed negative charge.
[11:00:10] Removed negative charge.
[11:00:10] Initializing MetalDisconnector
[11:00:10] Running MetalDisco

Total records: 1349
Successfully standardized: 1349
Standardization failures: 0
Unique original InChIKeys: 1348
Unique standardized parent keys: 1164

[B] Duplicate standardized parent groups
Duplicate parent groups: 160
Records in duplicate parent groups: 345
Groups with label disagreement: 36
Groups spanning outer folds: 33
Groups with multiple original InChIKeys: 160


,parent_inchikey,n_records,n_original_keys,n_labels,n_scaffolds,n_outer_folds,records
0,AAKJLRGGTJKAMG-UHFFFAOYSA-N,2,2,1,1,1,"[ERLOTINIB, ERLOTINIB HYDROCHLORIDE]"
1,ADEBPBSSDYVVLD-UHFFFAOYSA-N,2,2,1,1,1,"[DONEPEZIL HYDROCHLORIDE, DONEPEZIL]"
2,AELCINSCMGFISI-UHFFFAOYSA-N,2,2,1,2,2,"[TRANYLCYPROMINE SULFATE, TRANYLCYPROMINE]"
3,AHOUBRCZNHFOSL-UHFFFAOYSA-N,3,3,1,2,2,"[PAROXETINE, PAROXETINE MESYLATE, PAROXETINE H..."
4,ALWKGYPQUAPLQC-UHFFFAOYSA-N,2,2,1,1,1,"[NEOSTIGMINE METHYLSULFATE, NEOSTIGMINE]"
...,...,...,...,...,...,...,...
155,YXRDKMPIGHSVRX-UHFFFAOYSA-N,2,2,2,1,1,"[ROCURONIUM, ROCURONIUM BROMIDE]"
156,ZBPLOVFIXSTCRZ-UHFFFAOYSA-N,2,2,1,1,1,"[BROMFENAC, BROMFENAC OPHTHALMIC SOLUTION 0.09%]"
157,ZEWQUBUPAILYHI-UHFFFAOYSA-N,2,2,1,1,1,"[TRIFLUOPERAZINE HYDROCHLORIDE, TRIFLUOPERAZINE]"
158,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,4,4,1,2,2,"[HYDROXYZINE HYDROCHLORIDE, HYDROXYZINE PAMOAT..."


,row_index,generic_name,original_inchikey,parent_inchikey,parent_smiles,liver_label,scaffold,outer_fold
309,309,ERLOTINIB,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG-UHFFFAOYSA-N,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,1,c1ccc(Nc2ncnc3ccccc23)cc1,5
310,310,ERLOTINIB HYDROCHLORIDE,GTTBEUCJPZQMDZ-UHFFFAOYSA-N,AAKJLRGGTJKAMG-UHFFFAOYSA-N,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,1,c1ccc(Nc2ncnc3ccccc23)cc1,5
1073,1073,DONEPEZIL,ADEBPBSSDYVVLD-UHFFFAOYSA-N,ADEBPBSSDYVVLD-UHFFFAOYSA-N,COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2,0,O=C1c2ccccc2CC1CC1CCN(Cc2ccccc2)CC1,7
267,267,DONEPEZIL HYDROCHLORIDE,XWAIAVWHZJNZQQ-UHFFFAOYSA-N,ADEBPBSSDYVVLD-UHFFFAOYSA-N,COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2,0,O=C1c2ccccc2CC1CC1CCN(Cc2ccccc2)CC1,7
1322,1322,TRANYLCYPROMINE,AELCINSCMGFISI-UHFFFAOYSA-N,AELCINSCMGFISI-UHFFFAOYSA-N,NC1CC1c1ccccc1,1,c1ccc(C2CC2)cc1,9
...,...,...,...,...,...,...,...,...
1143,1143,HYDROXYZINE HCL,IPSVAUIEEPSJRZ-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1,4
434,434,HYDROXYZINE HYDROCHLORIDE,ANOMHKZSQFYSBR-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1,4
435,435,HYDROXYZINE PAMOATE,ASDOKGIIKXGMNB-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,OCCOCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1.c1ccc2c(Cc3cccc4...,7
254,254,DIMENHYDRINATE,NFLLKCVHYJRNRH-UHFFFAOYSA-N,ZZVUWRFHKOJYTH-UHFFFAOYSA-N,CN(C)CCOC(c1ccccc1)c1ccccc1,0,O=c1[nH]c(=O)c2[nH]cnc2[nH]1.c1ccc(Cc2ccccc2)cc1,4



[C] Original identity versus parent identity
Parent groups combining multiple original InChIKeys: 160

[D] Compare current scaffold partitions

Canonical work_df fold


,n_samples,n_negative,n_positive,n_scaffolds,positive_rate
fold,,,,,
0,163,90,73,40,0.4479
1,132,58,74,56,0.5606
2,132,58,74,87,0.5606
3,132,58,74,88,0.5606
4,132,58,74,89,0.5606
5,132,58,74,88,0.5606
6,132,59,73,89,0.5530
7,132,58,74,89,0.5606
8,131,58,73,88,0.5573



Alternative fold_indices partition


,n_samples,n_negative,n_positive,n_scaffolds,positive_rate
fold,,,,,
0,135,95,40,14,0.2963
1,135,64,71,59,0.5259
2,135,74,61,90,0.4519
3,135,56,79,91,0.5852
4,135,53,82,91,0.6074
5,135,50,85,91,0.6296
6,135,54,81,91,0.6000
7,135,52,83,92,0.6148
8,135,55,80,92,0.5926



[E] Original fold creation cells

fold_indices: 9 matching cells

Cell 91:
100: 
101: # ------------------------------------------------------------
102: # C. Inspect the structure of fold_indices
103: # ------------------------------------------------------------
104: print("\n" + "=" * 80)
105: print("[C] Existing fold_indices structure")
106: 
107: if "fold_indices" in globals():
108:     print("Number of entries:", len(fold_indices))
109: 
110:     if len(fold_indices) > 0:
111:         first = fold_indices[0]

Cell 92:
83: 
84: # ------------------------------------------------------------
85: # C. Inspect fold_indices
86: # ------------------------------------------------------------
87: print("\n" + "=" * 80)
88: print("[C] fold_indices comparison with canonical outer folds")
89: 
90: canonical = work_df["fold"].to_numpy()
91: n = len(work_df)
92: 
93: if "fold_indices" in globals():
94:     fold_index_arrays = [
95:         np.asarray(x, dtype=int).ravel()
96:         for x in

In [96]:

# STEP 2B.1 — PARENT IDENTITY REVIEW + ORIGINAL METHODOLOGY RECOVERY
# Diagnostic only. Does not change datasets, fingerprints or folds.

import numpy as np
import pandas as pd
from rdkit import Chem

print("=" * 90)
print("STEP 2B.1 — PARENT IDENTITY AND METHODOLOGY REVIEW")
print("=" * 90)

# ------------------------------------------------------------
# A. Prepare parent-group review table
# ------------------------------------------------------------
audit = standardized_audit_df.copy()

# Alternative scaffold fold assignment
alternative_fold = np.full(len(work_df), -1, dtype=int)

for k, idx in enumerate(fold_indices):
    alternative_fold[np.asarray(idx, dtype=int)] = k

if np.any(alternative_fold < 0):
    raise ValueError("Some rows are not assigned to alternative folds.")

audit["alternative_fold"] = alternative_fold

# ------------------------------------------------------------
# B. Summarize all standardized parent groups
# ------------------------------------------------------------
parent_summary = (
    audit.groupby("parent_inchikey", dropna=False)
    .agg(
        n_records=("row_index", "size"),
        n_original_keys=("original_inchikey", "nunique"),
        n_labels=("liver_label", "nunique"),
        labels=("liver_label", lambda x: sorted(set(x.dropna()))),
        n_scaffolds=("scaffold", "nunique"),
        n_current_folds=("outer_fold", "nunique"),
        n_alternative_folds=("alternative_fold", "nunique"),
        current_folds=("outer_fold", lambda x: sorted(set(x))),
        alternative_folds=("alternative_fold", lambda x: sorted(set(x))),
        names=("generic_name", list)
    )
    .reset_index()
)

# Keep groups representing multiple records.
duplicate_groups = parent_summary[
    parent_summary["n_records"] > 1
].copy()

# Flag groups that need investigation.
duplicate_groups["label_conflict"] = (
    duplicate_groups["n_labels"] > 1
)

duplicate_groups["crosses_current_folds"] = (
    duplicate_groups["n_current_folds"] > 1
)

duplicate_groups["crosses_alternative_folds"] = (
    duplicate_groups["n_alternative_folds"] > 1
)

duplicate_groups["crosses_any_fold"] = (
    duplicate_groups["crosses_current_folds"]
    | duplicate_groups["crosses_alternative_folds"]
)

print("\n[A] Complete parent-group statistics")

print("Duplicate parent groups:", len(duplicate_groups))
print(
    "Groups with conflicting labels:",
    int(duplicate_groups["label_conflict"].sum())
)
print(
    "Groups crossing work_df folds:",
    int(duplicate_groups["crosses_current_folds"].sum())
)
print(
    "Groups crossing alternative folds:",
    int(duplicate_groups["crosses_alternative_folds"].sum())
)
print(
    "Groups crossing either partition:",
    int(duplicate_groups["crosses_any_fold"].sum())
)
print(
    "Groups with both label conflict and fold leakage:",
    int((
        duplicate_groups["label_conflict"]
        & duplicate_groups["crosses_any_fold"]
    ).sum())
)

# ------------------------------------------------------------
# C. Display groups requiring investigation
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[B] Groups with label conflicts or fold crossings")

problem_groups = duplicate_groups[
    duplicate_groups["label_conflict"]
    | duplicate_groups["crosses_any_fold"]
].copy()

display_columns = [
    "parent_inchikey",
    "n_records",
    "n_original_keys",
    "labels",
    "n_scaffolds",
    "current_folds",
    "alternative_folds",
    "label_conflict",
    "crosses_current_folds",
    "crosses_alternative_folds",
    "names"
]

print("Number of groups requiring review:", len(problem_groups))
display(problem_groups[display_columns].reset_index(drop=True))

# ------------------------------------------------------------
# D. Inspect the original fragments for flagged groups
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[C] Fragment composition of flagged compounds")

flagged_keys = set(problem_groups["parent_inchikey"])

flagged_rows = audit[
    audit["parent_inchikey"].isin(flagged_keys)
].copy()

fragment_records = []

for row_index in flagged_rows["row_index"].astype(int):
    row = work_df.iloc[row_index]
    mol = Chem.MolFromSmiles(str(row["smiles"]))

    if mol is None:
        fragment_list = None
    else:
        fragments = Chem.GetMolFrags(mol, asMols=True)

        fragment_list = [
            {
                "heavy_atoms": frag.GetNumHeavyAtoms(),
                "smiles": Chem.MolToSmiles(frag, canonical=True)
            }
            for frag in fragments
        ]

    fragment_records.append({
        "row_index": row_index,
        "generic_name": row["generic_name"],
        "parent_inchikey": audit.loc[
            audit["row_index"] == row_index,
            "parent_inchikey"
        ].iloc[0],
        "original_inchikey": row["inchikey"],
        "label": row["liver_toxicity_binary_rating_0_1"],
        "fold": row["fold"],
        "fragments": fragment_list
    })

fragment_review = pd.DataFrame(fragment_records)

# Display only the compound groups involved in a potential label conflict.
conflict_keys = set(
    problem_groups.loc[
        problem_groups["label_conflict"],
        "parent_inchikey"
    ]
)

conflict_fragment_review = fragment_review[
    fragment_review["parent_inchikey"].isin(conflict_keys)
]

display(conflict_fragment_review)

# ------------------------------------------------------------
# E. Compare the two scaffold partitions at dataset level
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[D] Partition integrity")

for label, fold_array in [
    ("work_df fold", work_df["fold"].to_numpy()),
    ("alternative fold_indices", alternative_fold)
]:
    temp = pd.DataFrame({
        "fold": fold_array,
        "scaffold": work_df["scaffold"].astype(str).to_numpy()
    })

    scaffold_spread = temp.groupby("scaffold")["fold"].nunique()

    parent_spread = (
        audit.assign(fold=fold_array)
        .groupby("parent_inchikey")["fold"]
        .nunique()
    )

    print(f"\n{label}")
    print("Unique folds:", len(np.unique(fold_array)))
    print("Scaffolds crossing folds:",
          int((scaffold_spread > 1).sum()))
    print(
        "Standardized parent identities crossing folds:",
        int((parent_spread > 1).sum())
    )

# ------------------------------------------------------------
# F. Recover exact original notebook cells
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[E] Original notebook methodology cells")

history = globals().get("In", [])

important_cells = [18, 22, 75, 76, 77, 78, 79, 80]

for cell_number in important_cells:
    print("\n" + "#" * 90)
    print(f"ORIGINAL NOTEBOOK CELL In[{cell_number}]")
    print("#" * 90)

    if cell_number < len(history):
        source = history[cell_number]

        if isinstance(source, str):
            print(source)
        else:
            print("Cell does not contain string source.")
    else:
        print("Cell number unavailable in notebook history.")

# ------------------------------------------------------------
# G. Save the review tables without touching source data
# ------------------------------------------------------------
parent_summary.to_csv(
    "step_2B_parent_identity_summary.csv",
    index=False
)

fragment_review.to_csv(
    "step_2B_parent_fragment_review.csv",
    index=False
)

print("\n" + "=" * 90)
print("STEP 2B.1 COMPLETE")
print("Saved:")
print("  step_2B_parent_identity_summary.csv")
print("  step_2B_parent_fragment_review.csv")
print("Original datasets, fingerprints and folds were not modified.")
print("=" * 90)


STEP 2B.1 — PARENT IDENTITY AND METHODOLOGY REVIEW

[A] Complete parent-group statistics
Duplicate parent groups: 160
Groups with conflicting labels: 36
Groups crossing work_df folds: 33
Groups crossing alternative folds: 28
Groups crossing either partition: 33
Groups with both label conflict and fold leakage: 6

[B] Groups with label conflicts or fold crossings
Number of groups requiring review: 63


,parent_inchikey,n_records,n_original_keys,labels,n_scaffolds,current_folds,alternative_folds,label_conflict,crosses_current_folds,crosses_alternative_folds,names
0,AELCINSCMGFISI-UHFFFAOYSA-N,2,2,[1],2,"[0, 9]","[3, 9]",False,True,True,"[TRANYLCYPROMINE SULFATE, TRANYLCYPROMINE]"
1,AHOUBRCZNHFOSL-UHFFFAOYSA-N,3,3,[1],2,"[2, 6]","[3, 4]",False,True,True,"[PAROXETINE, PAROXETINE MESYLATE, PAROXETINE H..."
2,AQHHHDLHHXJYJD-UHFFFAOYSA-N,2,2,"[0, 1]",1,[3],[9],True,False,False,"[PROPRANOLOL HYDROCHLORIDE, PROPRANOLOL]"
3,BARDROPHSZEBKC-UHFFFAOYSA-N,2,2,"[0, 1]",1,[8],[2],True,False,False,"[FOSAPREPITANT, FOSAPREPITANT DIMEGLUMINE]"
4,BLCLNMBMMGCOAS-UHFFFAOYSA-N,2,2,"[0, 1]",1,[3],[3],True,False,False,"[GOSERELIN, GOSERELIN ACETATE]"
...,...,...,...,...,...,...,...,...,...,...,...
58,XUKUURHRXDUEBC-UHFFFAOYSA-N,3,3,[1],2,"[1, 4]","[6, 7]",False,True,True,"[ATORVASTATIN CALCIUM, ATORVALIQ, ATORVASTATIN]"
59,YREYEVIYCVEVJK-UHFFFAOYSA-N,2,2,[1],2,"[2, 4]","[4, 8]",False,True,True,"[RABEPRAZOLE SODIUM, RABEPRAZOLE]"
60,YXRDKMPIGHSVRX-UHFFFAOYSA-N,2,2,"[0, 1]",1,[2],[7],True,False,False,"[ROCURONIUM, ROCURONIUM BROMIDE]"
61,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,4,4,[0],2,"[4, 7]","[6, 8]",False,True,True,"[HYDROXYZINE HYDROCHLORIDE, HYDROXYZINE PAMOAT..."



[C] Fragment composition of flagged compounds


,row_index,generic_name,parent_inchikey,original_inchikey,label,fold,fragments
0,0,68GA-DOTATATE,QVFLVLMYXXNJDT-UHFFFAOYSA-N,XBJPSVQFCQFGDC-UHFFFAOYSA-K,0,7,"[{'heavy_atoms': 100, 'smiles': 'CC(O)C(NC(=O)..."
2,13,ACYCLOVIR,MKUXAQIIEYXACX-UHFFFAOYSA-N,MKUXAQIIEYXACX-UHFFFAOYSA-N,0,2,"[{'heavy_atoms': 16, 'smiles': 'Nc1nc2c(ncn2CO..."
5,24,ALMOTRIPTAN,WKEMJKQOLOHJLZ-UHFFFAOYSA-N,WKEMJKQOLOHJLZ-UHFFFAOYSA-N,0,9,"[{'heavy_atoms': 23, 'smiles': 'CN(C)CCc1c[nH]..."
6,25,ALOSETRON HYDROCHLORIDE,JSWZEAMFRNKZNL-UHFFFAOYSA-N,FNYQZOVOVDSGJH-UHFFFAOYSA-N,0,4,"[{'heavy_atoms': 22, 'smiles': 'Cc1[nH]cnc1CN1..."
7,39,AMMONIA N-13,QGZKDVFQNNGYKY-UHFFFAOYSA-N,QGZKDVFQNNGYKY-UHFFFAOYSA-N,0,0,"[{'heavy_atoms': 1, 'smiles': 'N'}]"
...,...,...,...,...,...,...,...
136,1281,ROCURONIUM BROMIDE,YXRDKMPIGHSVRX-UHFFFAOYSA-N,OYTJKRAYGYRUJK-UHFFFAOYSA-M,0,2,"[{'heavy_atoms': 38, 'smiles': 'C=CC[N+]1(C2CC..."
137,1288,SELEGILINE,MEZLKOACVSPNER-UHFFFAOYSA-N,MEZLKOACVSPNER-UHFFFAOYSA-N,0,1,"[{'heavy_atoms': 14, 'smiles': 'C#CCN(C)C(C)Cc..."
138,1320,TOPOTECAN HYDROCHLORIDE,UCFGDBYHRUNTLO-UHFFFAOYSA-N,DGHHQBMTXTWTJV-UHFFFAOYSA-N,1,8,"[{'heavy_atoms': 31, 'smiles': 'CCC1(O)C(=O)OC..."
140,1329,TRIMETHOBENZAMIDE,FEZBIKUBAYAZIU-UHFFFAOYSA-N,FEZBIKUBAYAZIU-UHFFFAOYSA-N,1,5,"[{'heavy_atoms': 28, 'smiles': 'COc1cc(C(=O)NC..."



[D] Partition integrity

work_df fold
Unique folds: 10
Scaffolds crossing folds: 0
Standardized parent identities crossing folds: 33

alternative fold_indices
Unique folds: 10
Scaffolds crossing folds: 0
Standardized parent identities crossing folds: 28

[E] Original notebook methodology cells

##########################################################################################
ORIGINAL NOTEBOOK CELL In[18]
##########################################################################################
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# Scaffold groups
# ============================================================

scaffold_groups = defaultdict(list)

for idx, scaffold in enumerate(df["scaffold"]):
    scaffold_groups[scaffold].append(idx)

# Convert to lists
scaffold_groups = {
    scaffold: np.array(indices)
    for scaffold, indices in scaffold_groups.items()
}

print("Number 

In [97]:

# STEP 2B.2 — DETAILED PARENT-FRAGMENT REVIEW
# Identifies possible salt variants and multiple-organic-fragment compounds.

import numpy as np
import pandas as pd
from rdkit import Chem

audit = standardized_audit_df.copy()

# Reconstruct alternative fold assignment
alternative_fold = np.full(len(work_df), -1, dtype=int)

for k, idx in enumerate(fold_indices):
    alternative_fold[np.asarray(idx, dtype=int)] = k

audit["alternative_fold"] = alternative_fold

record_details = []

for idx, row in work_df.iterrows():

    mol = Chem.MolFromSmiles(str(row["smiles"]))

    if mol is None:
        fragments = []
    else:
        fragments = Chem.GetMolFrags(mol, asMols=True)

    fragment_details = []

    for frag in fragments:
        heavy = frag.GetNumHeavyAtoms()
        carbon = sum(
            atom.GetAtomicNum() == 6
            for atom in frag.GetAtoms()
        )
        charge = sum(
            atom.GetFormalCharge()
            for atom in frag.GetAtoms()
        )

        fragment_details.append({
            "heavy_atoms": heavy,
            "carbon_atoms": carbon,
            "charge": charge,
            "organic": carbon > 0,
            "smiles": Chem.MolToSmiles(frag, canonical=True)
        })

    organic_fragments = [
        f for f in fragment_details if f["organic"]
    ]

    # This is only a screening flag; it does not prove a combination drug.
    substantial_organic_fragments = [
        f for f in organic_fragments if f["heavy_atoms"] >= 8
    ]

    record_details.append({
        "row_index": idx,
        "generic_name": row["generic_name"],
        "original_inchikey": str(row["inchikey"]),
        "parent_inchikey": audit.iloc[idx]["parent_inchikey"],
        "liver_label": int(
            row["liver_toxicity_binary_rating_0_1"]
        ),
        "scaffold": str(row["scaffold"]),
        "current_fold": int(row["fold"]),
        "alternative_fold": int(alternative_fold[idx]),
        "n_fragments": len(fragment_details),
        "n_organic_fragments": len(organic_fragments),
        "n_substantial_organic_fragments": len(
            substantial_organic_fragments
        ),
        "organic_fragment_sizes": sorted(
            [f["heavy_atoms"] for f in organic_fragments],
            reverse=True
        ),
        "fragment_details": " || ".join(
            f"HA={f['heavy_atoms']}, C={f['carbon_atoms']}, "
            f"charge={f['charge']}, SMILES={f['smiles']}"
            for f in fragment_details
        )
    })

fragment_record_df = pd.DataFrame(record_details)

# Group-level review
group_review = (
    fragment_record_df.groupby("parent_inchikey")
    .agg(
        n_records=("row_index", "size"),
        names=("generic_name", list),
        labels=("liver_label", lambda x: sorted(set(x))),
        current_folds=("current_fold", lambda x: sorted(set(x))),
        alternative_folds=("alternative_fold", lambda x: sorted(set(x))),
        scaffolds=("scaffold", lambda x: sorted(set(x))),
        records_with_multiple_organic_fragments=(
            "n_substantial_organic_fragments",
            lambda x: int((x > 1).sum())
        ),
        fragment_composition=("organic_fragment_sizes", list)
    )
    .reset_index()
)

group_review["label_conflict"] = (
    group_review["labels"].apply(len) > 1
)

group_review["current_fold_crossing"] = (
    group_review["current_folds"].apply(len) > 1
)

group_review["alternative_fold_crossing"] = (
    group_review["alternative_folds"].apply(len) > 1
)

group_review["potential_combination_candidate"] = (
    group_review["records_with_multiple_organic_fragments"] > 0
)

# Keep all repeated-parent groups.
group_review = group_review[
    group_review["n_records"] > 1
].copy()

# Focus review on any group with a conflict, fold crossing,
# or multiple substantial organic fragments.
review_subset = group_review[
    group_review["label_conflict"]
    | group_review["current_fold_crossing"]
    | group_review["alternative_fold_crossing"]
    | group_review["potential_combination_candidate"]
].copy()

print("=" * 85)
print("STEP 2B.2 — PARENT GROUP CLASSIFICATION")
print("=" * 85)

print("All repeated parent groups:", len(group_review))
print("Groups needing detailed review:", len(review_subset))
print(
    "Potential multiple-organic-fragment candidates:",
    int(group_review["potential_combination_candidate"].sum())
)
print(
    "Groups with conflicting labels:",
    int(group_review["label_conflict"].sum())
)
print(
    "Groups crossing current folds:",
    int(group_review["current_fold_crossing"].sum())
)
print(
    "Groups crossing alternative folds:",
    int(group_review["alternative_fold_crossing"].sum())
)

print("\nDetailed group summary:")
display(review_subset.sort_values(
    ["label_conflict", "current_fold_crossing"],
    ascending=False
))

print("\nIndividual molecular fragment details:")
review_indices = set(
    review_subset["parent_inchikey"]
)

display(
    fragment_record_df[
        fragment_record_df["parent_inchikey"].isin(review_indices)
    ].sort_values(["parent_inchikey", "generic_name"])
)

# Save full audit outputs
group_review.to_csv(
    "step_2B2_all_parent_groups.csv",
    index=False
)

review_subset.to_csv(
    "step_2B2_parent_groups_for_review.csv",
    index=False
)

fragment_record_df.to_csv(
    "step_2B2_molecular_fragment_details.csv",
    index=False
)

print("\nSaved three review files:")
print("1. step_2B2_all_parent_groups.csv")
print("2. step_2B2_parent_groups_for_review.csv")
print("3. step_2B2_molecular_fragment_details.csv")


[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:40] WARNING: not removing hydrogen atom without neighbors
[17:11:41] WARNING: not removing hydrogen atom without neighbors
[17:11:42] WARNING: not removing hydrogen atom without neighbors
[17:11:42] WARNING: not removing hydrogen atom without neighbors


STEP 2B.2 — PARENT GROUP CLASSIFICATION
All repeated parent groups: 160
Groups needing detailed review: 74
Potential multiple-organic-fragment candidates: 41
Groups with conflicting labels: 36
Groups crossing current folds: 33
Groups crossing alternative folds: 28

Detailed group summary:


,parent_inchikey,n_records,names,labels,current_folds,alternative_folds,scaffolds,records_with_multiple_organic_fragments,fragment_composition,label_conflict,current_fold_crossing,alternative_fold_crossing,potential_combination_candidate
215,FJZZPCZKBUKGGU-UHFFFAOYSA-N,2,"[ELIGLUSTAT, AGALSIDASE BETA]","[0, 1]","[0, 8]","[7, 9]","[c1cc2c(cc1CCCN1CCCC1)OCCO2, c1cc2c(cc1CCCN1CC...",1,"[[29], [29, 29, 10]]",True,True,True,True
545,MKUXAQIIEYXACX-UHFFFAOYSA-N,2,"[ACYCLOVIR, ACYCLOVIR SODIUM]","[0, 1]","[2, 7]","[5, 9]","[O=c1[nH]cnc2[nH]cnc12, c1ncc2nc[nH]c2n1]",0,"[[16], [16]]",True,True,True,False
755,QVFLVLMYXXNJDT-UHFFFAOYSA-N,3,"[68GA-DOTATATE, COPPER CU 64 DOTATATE, LUTETIU...","[0, 1]","[7, 8]","[3, 5]",[O=C1CNC(=O)CNC(=O)C(Cc2c[nH]c3ccccc23)NC(=O)C...,0,"[[100], [100], [100]]",True,True,True,False
897,ULGZDMOVFRHVEP-UHFFFAOYSA-N,3,"[ERYTHROMYCIN LACTOBIONATE, ERYTHROMYCIN STEAR...","[0, 1]","[4, 8]","[3, 9]",[C1CCOCC1.O=C1CCCCOC(=O)CC(OC2CCCCO2)CC(OC2CCC...,2,"[[51, 24], [51, 20], [51]]",True,True,True,True
922,UXOWGYHJODZGMF-UHFFFAOYSA-N,2,"[ALISKIREN, ALISKIREN HEMIFUMARATE]","[0, 1]","[1, 6]","[1, 7]","[c1ccccc1, c1ccccc1.c1ccccc1]",1,"[[39], [39, 39, 8]]",True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
651,OOGJQPCLVADCPB-UHFFFAOYSA-N,2,"[TOLTERODINE TARTRATE, TOLTERODINE]",[0],[5],[6],[c1ccc(Cc2ccccc2)cc1],1,"[[24, 10], [24]]",False,False,False,True
659,OUYCCCASQSFEME-UHFFFAOYSA-N,2,"[GLATIRAMER ACETATE, GLATIRAMER]",[1],[1],[1],[c1ccccc1],2,"[[13, 10, 10, 6, 4], [13, 10, 10, 6]]",False,False,False,True
901,UNBRKDKAWYKMIV-UHFFFAOYSA-N,2,"[METHYLERGONOVINE MALEATE, METHYLERGONOVINE]",[0],[3],[2],[C1=C2c3cccc4[nH]cc(c34)CC2NCC1],1,"[[25, 8], [25]]",False,False,False,True
963,VSWBSWWIRNCQIJ-UHFFFAOYSA-N,2,"[ASENAPINE MALEATE, ASENAPINE]",[1],[6],[7],[c1ccc2c(c1)Oc1ccccc1C1CNCC21],1,"[[20, 8], [20]]",False,False,False,True



Individual molecular fragment details:


,row_index,generic_name,original_inchikey,parent_inchikey,liver_label,scaffold,current_fold,alternative_fold,n_fragments,n_organic_fragments,n_substantial_organic_fragments,organic_fragment_sizes,fragment_details
1322,1322,TRANYLCYPROMINE,AELCINSCMGFISI-UHFFFAOYSA-N,AELCINSCMGFISI-UHFFFAOYSA-N,1,c1ccc(C2CC2)cc1,9,9,1,1,1,[10],"HA=10, C=9, charge=0, SMILES=NC1CC1c1ccccc1"
895,895,TRANYLCYPROMINE SULFATE,BKPRVQDIOGQWTG-UHFFFAOYSA-N,AELCINSCMGFISI-UHFFFAOYSA-N,1,c1ccc(C2CC2)cc1.c1ccc(C2CC2)cc1,0,3,3,2,2,"[10, 10]","HA=10, C=9, charge=0, SMILES=NC1CC1c1ccccc1 ||..."
660,660,PAROXETINE,AHOUBRCZNHFOSL-UHFFFAOYSA-N,AHOUBRCZNHFOSL-UHFFFAOYSA-N,1,c1ccc(C2CCNCC2COc2ccc3c(c2)OCO3)cc1,2,3,1,1,1,[24],"HA=24, C=19, charge=0, SMILES=Fc1ccc(C2CCNCC2C..."
1239,1239,PAROXETINE HYDROCHLORIDE HEMIHYDRATE,MQZOATSIFWSKKT-UHFFFAOYSA-N,AHOUBRCZNHFOSL-UHFFFAOYSA-N,1,c1ccc(C2CCNCC2COc2ccc3c(c2)OCO3)cc1.c1ccc(C2CC...,6,4,5,2,2,"[24, 24]","HA=1, C=0, charge=0, SMILES=Cl || HA=1, C=0, c..."
661,661,PAROXETINE MESYLATE,SHIJTGJXUHTGGZ-UHFFFAOYSA-N,AHOUBRCZNHFOSL-UHFFFAOYSA-N,1,c1ccc(C2CCNCC2COc2ccc3c(c2)OCO3)cc1,2,3,2,2,1,"[24, 5]","HA=5, C=1, charge=0, SMILES=CS(=O)(=O)O || HA=..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1143,1143,HYDROXYZINE HCL,IPSVAUIEEPSJRZ-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1,4,6,2,1,1,[26],"HA=1, C=0, charge=0, SMILES=Cl || HA=26, C=21,..."
434,434,HYDROXYZINE HYDROCHLORIDE,ANOMHKZSQFYSBR-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1,4,6,3,1,1,[26],"HA=1, C=0, charge=0, SMILES=Cl || HA=1, C=0, c..."
435,435,HYDROXYZINE PAMOATE,ASDOKGIIKXGMNB-UHFFFAOYSA-N,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,0,c1ccc(C(c2ccccc2)N2CCNCC2)cc1.c1ccc2c(Cc3cccc4...,7,8,2,2,2,"[29, 26]","HA=29, C=23, charge=0, SMILES=O=C(O)c1cc2ccccc..."
254,254,DIMENHYDRINATE,NFLLKCVHYJRNRH-UHFFFAOYSA-N,ZZVUWRFHKOJYTH-UHFFFAOYSA-N,0,O=c1[nH]c(=O)c2[nH]cnc2[nH]1.c1ccc(Cc2ccccc2)cc1,4,5,2,2,2,"[19, 14]","HA=19, C=17, charge=0, SMILES=CN(C)CCOC(c1cccc..."



Saved three review files:
1. step_2B2_all_parent_groups.csv
2. step_2B2_parent_groups_for_review.csv
3. step_2B2_molecular_fragment_details.csv


In [98]:

# STEP 2B.3 — CANDIDATE IDENTITY + SCAFFOLD GROUPED OUTER FOLDS
#
# Preserves all original records and labels.
# Does not overwrite work_df["fold"], fold_indices or old scores.

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

print("=" * 85)
print("STEP 2B.3 — CANDIDATE FINAL OUTER FOLD CONSTRUCTION")
print("=" * 85)

# ------------------------------------------------------------
# A. Verify row alignment
# ------------------------------------------------------------
N = len(work_df)

assert len(standardized_audit_df) == N
assert np.array_equal(
    standardized_audit_df["row_index"].to_numpy(),
    np.arange(N)
)

assert np.array_equal(
    standardized_audit_df["original_inchikey"].astype(str).to_numpy(),
    work_df["inchikey"].astype(str).to_numpy()
)

scaffolds = work_df["scaffold"].astype(str).to_numpy()
parent_keys = standardized_audit_df["parent_inchikey"].astype(str).to_numpy()
y = work_df["liver_toxicity_binary_rating_0_1"].to_numpy().astype(int)

assert not pd.isna(parent_keys).any()

print("\n[A] Input verification")
print("Samples:", N)
print("Unique Murcko scaffolds:", len(np.unique(scaffolds)))
print("Unique standardized parent keys:", len(np.unique(parent_keys)))
print("Target distribution:", pd.Series(y).value_counts().sort_index().to_dict())

# ------------------------------------------------------------
# B. Union-find: connect shared scaffold or parent identity
# ------------------------------------------------------------
parent = np.arange(N)
rank = np.zeros(N, dtype=int)

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)

    if ra == rb:
        return

    if rank[ra] < rank[rb]:
        ra, rb = rb, ra

    parent[rb] = ra

    if rank[ra] == rank[rb]:
        rank[ra] += 1

# Connect records sharing the same Murcko scaffold.
scaffold_first = {}

for i, scaffold in enumerate(scaffolds):
    if scaffold in scaffold_first:
        union(i, scaffold_first[scaffold])
    else:
        scaffold_first[scaffold] = i

# Connect records sharing the same standardized parent key.
parent_first = {}

for i, key in enumerate(parent_keys):
    if key in parent_first:
        union(i, parent_first[key])
    else:
        parent_first[key] = i

# Stable component identifiers, ordered by first dataset occurrence.
roots = np.array([find(i) for i in range(N)])

root_to_component = {}
component_ids = np.empty(N, dtype=int)

for i, root in enumerate(roots):
    if root not in root_to_component:
        root_to_component[root] = len(root_to_component)

    component_ids[i] = root_to_component[root]

n_components = len(np.unique(component_ids))

print("\n[B] Connected-component grouping")
print("Number of components:", n_components)

component_df = pd.DataFrame({
    "component": component_ids,
    "scaffold": scaffolds,
    "parent_key": parent_keys,
    "label": y
})

component_summary = (
    component_df.groupby("component")
    .agg(
        n_samples=("label", "size"),
        n_negative=("label", lambda x: int((x == 0).sum())),
        n_positive=("label", lambda x: int((x == 1).sum())),
        n_scaffolds=("scaffold", "nunique"),
        n_parent_keys=("parent_key", "nunique"),
        label_conflict=("label", "nunique")
    )
)

component_summary["label_conflict"] = (
    component_summary["label_conflict"] > 1
)

component_sizes = component_summary["n_samples"]

print("Largest component:", int(component_sizes.max()))
print("Median component size:", float(component_sizes.median()))
print("95th percentile component size:",
      float(component_sizes.quantile(0.95)))
print("Components with conflicting labels:",
      int(component_summary["label_conflict"].sum()))

print("\nLargest 15 components:")
display(
    component_summary.sort_values(
        "n_samples", ascending=False
    ).head(15)
)

# ------------------------------------------------------------
# C. Construct candidate stratified grouped folds
# ------------------------------------------------------------
N_SPLITS = 10
RANDOM_STATE = 42

if n_components < N_SPLITS:
    raise ValueError(
        f"Only {n_components} connected components are available; "
        f"cannot create {N_SPLITS} grouped folds."
    )

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

candidate_outer_fold_ids = np.full(N, -1, dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    cv.split(
        np.zeros((N, 1)),
        y,
        groups=component_ids
    )
):
    if np.any(candidate_outer_fold_ids[test_idx] != -1):
        raise RuntimeError("A sample was assigned to multiple test folds.")

    candidate_outer_fold_ids[test_idx] = fold

if np.any(candidate_outer_fold_ids == -1):
    raise RuntimeError("Some samples were not assigned to a test fold.")

# ------------------------------------------------------------
# D. Verify leakage constraints
# ------------------------------------------------------------
fold_check_df = pd.DataFrame({
    "fold": candidate_outer_fold_ids,
    "component": component_ids,
    "scaffold": scaffolds,
    "parent_key": parent_keys,
    "label": y
})

scaffold_fold_counts = (
    fold_check_df.groupby("scaffold")["fold"].nunique()
)

parent_fold_counts = (
    fold_check_df.groupby("parent_key")["fold"].nunique()
)

component_fold_counts = (
    fold_check_df.groupby("component")["fold"].nunique()
)

scaffold_leaks = int((scaffold_fold_counts > 1).sum())
parent_leaks = int((parent_fold_counts > 1).sum())
component_leaks = int((component_fold_counts > 1).sum())

print("\n[C] Leakage verification")
print("Scaffolds across multiple folds:", scaffold_leaks)
print("Parent keys across multiple folds:", parent_leaks)
print("Components across multiple folds:", component_leaks)

if scaffold_leaks or parent_leaks or component_leaks:
    raise RuntimeError("Candidate fold assignment failed leakage checks.")

# ------------------------------------------------------------
# E. Inspect fold balance
# ------------------------------------------------------------
fold_rows = []

for fold in range(N_SPLITS):
    subset = fold_check_df[
        fold_check_df["fold"] == fold
    ]

    fold_rows.append({
        "fold": fold,
        "n_test": len(subset),
        "negative": int((subset["label"] == 0).sum()),
        "positive": int((subset["label"] == 1).sum()),
        "positive_rate": float(subset["label"].mean()),
        "n_components": subset["component"].nunique(),
        "n_scaffolds": subset["scaffold"].nunique(),
        "n_parent_keys": subset["parent_key"].nunique()
    })

candidate_fold_summary = pd.DataFrame(fold_rows)

print("\n[D] Candidate outer fold summary")
display(candidate_fold_summary.round(4))

print("\nFold-size range:",
      int(candidate_fold_summary["n_test"].min()),
      "to",
      int(candidate_fold_summary["n_test"].max()))

print("Positive-rate range:",
      round(candidate_fold_summary["positive_rate"].min(), 4),
      "to",
      round(candidate_fold_summary["positive_rate"].max(), 4))

# ------------------------------------------------------------
# F. Save candidate fold assignment separately
# ------------------------------------------------------------
candidate_split_df = pd.DataFrame({
    "row_index": np.arange(N),
    "generic_name": work_df["generic_name"].to_numpy(),
    "original_inchikey": work_df["inchikey"].astype(str).to_numpy(),
    "parent_inchikey": parent_keys,
    "scaffold": scaffolds,
    "component_id": component_ids,
    "liver_label": y,
    "candidate_outer_fold": candidate_outer_fold_ids
})

candidate_split_df.to_csv(
    "candidate_identity_scaffold_outer_folds.csv",
    index=False
)

candidate_fold_summary.to_csv(
    "candidate_identity_scaffold_fold_summary.csv",
    index=False
)

component_summary.to_csv(
    "candidate_identity_scaffold_component_summary.csv"
)

print("\n[E] Saved candidate files")
print("1. candidate_identity_scaffold_outer_folds.csv")
print("2. candidate_identity_scaffold_fold_summary.csv")
print("3. candidate_identity_scaffold_component_summary.csv")

print("\n" + "=" * 85)
print("STEP 2B.3 COMPLETE")
print("Candidate split created separately; original folds are unchanged.")
print("=" * 85)


STEP 2B.3 — CANDIDATE FINAL OUTER FOLD CONSTRUCTION

[A] Input verification
Samples: 1349
Unique Murcko scaffolds: 802
Unique standardized parent keys: 1164
Target distribution: {0: 613, 1: 736}

[B] Connected-component grouping
Number of components: 772
Largest component: 122
Median component size: 1.0
95th percentile component size: 3.449999999999932
Components with conflicting labels: 62

Largest 15 components:


,n_samples,n_negative,n_positive,n_scaffolds,n_parent_keys,label_conflict
component,,,,,,
9,122,90,32,1,99,True
7,85,45,40,2,72,True
470,15,13,2,1,14,True
259,12,5,7,1,12,True
10,11,2,9,2,7,True
177,11,1,10,2,7,True
23,9,5,4,2,8,True
188,9,8,1,2,6,True
261,8,0,8,1,5,False



[C] Leakage verification
Scaffolds across multiple folds: 0
Parent keys across multiple folds: 0
Components across multiple folds: 0

[D] Candidate outer fold summary


,fold,n_test,negative,positive,positive_rate,n_components,n_scaffolds,n_parent_keys
0,0,160,90,70,0.4375,39,39,137
1,1,135,60,75,0.5556,78,82,119
2,2,131,57,74,0.5649,91,93,115
3,3,132,58,74,0.5606,91,93,114
4,4,132,58,74,0.5606,90,93,112
5,5,133,59,74,0.5564,89,93,113
6,6,132,58,74,0.5606,89,93,119
7,7,131,58,73,0.5573,36,38,115
8,8,132,58,74,0.5606,86,89,113
9,9,131,57,74,0.5649,83,89,107



Fold-size range: 131 to 160
Positive-rate range: 0.4375 to 0.5649

[E] Saved candidate files
1. candidate_identity_scaffold_outer_folds.csv
2. candidate_identity_scaffold_fold_summary.csv
3. candidate_identity_scaffold_component_summary.csv

STEP 2B.3 COMPLETE
Candidate split created separately; original folds are unchanged.


In [99]:

# STEP 2B.4 — OPTIMIZE OUTER FOLD BALANCE
# Uses only group assignments and class distributions.
# No model performance is used for split selection.

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

# ------------------------------------------------------------
# 1. Verify prerequisites
# ------------------------------------------------------------
N = len(work_df)

assert "component_ids" in globals()
assert len(component_ids) == N

y_balance = work_df[
    "liver_toxicity_binary_rating_0_1"
].to_numpy().astype(int)

scaffolds_balance = work_df["scaffold"].astype(str).to_numpy()
parents_balance = (
    standardized_audit_df["parent_inchikey"].astype(str).to_numpy()
)

N_SPLITS = 10
GLOBAL_RATE = y_balance.mean()
TARGET_SIZE = N / N_SPLITS
PREVALENCE_SCALE = np.sqrt(GLOBAL_RATE * (1 - GLOBAL_RATE))

# ------------------------------------------------------------
# 2. Score a candidate partition
# ------------------------------------------------------------
def evaluate_partition(fold_assignment, seed):
    sizes = []
    positive_rates = []

    for fold in range(N_SPLITS):
        idx = np.flatnonzero(fold_assignment == fold)

        sizes.append(len(idx))
        positive_rates.append(y_balance[idx].mean())

    sizes = np.asarray(sizes, dtype=float)
    positive_rates = np.asarray(positive_rates, dtype=float)

    size_rmse = np.sqrt(
        np.mean(((sizes - TARGET_SIZE) / TARGET_SIZE) ** 2)
    )

    prevalence_rmse = np.sqrt(
        np.mean((positive_rates - GLOBAL_RATE) ** 2)
    ) / PREVALENCE_SCALE

    # Equal contribution from relative size and prevalence imbalance.
    score = size_rmse + prevalence_rmse

    return {
        "seed": seed,
        "objective": score,
        "relative_size_rmse": size_rmse,
        "normalized_prevalence_rmse": prevalence_rmse,
        "min_fold_size": int(sizes.min()),
        "max_fold_size": int(sizes.max()),
        "fold_size_range": int(sizes.max() - sizes.min()),
        "min_positive_rate": float(positive_rates.min()),
        "max_positive_rate": float(positive_rates.max()),
        "positive_rate_range": float(
            positive_rates.max() - positive_rates.min()
        ),
        "fold_sizes": sizes.astype(int).tolist(),
        "positive_rates": positive_rates.tolist(),
        "assignment": fold_assignment.copy()
    }

# ------------------------------------------------------------
# 3. Search deterministic seeds
# ------------------------------------------------------------
print("Testing 500 split seeds...")

candidate_results = []

for seed in range(500):
    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    assignment = np.full(N, -1, dtype=int)

    for fold, (_, test_idx) in enumerate(
        splitter.split(
            np.zeros((N, 1)),
            y_balance,
            groups=component_ids
        )
    ):
        assignment[test_idx] = fold

    if np.any(assignment == -1):
        raise RuntimeError("A candidate split left samples unassigned.")

    result = evaluate_partition(assignment, seed)
    candidate_results.append(result)

# ------------------------------------------------------------
# 4. Compare current seed 42 with the best candidate
# ------------------------------------------------------------
candidate_results_sorted = sorted(
    candidate_results,
    key=lambda x: x["objective"]
)

best_result = candidate_results_sorted[0]

seed42_result = next(
    result for result in candidate_results
    if result["seed"] == 42
)

print("\n=== Balance comparison ===")

comparison = pd.DataFrame([
    {
        "partition": "Previous seed 42",
        "seed": seed42_result["seed"],
        "objective": seed42_result["objective"],
        "size_RMSE": seed42_result["relative_size_rmse"],
        "prevalence_RMSE": seed42_result[
            "normalized_prevalence_rmse"
        ],
        "min_size": seed42_result["min_fold_size"],
        "max_size": seed42_result["max_fold_size"],
        "positive_rate_range": seed42_result[
            "positive_rate_range"
        ]
    },
    {
        "partition": "Best of 500 seeds",
        "seed": best_result["seed"],
        "objective": best_result["objective"],
        "size_RMSE": best_result["relative_size_rmse"],
        "prevalence_RMSE": best_result[
            "normalized_prevalence_rmse"
        ],
        "min_size": best_result["min_fold_size"],
        "max_size": best_result["max_fold_size"],
        "positive_rate_range": best_result[
            "positive_rate_range"
        ]
    }
])

display(comparison.round(5))

# ------------------------------------------------------------
# 5. Save optimized candidate separately
# ------------------------------------------------------------
optimized_fold_ids = best_result["assignment"]

optimized_split_df = pd.DataFrame({
    "row_index": np.arange(N),
    "generic_name": work_df["generic_name"].to_numpy(),
    "original_inchikey": work_df["inchikey"].astype(str).to_numpy(),
    "parent_inchikey": parents_balance,
    "scaffold": scaffolds_balance,
    "component_id": component_ids,
    "liver_label": y_balance,
    "candidate_outer_fold": optimized_fold_ids
})

optimized_summary_rows = []

for fold in range(N_SPLITS):
    idx = np.flatnonzero(optimized_fold_ids == fold)

    optimized_summary_rows.append({
        "fold": fold,
        "n_test": len(idx),
        "negative": int((y_balance[idx] == 0).sum()),
        "positive": int((y_balance[idx] == 1).sum()),
        "positive_rate": float(y_balance[idx].mean()),
        "n_components": len(np.unique(component_ids[idx])),
        "n_scaffolds": len(np.unique(scaffolds_balance[idx])),
        "n_parent_keys": len(np.unique(parents_balance[idx]))
    })

optimized_summary = pd.DataFrame(optimized_summary_rows)

# ------------------------------------------------------------
# 6. Final structural leakage checks
# ------------------------------------------------------------
check_df = pd.DataFrame({
    "fold": optimized_fold_ids,
    "component": component_ids,
    "scaffold": scaffolds_balance,
    "parent": parents_balance
})

scaffold_leaks = (
    check_df.groupby("scaffold")["fold"].nunique() > 1
).sum()

parent_leaks = (
    check_df.groupby("parent")["fold"].nunique() > 1
).sum()

component_leaks = (
    check_df.groupby("component")["fold"].nunique() > 1
).sum()

print("\n=== Best candidate fold summary ===")
display(optimized_summary.round(4))

print("\n=== Leakage verification ===")
print("Scaffold leaks:", int(scaffold_leaks))
print("Parent identity leaks:", int(parent_leaks))
print("Component leaks:", int(component_leaks))

if scaffold_leaks or parent_leaks or component_leaks:
    raise RuntimeError("Optimized fold assignment failed leakage checks.")

# ------------------------------------------------------------
# 7. Save files and retain candidate variables
# ------------------------------------------------------------
optimized_split_df.to_csv(
    "candidate_optimized_identity_scaffold_folds.csv",
    index=False
)

optimized_summary.to_csv(
    "candidate_optimized_fold_summary.csv",
    index=False
)

pd.DataFrame([
    {
        "seed": x["seed"],
        "objective": x["objective"],
        "relative_size_rmse": x["relative_size_rmse"],
        "normalized_prevalence_rmse": x[
            "normalized_prevalence_rmse"
        ]
    }
    for x in candidate_results_sorted
]).to_csv(
    "candidate_fold_seed_search.csv",
    index=False
)

candidate_outer_fold_ids_optimized = optimized_fold_ids.copy()

print("\nBest seed:", best_result["seed"])
print("Best objective:", round(best_result["objective"], 6))
print("Saved optimized candidate files.")
print("Original experimental objects remain unchanged.")


Testing 500 split seeds...

=== Balance comparison ===


,partition,seed,objective,size_RMSE,prevalence_RMSE,min_size,max_size,positive_rate_range
0,Previous seed 42,42,0.13688,0.06259,0.07429,131,160,0.12739
1,Best of 500 seeds,75,0.13608,0.05962,0.07645,132,159,0.12995



=== Best candidate fold summary ===


,fold,n_test,negative,positive,positive_rate,n_components,n_scaffolds,n_parent_keys
0,0,159,90,69,0.4340,38,38,136
1,1,133,59,74,0.5564,92,96,119
2,2,132,58,74,0.5606,83,86,112
3,3,132,58,74,0.5606,87,88,115
4,4,132,58,74,0.5606,83,85,117
5,5,133,58,75,0.5639,92,94,117
6,6,132,58,74,0.5606,88,92,111
7,7,132,58,74,0.5606,88,92,114
8,8,132,58,74,0.5606,81,89,110
9,9,132,58,74,0.5606,40,42,113



=== Leakage verification ===
Scaffold leaks: 0
Parent identity leaks: 0
Component leaks: 0

Best seed: 75
Best objective: 0.136076
Saved optimized candidate files.
Original experimental objects remain unchanged.


In [101]:

# STEP 2B.5 — CORRECTED IDENTITY BRIDGE REVIEW
# Read-only diagnostic. Does not modify original datasets or folds.

import numpy as np
import pandas as pd
from rdkit import Chem

print("=" * 90)
print("STEP 2B.5 — CORRECTED IDENTITY BRIDGE REVIEW")
print("=" * 90)

# ------------------------------------------------------------
# A. Reconstruct the parent identity summary
# ------------------------------------------------------------
base = standardized_audit_df.copy()

base["generic_name"] = work_df["generic_name"].to_numpy()
base["original_smiles"] = work_df["smiles"].to_numpy()
base["current_fold"] = work_df["fold"].to_numpy()
base["component_id"] = np.asarray(component_ids)

parent_summary_review = (
    base.groupby("parent_inchikey")
    .agg(
        n_records=("row_index", "size"),
        n_original_keys=("original_inchikey", "nunique"),
        n_scaffolds=("scaffold", "nunique"),
        n_labels=("liver_label", "nunique"),
        labels=("liver_label", lambda x: sorted(set(x))),
        names=("generic_name", list),
        scaffolds=("scaffold", lambda x: list(dict.fromkeys(x))),
        original_keys=(
            "original_inchikey",
            lambda x: list(dict.fromkeys(x))
        ),
        current_folds=("current_fold", lambda x: sorted(set(x))),
        components=("component_id", lambda x: sorted(set(x)))
    )
    .reset_index()
)

fragment_summary = (
    fragment_record_df.groupby("parent_inchikey")
    .agg(
        n_records_with_multiple_substantial_organic=(
            "n_substantial_organic_fragments",
            lambda x: int((x > 1).sum())
        ),
        organic_fragment_sizes=("organic_fragment_sizes", list)
    )
    .reset_index()
)

parent_summary_review = parent_summary_review.merge(
    fragment_summary,
    on="parent_inchikey",
    how="left",
    validate="one_to_one"
)

# Explicitly reconstruct all derived flags.
parent_summary_review["label_conflict"] = (
    parent_summary_review["n_labels"] > 1
)

parent_summary_review["scaffold_bridge"] = (
    parent_summary_review["n_scaffolds"] > 1
)

parent_summary_review["potential_combination_flag"] = (
    parent_summary_review[
        "n_records_with_multiple_substantial_organic"
    ].fillna(0).astype(int) > 0
)

# ------------------------------------------------------------
# B. Select repeated parent groups
# ------------------------------------------------------------
repeated = parent_summary_review.loc[
    parent_summary_review["n_records"] > 1
].copy()

bridges = repeated.loc[
    repeated["scaffold_bridge"]
].copy()

# Explicitly verify derived columns before displaying.
required_flags = [
    "label_conflict",
    "scaffold_bridge",
    "potential_combination_flag"
]

for col in required_flags:
    if col not in bridges.columns:
        raise KeyError(
            f"Expected column {col!r} was not generated. "
            f"Available columns: {bridges.columns.tolist()}"
        )

print("\n[A] Parent bridge statistics")
print("Repeated parent groups:", len(repeated))
print("Parent groups connecting different scaffolds:", len(bridges))
print(
    "Potential combination candidates:",
    int(bridges["potential_combination_flag"].sum())
)
print(
    "Label-conflicting bridge groups:",
    int(bridges["label_conflict"].sum())
)

print("\n[B] Bridge group details")

bridge_cols = [
    "parent_inchikey",
    "n_records",
    "n_original_keys",
    "n_scaffolds",
    "labels",
    "names",
    "potential_combination_flag",
    "label_conflict",
    "current_folds",
    "organic_fragment_sizes"
]

bridge_view = bridges.loc[:, bridge_cols].copy()

# Sort on a simple stable column. Flags remain display columns.
bridge_view = bridge_view.sort_values(
    "n_records", ascending=False
).reset_index(drop=True)

display(bridge_view)

# ------------------------------------------------------------
# C. Display original fragments of bridge groups
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[C] Original fragment composition")

bridge_keys = set(bridges["parent_inchikey"])

bridge_fragment_details = fragment_record_df.loc[
    fragment_record_df["parent_inchikey"].isin(bridge_keys)
].copy()

fragment_cols = [
    "row_index",
    "generic_name",
    "parent_inchikey",
    "original_inchikey",
    "label",
    "fold",
    "n_fragments",
    "n_organic_fragments",
    "n_substantial_organic_fragments",
    "organic_fragment_sizes",
    "fragment_details"
]

fragment_cols = [
    c for c in fragment_cols
    if c in bridge_fragment_details.columns
]

display(
    bridge_fragment_details[fragment_cols]
    .sort_values("parent_inchikey")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# D. Label-conflicting parent groups
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[D] Label-conflicting parent identities")

conflicts = repeated.loc[
    repeated["label_conflict"]
].copy()

conflict_cols = [
    "parent_inchikey",
    "n_records",
    "n_scaffolds",
    "labels",
    "names",
    "current_folds",
    "organic_fragment_sizes",
    "potential_combination_flag"
]

display(
    conflicts[conflict_cols]
    .sort_values("n_records", ascending=False)
    .reset_index(drop=True)
)

print("Total label-conflicting groups:", len(conflicts))

# ------------------------------------------------------------
# E. Inspect the largest connected components
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[E] Largest connected components")

component_table = (
    base.groupby("component_id")
    .agg(
        n_records=("row_index", "size"),
        n_scaffolds=("scaffold", "nunique"),
        n_parent_keys=("parent_inchikey", "nunique"),
        n_labels=("liver_label", "nunique"),
        negatives=("liver_label", lambda x: int((x == 0).sum())),
        positives=("liver_label", lambda x: int((x == 1).sum())),
        names=("generic_name", list)
    )
    .sort_values("n_records", ascending=False)
)

display(component_table.head(10))

top_component_ids = component_table.head(3).index

top_component_records = base.loc[
    base["component_id"].isin(top_component_ids),
    [
        "row_index",
        "generic_name",
        "original_inchikey",
        "parent_inchikey",
        "liver_label",
        "scaffold",
        "current_fold",
        "component_id"
    ]
].sort_values(["component_id", "generic_name"])

print("\nRecords in the three largest components:")
display(top_component_records.reset_index(drop=True))

# ------------------------------------------------------------
# F. Save review outputs
# ------------------------------------------------------------
bridge_view.to_csv(
    "step_2B5_parent_identity_bridges.csv",
    index=False
)

bridge_fragment_details.to_csv(
    "step_2B5_bridge_fragment_details.csv",
    index=False
)

conflicts.to_csv(
    "step_2B5_label_conflicting_parent_groups.csv",
    index=False
)

component_table.to_csv(
    "step_2B5_connected_component_summary.csv"
)

top_component_records.to_csv(
    "step_2B5_largest_component_records.csv",
    index=False
)

print("\nSaved five diagnostic CSV files.")
print("STEP 2B.5 COMPLETE — original data unchanged.")
print("=" * 90)


STEP 2B.5 — CORRECTED IDENTITY BRIDGE REVIEW

[A] Parent bridge statistics
Repeated parent groups: 160
Parent groups connecting different scaffolds: 33
Potential combination candidates: 24
Label-conflicting bridge groups: 6

[B] Bridge group details


,parent_inchikey,n_records,n_original_keys,n_scaffolds,labels,names,potential_combination_flag,label_conflict,current_folds,organic_fragment_sizes
0,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,4,4,2,[0],"[HYDROXYZINE HYDROCHLORIDE, HYDROXYZINE PAMOAT...",True,False,"[4, 7]","[[26], [29, 26], [26], [26]]"
1,XUKUURHRXDUEBC-UHFFFAOYSA-N,3,3,2,[1],"[ATORVASTATIN CALCIUM, ATORVALIQ, ATORVASTATIN]",True,False,"[1, 4]","[[41, 41], [41, 41], [41]]"
2,AHOUBRCZNHFOSL-UHFFFAOYSA-N,3,3,2,[1],"[PAROXETINE, PAROXETINE MESYLATE, PAROXETINE H...",True,False,"[2, 6]","[[24], [24, 5], [24, 24]]"
3,JGSARLDLIJGVTE-UHFFFAOYSA-N,3,3,2,[0],"[PENICILLIN G BENZATHINE, PENICILLIN G SODIUM,...",True,False,"[4, 8]","[[23, 23, 18], [23], [23]]"
4,BOEGTKLJZSQCCD-UHFFFAOYSA-N,3,3,2,[1],"[CEFADROXIL, CEFADROXIL HEMIHYDRATE, CEFADROXI...",True,False,"[0, 9]","[[25], [25, 25], [25]]"
5,SGKRLCUYIXIAHR-UHFFFAOYSA-N,3,3,2,[1],"[DOXYCYCLINE, DOXYCYCLINE HYCLATE, DOXYCYCLINE...",True,False,"[2, 4]","[[32], [32, 32, 3], [32]]"
6,ULGZDMOVFRHVEP-UHFFFAOYSA-N,3,3,2,"[0, 1]","[ERYTHROMYCIN LACTOBIONATE, ERYTHROMYCIN STEAR...",True,True,"[4, 8]","[[51, 24], [51, 20], [51]]"
7,QVFLVLMYXXNJDT-UHFFFAOYSA-N,3,3,2,"[0, 1]","[68GA-DOTATATE, COPPER CU 64 DOTATATE, LUTETIU...",False,True,"[7, 8]","[[100], [100], [100]]"
8,NDAUXUAQIAJITI-UHFFFAOYSA-N,3,3,2,[0],"[ALBUTEROL, ALBUTEROL SULFATE, LEVALBUTEROL TA...",True,False,"[1, 6]","[[17], [17, 17], [17, 17, 10]]"
9,VGYFMXBACGZSIL-UHFFFAOYSA-N,3,3,2,[1],"[PITAVASTATIN CALCIUM, PITAVASTATIN MAGNESIUM,...",True,False,"[2, 8]","[[31, 31], [31, 31], [31]]"



[C] Original fragment composition


,row_index,generic_name,parent_inchikey,original_inchikey,n_fragments,n_organic_fragments,n_substantial_organic_fragments,organic_fragment_sizes,fragment_details
0,1322,TRANYLCYPROMINE,AELCINSCMGFISI-UHFFFAOYSA-N,AELCINSCMGFISI-UHFFFAOYSA-N,1,1,1,[10],"HA=10, C=9, charge=0, SMILES=NC1CC1c1ccccc1"
1,895,TRANYLCYPROMINE SULFATE,AELCINSCMGFISI-UHFFFAOYSA-N,BKPRVQDIOGQWTG-UHFFFAOYSA-N,3,2,2,"[10, 10]","HA=10, C=9, charge=0, SMILES=NC1CC1c1ccccc1 ||..."
2,661,PAROXETINE MESYLATE,AHOUBRCZNHFOSL-UHFFFAOYSA-N,SHIJTGJXUHTGGZ-UHFFFAOYSA-N,2,2,1,"[24, 5]","HA=5, C=1, charge=0, SMILES=CS(=O)(=O)O || HA=..."
3,660,PAROXETINE,AHOUBRCZNHFOSL-UHFFFAOYSA-N,AHOUBRCZNHFOSL-UHFFFAOYSA-N,1,1,1,[24],"HA=24, C=19, charge=0, SMILES=Fc1ccc(C2CCNCC2C..."
4,1239,PAROXETINE HYDROCHLORIDE HEMIHYDRATE,AHOUBRCZNHFOSL-UHFFFAOYSA-N,MQZOATSIFWSKKT-UHFFFAOYSA-N,5,2,2,"[24, 24]","HA=1, C=0, charge=0, SMILES=Cl || HA=1, C=0, c..."
...,...,...,...,...,...,...,...,...,...
72,1143,HYDROXYZINE HCL,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,IPSVAUIEEPSJRZ-UHFFFAOYSA-N,2,1,1,[26],"HA=1, C=0, charge=0, SMILES=Cl || HA=26, C=21,..."
73,434,HYDROXYZINE HYDROCHLORIDE,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,ANOMHKZSQFYSBR-UHFFFAOYSA-N,3,1,1,[26],"HA=1, C=0, charge=0, SMILES=Cl || HA=1, C=0, c..."
74,435,HYDROXYZINE PAMOATE,ZQDWXGKKHFNSQK-UHFFFAOYSA-N,ASDOKGIIKXGMNB-UHFFFAOYSA-N,2,2,2,"[29, 26]","HA=29, C=23, charge=0, SMILES=O=C(O)c1cc2ccccc..."
75,258,DIPHENHYDRAMINE HYDROCHLORIDE,ZZVUWRFHKOJYTH-UHFFFAOYSA-N,PCHPORCSPXIHLZ-UHFFFAOYSA-N,2,1,1,[19],"HA=19, C=17, charge=0, SMILES=CN(C)CCOC(c1cccc..."



[D] Label-conflicting parent identities


,parent_inchikey,n_records,n_scaffolds,labels,names,current_folds,organic_fragment_sizes,potential_combination_flag
0,QAOWNCQODCNURD-UHFFFAOYSA-N,5,1,"[0, 1]","[BARIUM SULFATE, IRON DEXTRAN, MAGNESIUM SULFA...",[0],"[[], [], [], [], []]",False
1,QVFLVLMYXXNJDT-UHFFFAOYSA-N,3,2,"[0, 1]","[68GA-DOTATATE, COPPER CU 64 DOTATATE, LUTETIU...","[7, 8]","[[100], [100], [100]]",False
2,KYYIDSXMWOZKMP-UHFFFAOYSA-N,3,1,"[0, 1]","[DESVENLAFAXINE, DESVENLAFAXINE SUCCINATE, DES...",[7],"[[19], [19, 8], [19, 8]]",True
3,MQTOSJVFKKJCRP-UHFFFAOYSA-N,3,1,"[0, 1]","[AZITHROMYCIN, AZITHROMYCIN DIHYDRATE, AZITHRO...",[4],"[[52], [52], [52]]",False
4,ULGZDMOVFRHVEP-UHFFFAOYSA-N,3,2,"[0, 1]","[ERYTHROMYCIN LACTOBIONATE, ERYTHROMYCIN STEAR...","[4, 8]","[[51, 24], [51, 20], [51]]",True
5,BLCLNMBMMGCOAS-UHFFFAOYSA-N,2,1,"[0, 1]","[GOSERELIN, GOSERELIN ACETATE]",[3],"[[91], [91, 4]]",False
6,BARDROPHSZEBKC-UHFFFAOYSA-N,2,1,"[0, 1]","[FOSAPREPITANT, FOSAPREPITANT DIMEGLUMINE]",[8],"[[41], [41, 13, 13]]",True
7,AQHHHDLHHXJYJD-UHFFFAOYSA-N,2,1,"[0, 1]","[PROPRANOLOL HYDROCHLORIDE, PROPRANOLOL]",[3],"[[19], [19]]",False
8,BUCORZSTKDOEKQ-UHFFFAOYSA-N,2,1,"[0, 1]","[CHLORDIAZEPOXIDE HYDROCHLORIDE, CHLORDIAZEPOX...",[4],"[[21], [21]]",False
9,HXGBXQDTNZMWGS-UHFFFAOYSA-N,2,1,"[0, 1]","[DARIFENACIN, DARIFENACIN HYDROBROMIDE]",[5],"[[32], [32]]",False


Total label-conflicting groups: 36

[E] Largest connected components


,n_records,n_scaffolds,n_parent_keys,n_labels,negatives,positives,names
component_id,,,,,,,
9,122,1,99,2,90,32,"[ACETOHYDROXAMIC ACID, ACETYLCHOLINE CHLORIDE,..."
7,85,2,72,2,45,40,"[ACEBUTOLOL HYDROCHLORIDE, ACETYLSALICYLIC ACI..."
470,15,1,14,2,13,2,"[PREDNISOLONE SODIUM PHOSPHATE, BECLOMETHASONE..."
259,12,1,12,2,5,7,"[FLUDROCORTISONE ACETATE, HYDROCORTISONE, HYDR..."
10,11,2,7,2,2,9,"[ACYCLOVIR, ADEFOVIR DIPIVOXIL, FAMCICLOVIR, G..."
177,11,2,7,2,1,10,"[DEMECLOCYCLINE HYDROCHLORIDE, DOXYCYCLINE, DO..."
23,9,2,8,2,5,4,"[AMIFAMPRIDINE PHOSPHATE, DALFAMPRIDINE, ETHIO..."
188,9,2,6,2,8,1,"[DIMENHYDRINATE, DIPHENHYDRAMINE HYDROCHLORIDE..."
261,8,1,5,1,0,8,"[FLUPHENAZINE DECANOATE, FLUPHENAZINE HYDROCHL..."



Records in the three largest components:


,row_index,generic_name,original_inchikey,parent_inchikey,liver_label,scaffold,current_fold,component_id
0,7,ACEBUTOLOL HYDROCHLORIDE,KTUFKADDDORSSI-UHFFFAOYSA-N,GOEMGAFJFRBGGG-UHFFFAOYSA-N,1,c1ccccc1,1,7
1,11,ACETYLSALICYLIC ACID,BSYNRYMUTXBXSQ-UHFFFAOYSA-N,BSYNRYMUTXBXSQ-UHFFFAOYSA-N,0,c1ccccc1,1,7
2,12,ACITRETIN,IHUNBGSDBOWDMA-UHFFFAOYSA-N,IHUNBGSDBOWDMA-UHFFFAOYSA-N,1,c1ccccc1,1,7
3,18,ALBUTEROL,NDAUXUAQIAJITI-UHFFFAOYSA-N,NDAUXUAQIAJITI-UHFFFAOYSA-N,0,c1ccccc1,1,7
4,971,ALBUTEROL SULFATE,BNPSSFBOAGDEEL-UHFFFAOYSA-N,NDAUXUAQIAJITI-UHFFFAOYSA-N,0,c1ccccc1.c1ccccc1,6,7
...,...,...,...,...,...,...,...,...
217,1191,METHYLPREDNISOLONE ACETATE,PLBHSZGDDKCEHR-UHFFFAOYSA-N,PLBHSZGDDKCEHR-UHFFFAOYSA-N,1,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,2,470
218,1205,MOMETASONE,QLIIKPVHVRXHRI-UHFFFAOYSA-N,QLIIKPVHVRXHRI-UHFFFAOYSA-N,0,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,2,470
219,1259,PREDNISOLONE ACETATE,LRJOMUJRLNCICJ-UHFFFAOYSA-N,LRJOMUJRLNCICJ-UHFFFAOYSA-N,0,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,2,470
220,711,PREDNISOLONE SODIUM PHOSPHATE,VJZLQIPZNBPASX-UHFFFAOYSA-L,JDOZJEUDSLGTLU-UHFFFAOYSA-N,0,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,2,470



Saved five diagnostic CSV files.
STEP 2B.5 COMPLETE — original data unchanged.


In [102]:

# STEP 2B.6 — CURATED IDENTITY MAP + CANDIDATE FINAL FOLDS
#
# Preserves every original molecule, fingerprint and label.
# Builds a separate, curated grouping and fold assignment.
# Does not overwrite old fold or reliability objects.

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

print("=" * 90)
print("STEP 2B.6 — CURATED IDENTITY POLICY AND FOLD CONSTRUCTION")
print("=" * 90)

# ------------------------------------------------------------
# A. Identify the 33 cross-scaffold standardized-parent bridges
# ------------------------------------------------------------
audit = standardized_audit_df.copy()

audit["generic_name"] = work_df["generic_name"].to_numpy()
audit["original_smiles"] = work_df["smiles"].to_numpy()
audit["current_fold"] = work_df["fold"].to_numpy()

parent_group_info = (
    audit.groupby("parent_inchikey")
    .agg(
        n_records=("row_index", "size"),
        n_scaffolds=("scaffold", "nunique"),
        names=("generic_name", list),
        labels=("liver_label", lambda x: sorted(set(x)))
    )
)

bridge_keys = set(
    parent_group_info.loc[
        parent_group_info["n_scaffolds"] > 1
    ].index.astype(str)
)

print("Cross-scaffold parent bridges:", len(bridge_keys))

if len(bridge_keys) != 33:
    raise ValueError(
        f"Expected 33 bridges based on the audit, found {len(bridge_keys)}."
    )

# ------------------------------------------------------------
# B. Define the reviewed identity policy
# ------------------------------------------------------------

# These two parent matches are false equivalences caused by the
# largest-fragment procedure; do not connect them using parent identity.
exclude_parent_links = {
    "FJZZPCZKBUKGGU-UHFFFAOYSA-N":
        "ELIGLUSTAT / AGALSIDASE BETA: distinct active therapies",

    "WLJNZVDCPSBLRP-UHFFFAOYSA-N":
        "IMIPRAMINE PAMOATE / OLANZAPINE PAMOATE: shared pamoate counterion, different active ingredients"
}

# These are chemically related groups, retained together for
# conservative evaluation, without claiming identical drug products.
related_parent_links = {
    "QVFLVLMYXXNJDT-UHFFFAOYSA-N":
        "DOTATATE radiopharmaceuticals: shared peptide targeting component",

    "NDAUXUAQIAJITI-UHFFFAOYSA-N":
        "ALBUTEROL / LEVALBUTEROL: racemate and R-enantiomer relationship",

    "JKKFKPJIXZFSSB-UHFFFAOYSA-N":
        "CONJUGATED ESTROGENS / ESTROPIPATE: related estrogenic/estrone-sulfate component",

    "ZZVUWRFHKOJYTH-UHFFFAOYSA-N":
        "DIMENHYDRINATE / DIPHENHYDRAMINE: shared active component in a combination product"
}

# Ensure the manually reviewed keys are present.
assert set(exclude_parent_links).issubset(bridge_keys)
assert set(related_parent_links).issubset(bridge_keys)
assert not (set(exclude_parent_links) & set(related_parent_links))

# Remaining bridges are provisionally treated as salt/formulation
# variants of the same active parent based on the reviewed names.
same_parent_links = (
    bridge_keys
    - set(exclude_parent_links)
    - set(related_parent_links)
)

policy_records = []

for key in sorted(bridge_keys):

    info = parent_group_info.loc[key]

    if key in exclude_parent_links:
        category = "exclude_false_parent_equivalence"
        connect = False
        rationale = exclude_parent_links[key]

    elif key in related_parent_links:
        category = "related_active_component_or_family"
        connect = True
        rationale = related_parent_links[key]

    else:
        category = "same_active_parent_salt_or_formulation"
        connect = True
        rationale = (
            "Provisionally grouped as salt/formulation variants "
            "of the same active parent; original records and labels retained."
        )

    policy_records.append({
        "parent_inchikey": key,
        "names": info["names"],
        "labels": info["labels"],
        "n_records": int(info["n_records"]),
        "n_scaffolds": int(info["n_scaffolds"]),
        "policy_category": category,
        "connect_for_outer_split": connect,
        "rationale": rationale
    })

identity_bridge_policy = pd.DataFrame(policy_records)

print("\n[A] Curated bridge policy")
print(
    identity_bridge_policy["policy_category"]
    .value_counts()
    .to_string()
)

display(
    identity_bridge_policy[
        [
            "parent_inchikey",
            "names",
            "policy_category",
            "connect_for_outer_split",
            "rationale"
        ]
    ]
)

identity_bridge_policy.to_csv(
    "curated_parent_identity_bridge_policy.csv",
    index=False
)

# ------------------------------------------------------------
# C. Create conservative grouping:
#    always join same scaffold;
#    join only accepted parent-identity bridges.
# ------------------------------------------------------------
N = len(work_df)

scaffolds = work_df["scaffold"].astype(str).to_numpy()
parent_keys = audit["parent_inchikey"].astype(str).to_numpy()
y = work_df[
    "liver_toxicity_binary_rating_0_1"
].to_numpy().astype(int)

uf_parent = np.arange(N)
uf_rank = np.zeros(N, dtype=int)

def uf_find(x):
    while uf_parent[x] != x:
        uf_parent[x] = uf_parent[uf_parent[x]]
        x = uf_parent[x]
    return x

def uf_union(a, b):
    ra, rb = uf_find(a), uf_find(b)

    if ra == rb:
        return

    if uf_rank[ra] < uf_rank[rb]:
        ra, rb = rb, ra

    uf_parent[rb] = ra

    if uf_rank[ra] == uf_rank[rb]:
        uf_rank[ra] += 1

# First connect all records with the same Murcko scaffold.
scaffold_first = {}

for i, key in enumerate(scaffolds):
    if key in scaffold_first:
        uf_union(i, scaffold_first[key])
    else:
        scaffold_first[key] = i

# Connect only parent groups accepted by the curated policy.
accepted_parent_keys = (
    set(same_parent_links) | set(related_parent_links)
)

parent_first = {}

for i, key in enumerate(parent_keys):

    if key not in accepted_parent_keys:
        continue

    if key in parent_first:
        uf_union(i, parent_first[key])
    else:
        parent_first[key] = i

# Convert union-find roots into stable component IDs.
roots = [uf_find(i) for i in range(N)]

root_map = {}
curated_component_ids = np.empty(N, dtype=int)

for i, root in enumerate(roots):
    if root not in root_map:
        root_map[root] = len(root_map)

    curated_component_ids[i] = root_map[root]

n_curated_components = len(np.unique(curated_component_ids))

print("\n[B] Curated grouping summary")
print("Total original records:", N)
print("Curated connected components:", n_curated_components)
print(
    "Records retained:",
    len(curated_component_ids)
)
print(
    "Distinct excluded parent bridges:",
    len(exclude_parent_links)
)

# ------------------------------------------------------------
# D. Audit conflicting labels within parent groups.
#    Never merge or overwrite labels.
# ------------------------------------------------------------
conflicting_parent_groups = (
    audit.groupby("parent_inchikey")["liver_label"]
    .nunique()
)

conflicting_parent_groups = conflicting_parent_groups[
    conflicting_parent_groups > 1
]

print(
    "Parent groups with different labels retained:",
    len(conflicting_parent_groups)
)

# ------------------------------------------------------------
# E. Search 500 seeds for balanced ten-fold grouped splits
# ------------------------------------------------------------
N_SPLITS = 10
GLOBAL_RATE = y.mean()
TARGET_SIZE = N / N_SPLITS
PREVALENCE_SCALE = np.sqrt(GLOBAL_RATE * (1 - GLOBAL_RATE))

def score_split(assignments, seed):
    sizes = []
    positive_rates = []

    for f in range(N_SPLITS):
        indices = np.flatnonzero(assignments == f)
        sizes.append(len(indices))
        positive_rates.append(y[indices].mean())

    sizes = np.asarray(sizes, dtype=float)
    rates = np.asarray(positive_rates, dtype=float)

    size_rmse = np.sqrt(
        np.mean(((sizes - TARGET_SIZE) / TARGET_SIZE) ** 2)
    )

    prevalence_rmse = np.sqrt(
        np.mean((rates - GLOBAL_RATE) ** 2)
    ) / PREVALENCE_SCALE

    return {
        "seed": seed,
        "objective": size_rmse + prevalence_rmse,
        "size_rmse": size_rmse,
        "prevalence_rmse": prevalence_rmse,
        "min_size": int(sizes.min()),
        "max_size": int(sizes.max()),
        "positive_rate_range": float(rates.max() - rates.min()),
        "sizes": sizes.astype(int).tolist(),
        "positive_rates": rates.tolist(),
        "assignment": assignments.copy()
    }

split_results = []

for seed in range(500):

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    assignments = np.full(N, -1, dtype=int)

    for f, (_, test_idx) in enumerate(
        splitter.split(
            np.zeros((N, 1)),
            y,
            groups=curated_component_ids
        )
    ):
        assignments[test_idx] = f

    if np.any(assignments < 0):
        raise RuntimeError("Candidate split has unassigned samples.")

    split_results.append(
        score_split(assignments, seed)
    )

split_results.sort(key=lambda x: x["objective"])

best = split_results[0]

print("\n[C] Best curated split")
print("Best seed:", best["seed"])
print("Balance objective:", round(best["objective"], 6))
print("Relative size RMSE:", round(best["size_rmse"], 6))
print("Normalized prevalence RMSE:",
      round(best["prevalence_rmse"], 6))

# ------------------------------------------------------------
# F. Construct and validate selected candidate split
# ------------------------------------------------------------
curated_outer_fold_ids = best["assignment"]

candidate_summary_rows = []

for f in range(N_SPLITS):
    idx = np.flatnonzero(curated_outer_fold_ids == f)

    candidate_summary_rows.append({
        "fold": f,
        "n_test": len(idx),
        "negative": int((y[idx] == 0).sum()),
        "positive": int((y[idx] == 1).sum()),
        "positive_rate": float(y[idx].mean()),
        "n_components": len(
            np.unique(curated_component_ids[idx])
        ),
        "n_scaffolds": len(np.unique(scaffolds[idx])),
        "n_parent_keys": len(np.unique(parent_keys[idx]))
    })

curated_fold_summary = pd.DataFrame(candidate_summary_rows)

check_df = pd.DataFrame({
    "fold": curated_outer_fold_ids,
    "component": curated_component_ids,
    "scaffold": scaffolds,
    "parent": parent_keys
})

scaffold_leaks = int(
    (check_df.groupby("scaffold")["fold"].nunique() > 1).sum()
)

component_leaks = int(
    (check_df.groupby("component")["fold"].nunique() > 1).sum()
)

# Parent identities accepted by the policy must remain together.
accepted_parent_leaks = int(
    (
        check_df[
            check_df["parent"].isin(accepted_parent_keys)
        ]
        .groupby("parent")["fold"]
        .nunique() > 1
    ).sum()
)

print("\n[D] Final structural verification")
print("Scaffold leaks:", scaffold_leaks)
print("Curated component leaks:", component_leaks)
print("Accepted parent identity leaks:", accepted_parent_leaks)

if scaffold_leaks or component_leaks or accepted_parent_leaks:
    raise RuntimeError("The curated split failed leakage verification.")

print("\n[E] Fold summary")
display(curated_fold_summary.round(4))

# ------------------------------------------------------------
# G. Save candidate files
# ------------------------------------------------------------
curated_split_df = pd.DataFrame({
    "row_index": np.arange(N),
    "generic_name": work_df["generic_name"].to_numpy(),
    "original_inchikey": work_df["inchikey"].astype(str).to_numpy(),
    "parent_inchikey": parent_keys,
    "scaffold": scaffolds,
    "curated_component_id": curated_component_ids,
    "liver_label": y,
    "candidate_outer_fold": curated_outer_fold_ids
})

curated_split_df.to_csv(
    "candidate_curated_identity_scaffold_folds.csv",
    index=False
)

curated_fold_summary.to_csv(
    "candidate_curated_identity_scaffold_fold_summary.csv",
    index=False
)

pd.DataFrame([
    {
        "seed": x["seed"],
        "objective": x["objective"],
        "size_rmse": x["size_rmse"],
        "prevalence_rmse": x["prevalence_rmse"]
    }
    for x in split_results
]).to_csv(
    "candidate_curated_fold_seed_search.csv",
    index=False
)

print("\n[F] Saved files:")
print("1. curated_parent_identity_bridge_policy.csv")
print("2. candidate_curated_identity_scaffold_folds.csv")
print("3. candidate_curated_identity_scaffold_fold_summary.csv")
print("4. candidate_curated_fold_seed_search.csv")

print("\nSTEP 2B.6 COMPLETE")
print("Original folds and original reliability scores remain unchanged.")
print("=" * 90)


STEP 2B.6 — CURATED IDENTITY POLICY AND FOLD CONSTRUCTION
Cross-scaffold parent bridges: 33

[A] Curated bridge policy
policy_category
same_active_parent_salt_or_formulation    27
related_active_component_or_family         4
exclude_false_parent_equivalence           2


,parent_inchikey,names,policy_category,connect_for_outer_split,rationale
0,AELCINSCMGFISI-UHFFFAOYSA-N,"[TRANYLCYPROMINE SULFATE, TRANYLCYPROMINE]",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
1,AHOUBRCZNHFOSL-UHFFFAOYSA-N,"[PAROXETINE, PAROXETINE MESYLATE, PAROXETINE H...",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
2,BOEGTKLJZSQCCD-UHFFFAOYSA-N,"[CEFADROXIL, CEFADROXIL HEMIHYDRATE, CEFADROXI...",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
3,BPRHUIZQVSMCRT-UHFFFAOYSA-N,"[ROSUVASTATIN, ROSUVASTATIN CALCIUM]",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
4,FJZZPCZKBUKGGU-UHFFFAOYSA-N,"[ELIGLUSTAT, AGALSIDASE BETA]",exclude_false_parent_equivalence,False,ELIGLUSTAT / AGALSIDASE BETA: distinct active ...
5,IIDJRNMFWXDHID-UHFFFAOYSA-N,"[RISEDRONATE SODIUM, RISEDRONATE SODIUM HEMI-P...",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
6,IRSCQMHQWWYFCW-UHFFFAOYSA-N,"[GANCICLOVIR SODIUM, GANCICLOVIR]",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
7,JBMKAUGHUNFTOL-UHFFFAOYSA-N,"[CHLOROTHIAZIDE, CHLOROTHIAZIDE SODIUM]",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
8,JGSARLDLIJGVTE-UHFFFAOYSA-N,"[PENICILLIN G BENZATHINE, PENICILLIN G SODIUM,...",same_active_parent_salt_or_formulation,True,Provisionally grouped as salt/formulation vari...
9,JKKFKPJIXZFSSB-UHFFFAOYSA-N,"[CONJUGATED ESTROGENS, ESTROPIPATE]",related_active_component_or_family,True,CONJUGATED ESTROGENS / ESTROPIPATE: related es...



[B] Curated grouping summary
Total original records: 1349
Curated connected components: 774
Records retained: 1349
Distinct excluded parent bridges: 2
Parent groups with different labels retained: 36

[C] Best curated split
Best seed: 397
Balance objective: 0.136095
Relative size RMSE: 0.059714
Normalized prevalence RMSE: 0.076381

[D] Final structural verification
Scaffold leaks: 0
Curated component leaks: 0
Accepted parent identity leaks: 0

[E] Fold summary


,fold,n_test,negative,positive,positive_rate,n_components,n_scaffolds,n_parent_keys
0,0,159,90,69,0.4340,38,38,136
1,1,134,59,75,0.5597,83,87,115
2,2,132,58,74,0.5606,92,94,115
3,3,132,58,74,0.5606,93,95,116
4,4,132,58,74,0.5606,90,93,109
5,5,132,58,74,0.5606,93,97,113
6,6,132,58,74,0.5606,73,76,115
7,7,132,58,74,0.5606,36,38,112
8,8,132,58,74,0.5606,88,91,118
9,9,132,58,74,0.5606,88,93,117



[F] Saved files:
1. curated_parent_identity_bridge_policy.csv
2. candidate_curated_identity_scaffold_folds.csv
3. candidate_curated_identity_scaffold_fold_summary.csv
4. candidate_curated_fold_seed_search.csv

STEP 2B.6 COMPLETE
Original folds and original reliability scores remain unchanged.


In [103]:

# STEP 3 — FINAL NESTED, IDENTITY-AWARE NEIGHBOUR RELIABILITY
#
# Uses the candidate seed-397 outer folds.
# Uses the original official UniTox Morgan fingerprints.
# Uses curated identity/scaffold components for inner cross-fitting.
# Does not overwrite original folds, fingerprints or old scores.

import numpy as np
import pandas as pd

from rdkit import DataStructs
from sklearn.model_selection import GroupKFold

print("=" * 90)
print("STEP 3 — NESTED MOLECULAR-NEIGHBOUR RELIABILITY")
print("=" * 90)

# ------------------------------------------------------------
# A. Validate all required objects
# ------------------------------------------------------------
required_objects = [
    "work_df",
    "fps",
    "curated_outer_fold_ids",
    "curated_component_ids",
    "standardized_audit_df"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Required objects are missing: " + str(missing)
    )

N = len(work_df)

y = work_df[
    "liver_toxicity_binary_rating_0_1"
].to_numpy().astype(int)

outer_folds = np.asarray(
    curated_outer_fold_ids, dtype=int
)

groups = np.asarray(
    curated_component_ids, dtype=int
)

original_keys = (
    work_df["inchikey"].astype(str).to_numpy()
)

parent_keys = (
    standardized_audit_df["parent_inchikey"]
    .astype(str)
    .to_numpy()
)

scaffolds = (
    work_df["scaffold"].astype(str).to_numpy()
)

assert len(fps) == N
assert len(y) == N
assert len(outer_folds) == N
assert len(groups) == N
assert len(original_keys) == N
assert len(parent_keys) == N

assert len(np.unique(outer_folds)) == 10

# Exact original chemical identities must also remain in one fold.
exact_key_fold_leaks = (
    pd.DataFrame({
        "key": original_keys,
        "fold": outer_folds
    })
    .groupby("key")["fold"]
    .nunique()
    .gt(1)
    .sum()
)

print("\n[A] Protocol verification")
print("Samples:", N)
print("Outer folds:", len(np.unique(outer_folds)))
print("Curated components:", len(np.unique(groups)))
print("Exact InChIKey fold crossings:", int(exact_key_fold_leaks))

if exact_key_fold_leaks:
    raise RuntimeError(
        "Identical original InChIKeys cross outer folds. "
        "Correct the split before calculating final reliability."
    )

if len(np.unique(groups)) != 774:
    print(
        "WARNING: connected-component count differs from "
        "the previous audit; inspect the grouping."
    )

# Check scaffold and component separation.
for column, values in [
    ("scaffold", scaffolds),
    ("component", groups)
]:
    leak_count = (
        pd.DataFrame({
            "group": values,
            "fold": outer_folds
        })
        .groupby("group")["fold"]
        .nunique()
        .gt(1)
        .sum()
    )

    print(f"{column} fold crossings:", int(leak_count))

    if leak_count:
        raise RuntimeError(f"{column} leakage detected.")

# ------------------------------------------------------------
# B. Configure the reliability calculation
# ------------------------------------------------------------
K_NEIGHBORS = 10
N_INNER_FOLDS = 5
NEUTRAL_FALLBACK = 0.5

# Ensure fingerprints can be indexed consistently.
if not hasattr(fps, "__getitem__"):
    raise TypeError("fps must be an indexable fingerprint collection.")

def calculate_neighbour_score(query_idx, reference_idx, k=10):
    """
    Similarity-weighted label agreement for one query.

    Exclusions:
    - Same original InChIKey.
    - Zero-similarity reference molecules.

    Outer/inner group exclusion is enforced by the calling procedure.
    """

    reference_idx = np.asarray(
        reference_idx, dtype=int
    )

    # Never use another record with the exact same original identity.
    reference_idx = reference_idx[
        original_keys[reference_idx] != original_keys[query_idx]
    ]

    if len(reference_idx) == 0:
        return NEUTRAL_FALLBACK, 0, 0.0

    similarities = np.asarray(
        DataStructs.BulkTanimotoSimilarity(
            fps[int(query_idx)],
            [fps[int(j)] for j in reference_idx]
        ),
        dtype=float
    )

    valid = np.flatnonzero(similarities > 0)

    if len(valid) == 0:
        return NEUTRAL_FALLBACK, 0, 0.0

    # Select the k largest positive similarities.
    order = np.argsort(
        similarities[valid], kind="mergesort"
    )[::-1]

    selected = valid[order[:k]]

    neighbour_idx = reference_idx[selected]
    neighbour_similarities = similarities[selected]

    agreement = (
        y[int(query_idx)] == y[neighbour_idx]
    ).astype(float)

    score = float(
        np.sum(neighbour_similarities * agreement)
        / np.sum(neighbour_similarities)
    )

    return (
        score,
        len(neighbour_idx),
        float(neighbour_similarities.mean())
    )

# ------------------------------------------------------------
# C. Generate cross-fitted reliability for training examples
# ------------------------------------------------------------
print("\n[B] Calculating inner-cross-fitted training reliability")

S_train_by_outer_fold = {}
S_test_by_outer_fold = {}

records = []

unique_outer_folds = np.sort(np.unique(outer_folds))

for fold_number, outer_fold in enumerate(
    unique_outer_folds, start=1
):

    outer_train_idx = np.flatnonzero(
        outer_folds != outer_fold
    )

    outer_test_idx = np.flatnonzero(
        outer_folds == outer_fold
    )

    outer_train_groups = groups[outer_train_idx]

    number_of_inner_groups = len(
        np.unique(outer_train_groups)
    )

    n_inner = min(
        N_INNER_FOLDS,
        number_of_inner_groups
    )

    if n_inner < 2:
        raise ValueError(
            f"Not enough inner groups for outer fold {outer_fold}."
        )

    inner_cv = GroupKFold(n_splits=n_inner)

    # Allocate scores in the same order as outer_train_idx.
    train_scores = np.full(
        len(outer_train_idx), np.nan
    )

    train_neighbour_counts = np.zeros(
        len(outer_train_idx), dtype=int
    )

    train_mean_similarities = np.zeros(
        len(outer_train_idx), dtype=float
    )

    for inner_train_pos, inner_valid_pos in inner_cv.split(
        np.zeros(len(outer_train_idx)),
        y[outer_train_idx],
        groups=outer_train_groups
    ):

        inner_reference_idx = (
            outer_train_idx[inner_train_pos]
        )

        inner_query_idx = (
            outer_train_idx[inner_valid_pos]
        )

        # Every query is from a component absent in its reference set.
        reference_components = set(
            groups[inner_reference_idx]
        )

        query_components = set(
            groups[inner_query_idx]
        )

        if reference_components.intersection(query_components):
            raise RuntimeError(
                "Inner component leakage detected."
            )

        for global_idx in inner_query_idx:

            # Translate global row index to outer-train position.
            pos = np.searchsorted(
                outer_train_idx, global_idx
            )

            score, n_neighbours, mean_similarity = (
                calculate_neighbour_score(
                    global_idx,
                    inner_reference_idx,
                    k=K_NEIGHBORS
                )
            )

            train_scores[pos] = score
            train_neighbour_counts[pos] = n_neighbours
            train_mean_similarities[pos] = mean_similarity

    if np.isnan(train_scores).any():
        raise RuntimeError(
            f"Missing inner reliability values in outer fold {outer_fold}."
        )

    # Store fold-specific training scores.
    train_result = pd.DataFrame({
        "row_index": outer_train_idx,
        "outer_fold": int(outer_fold),
        "role": "outer_training_cross_fitted",
        "unitox_label": y[outer_train_idx],
        "original_inchikey": original_keys[outer_train_idx],
        "parent_inchikey": parent_keys[outer_train_idx],
        "component_id": groups[outer_train_idx],
        "S_neighbour_final": train_scores,
        "n_neighbours": train_neighbour_counts,
        "mean_neighbour_similarity": train_mean_similarities
    })

    S_train_by_outer_fold[int(outer_fold)] = train_result.copy()

    records.extend(
        train_result.to_dict("records")
    )

    # --------------------------------------------------------
    # D. Calculate diagnostic scores for outer-test examples
    # --------------------------------------------------------
    # These scores are for reliability/external diagnostics only.
    # They are not used as model training weights.
    test_scores = np.full(
        len(outer_test_idx), np.nan
    )

    test_counts = np.zeros(
        len(outer_test_idx), dtype=int
    )

    test_mean_similarities = np.zeros(
        len(outer_test_idx), dtype=float
    )

    for pos, global_idx in enumerate(outer_test_idx):

        score, n_neighbours, mean_similarity = (
            calculate_neighbour_score(
                global_idx,
                outer_train_idx,
                k=K_NEIGHBORS
            )
        )

        test_scores[pos] = score
        test_counts[pos] = n_neighbours
        test_mean_similarities[pos] = mean_similarity

    test_result = pd.DataFrame({
        "row_index": outer_test_idx,
        "outer_fold": int(outer_fold),
        "role": "outer_test_diagnostic",
        "unitox_label": y[outer_test_idx],
        "original_inchikey": original_keys[outer_test_idx],
        "parent_inchikey": parent_keys[outer_test_idx],
        "component_id": groups[outer_test_idx],
        "S_neighbour_final": test_scores,
        "n_neighbours": test_counts,
        "mean_neighbour_similarity": test_mean_similarities
    })

    S_test_by_outer_fold[int(outer_fold)] = test_result.copy()

    records.extend(
        test_result.to_dict("records")
    )

    print(
        f"Outer fold {fold_number}/10 | "
        f"train={len(outer_train_idx)} | "
        f"test={len(outer_test_idx)} | "
        f"mean S_train={train_scores.mean():.4f} | "
        f"mean S_test={test_scores.mean():.4f}"
    )

# ------------------------------------------------------------
# E. Assemble and validate final fold-specific score dataset
# ------------------------------------------------------------
S_nested_df = pd.DataFrame(records)

S_training_all = S_nested_df[
    S_nested_df["role"] == "outer_training_cross_fitted"
].copy()

S_test_all = S_nested_df[
    S_nested_df["role"] == "outer_test_diagnostic"
].copy()

# Every compound must appear exactly once as an outer-test diagnostic.
test_coverage = S_test_all["row_index"].value_counts()

assert len(S_test_all) == N
assert test_coverage.nunique() == 1
assert test_coverage.iloc[0] == 1

# All training scores must be cross-fitted.
assert S_training_all["S_neighbour_final"].notna().all()
assert S_test_all["S_neighbour_final"].notna().all()

assert S_nested_df["S_neighbour_final"].between(0, 1).all()

# ------------------------------------------------------------
# F. Print summaries
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[C] FINAL S_i SUMMARY")
print("=" * 90)

print("\nTraining reliability scores (repeated across outer experiments):")
print(
    S_training_all["S_neighbour_final"]
    .describe()
    .round(5)
    .to_string()
)

print("\nHeld-out reliability scores (one per compound):")
print(
    S_test_all["S_neighbour_final"]
    .describe()
    .round(5)
    .to_string()
)

print("\nHeld-out neighbour counts:")
print(
    S_test_all["n_neighbours"]
    .describe()
    .round(3)
    .to_string()
)

print(
    "\nHeld-out queries with neutral fallback:",
    int(
        (
            S_test_all["n_neighbours"] == 0
        ).sum()
    )
)

# ------------------------------------------------------------
# G. Verify no exact-identity neighbours were included
# ------------------------------------------------------------
print("\n[D] Identity exclusion and fold coverage")

print("Outer test compounds:", len(S_test_all))
print("Unique test compounds:", S_test_all["row_index"].nunique())
print("Training score records:", len(S_training_all))
print("Outer folds completed:", S_nested_df["outer_fold"].nunique())

print(
    "Training score range:",
    round(S_training_all["S_neighbour_final"].min(), 5),
    "to",
    round(S_training_all["S_neighbour_final"].max(), 5)
)

print(
    "Held-out score range:",
    round(S_test_all["S_neighbour_final"].min(), 5),
    "to",
    round(S_test_all["S_neighbour_final"].max(), 5)
)

# ------------------------------------------------------------
# H. Save the final score files
# ------------------------------------------------------------
S_nested_df.to_csv(
    "final_nested_identity_aware_neighbour_reliability.csv",
    index=False
)

S_test_all.to_csv(
    "final_heldout_neighbour_reliability_per_compound.csv",
    index=False
)

print("\n[E] Saved files:")
print("1. final_nested_identity_aware_neighbour_reliability.csv")
print("2. final_heldout_neighbour_reliability_per_compound.csv")

print("\nSTEP 3 COMPLETE")
print("Original data, fingerprints and historical scores were not overwritten.")
print("=" * 90)


STEP 3 — NESTED MOLECULAR-NEIGHBOUR RELIABILITY

[A] Protocol verification
Samples: 1349
Outer folds: 10
Curated components: 774
Exact InChIKey fold crossings: 0
scaffold fold crossings: 0
component fold crossings: 0

[B] Calculating inner-cross-fitted training reliability
Outer fold 1/10 | train=1190 | test=159 | mean S_train=0.5770 | mean S_test=0.5191
Outer fold 2/10 | train=1215 | test=134 | mean S_train=0.5787 | mean S_test=0.5789
Outer fold 3/10 | train=1217 | test=132 | mean S_train=0.5712 | mean S_test=0.5700
Outer fold 4/10 | train=1217 | test=132 | mean S_train=0.5580 | mean S_test=0.6075
Outer fold 5/10 | train=1217 | test=132 | mean S_train=0.5713 | mean S_test=0.6013
Outer fold 6/10 | train=1217 | test=132 | mean S_train=0.5641 | mean S_test=0.6245
Outer fold 7/10 | train=1217 | test=132 | mean S_train=0.5709 | mean S_test=0.5900
Outer fold 8/10 | train=1217 | test=132 | mean S_train=0.5825 | mean S_test=0.5351
Outer fold 9/10 | train=1217 | test=132 | mean S_train=0.5659 

In [104]:

# STEP 4 — NESTED MODEL-AGREEMENT RELIABILITY (M_i)
#
# Continuous M_i = probability assigned to the observed UniTox label.
# Hard M agreement is also retained as a diagnostic.
#
# Uses the same curated outer folds and inner component grouping
# as Step 3. Does not modify historical models or datasets.

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

print("=" * 90)
print("STEP 4 — NESTED MODEL-AGREEMENT RELIABILITY")
print("=" * 90)

# ------------------------------------------------------------
# A. Validate inputs and protocol
# ------------------------------------------------------------
N = len(work_df)

X_model = np.asarray(X_morgan)
y_model = work_df[
    "liver_toxicity_binary_rating_0_1"
].to_numpy().astype(int)

outer_ids = np.asarray(
    curated_outer_fold_ids, dtype=int
)

group_ids = np.asarray(
    curated_component_ids, dtype=int
)

assert X_model.shape == (N, 2048)
assert len(y_model) == len(outer_ids) == len(group_ids)
assert len(np.unique(outer_ids)) == 10

if not np.array_equal(
    y_model,
    work_df["liver_toxicity_binary_rating_0_1"].to_numpy()
):
    raise RuntimeError("Target alignment error.")

N_INNER_FOLDS = 5
N_ESTIMATORS = 500
RANDOM_STATE = 42

print("\n[A] Input verification")
print("Feature matrix:", X_model.shape)
print("Outer folds:", len(np.unique(outer_ids)))
print("Curated groups:", len(np.unique(group_ids)))
print("Training target distribution:",
      pd.Series(y_model).value_counts().sort_index().to_dict())

# ------------------------------------------------------------
# B. Probability alignment helper
# ------------------------------------------------------------
def aligned_binary_probabilities(model, X_eval):
    """
    Return P(y=1) and P(y=0), correctly aligned to classes [0, 1].
    """

    raw = model.predict_proba(X_eval)

    aligned = np.zeros((len(X_eval), 2), dtype=float)

    for col_idx, class_label in enumerate(model.classes_):
        aligned[:, int(class_label)] = raw[:, col_idx]

    if not np.allclose(aligned.sum(axis=1), 1.0):
        raise RuntimeError(
            "Probability alignment failed. "
            "Check whether both classes exist in model training."
        )

    return aligned

# ------------------------------------------------------------
# C. Generate cross-fitted M_i for outer-training samples
# ------------------------------------------------------------
M_train_by_outer_fold = {}
M_records = []

unique_folds = np.sort(np.unique(outer_ids))

for fold_number, outer_fold in enumerate(
    unique_folds, start=1
):

    outer_train_idx = np.flatnonzero(
        outer_ids != outer_fold
    )

    outer_test_idx = np.flatnonzero(
        outer_ids == outer_fold
    )

    inner_groups = group_ids[outer_train_idx]

    n_inner = min(
        N_INNER_FOLDS,
        len(np.unique(inner_groups))
    )

    if n_inner < 2:
        raise ValueError(
            f"Insufficient inner groups in outer fold {outer_fold}."
        )

    inner_cv = GroupKFold(n_splits=n_inner)

    M_train = np.full(len(outer_train_idx), np.nan)
    hard_agreement = np.full(len(outer_train_idx), -1, dtype=int)
    p1_values = np.full(len(outer_train_idx), np.nan)
    model_confidence = np.full(len(outer_train_idx), np.nan)

    for inner_train_pos, inner_valid_pos in inner_cv.split(
        X_model[outer_train_idx],
        y_model[outer_train_idx],
        groups=inner_groups
    ):

        inner_train_idx = outer_train_idx[inner_train_pos]
        inner_valid_idx = outer_train_idx[inner_valid_pos]

        # Strict component-exclusion check.
        train_components = set(group_ids[inner_train_idx])
        valid_components = set(group_ids[inner_valid_idx])

        if train_components.intersection(valid_components):
            raise RuntimeError(
                "Inner component leakage detected during M_i calculation."
            )

        # Train the model without the inner validation components.
        model = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(
            X_model[inner_train_idx],
            y_model[inner_train_idx]
        )

        probability_matrix = aligned_binary_probabilities(
            model,
            X_model[inner_valid_idx]
        )

        p1 = probability_matrix[:, 1]
        y_valid = y_model[inner_valid_idx]

        p_true = np.where(
            y_valid == 1,
            p1,
            1.0 - p1
        )

        predicted = (p1 >= 0.5).astype(int)
        agreement = (predicted == y_valid).astype(int)

        confidence = np.maximum(p1, 1.0 - p1)

        M_train[inner_valid_pos] = p_true
        hard_agreement[inner_valid_pos] = agreement
        p1_values[inner_valid_pos] = p1
        model_confidence[inner_valid_pos] = confidence

    if np.isnan(M_train).any():
        raise RuntimeError(
            f"Missing M_i values in outer fold {outer_fold}."
        )

    if np.any(hard_agreement < 0):
        raise RuntimeError(
            f"Missing hard agreement values in outer fold {outer_fold}."
        )

    result = pd.DataFrame({
        "row_index": outer_train_idx,
        "outer_fold": int(outer_fold),
        "role": "outer_training_cross_fitted",
        "unitox_label": y_model[outer_train_idx],
        "M_model_agreement": M_train,
        "M_hard_agreement": hard_agreement,
        "M_probability_positive": p1_values,
        "M_model_confidence": model_confidence,
        "component_id": group_ids[outer_train_idx]
    })

    M_train_by_outer_fold[int(outer_fold)] = result.copy()
    M_records.extend(result.to_dict("records"))

    print(
        f"Outer fold {fold_number}/10 | "
        f"train={len(outer_train_idx)} | "
        f"test={len(outer_test_idx)} | "
        f"mean M={M_train.mean():.4f} | "
        f"inner hard agreement={hard_agreement.mean():.4f}"
    )

# ------------------------------------------------------------
# D. Assemble training reliability dataset
# ------------------------------------------------------------
M_nested_training_df = pd.DataFrame(M_records)

assert len(M_nested_training_df) == N * 9
assert M_nested_training_df["M_model_agreement"].between(0, 1).all()
assert M_nested_training_df["M_hard_agreement"].isin([0, 1]).all()

# Every outer-training appearance should have unique row + outer fold.
if M_nested_training_df.duplicated(
    subset=["row_index", "outer_fold"]
).any():
    raise RuntimeError("Duplicate row/outer-fold score records found.")

# ------------------------------------------------------------
# E. Summaries
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("[B] MODEL-AGREEMENT SCORE SUMMARY")
print("=" * 90)

print("\nContinuous M_i:")
print(
    M_nested_training_df["M_model_agreement"]
    .describe()
    .round(5)
    .to_string()
)

print("\nHard agreement rate:")
print(
    M_nested_training_df["M_hard_agreement"]
    .value_counts(normalize=True)
    .sort_index()
    .round(5)
    .to_string()
)

print("\nModel confidence:")
print(
    M_nested_training_df["M_model_confidence"]
    .describe()
    .round(5)
    .to_string()
)

print("\nM_i by observed UniTox label:")
display(
    M_nested_training_df.groupby("unitox_label").agg(
        n=("M_model_agreement", "size"),
        mean_M=("M_model_agreement", "mean"),
        std_M=("M_model_agreement", "std"),
        mean_hard_agreement=("M_hard_agreement", "mean")
    ).round(5)
)

# ------------------------------------------------------------
# F. Save results
# ------------------------------------------------------------
M_nested_training_df.to_csv(
    "final_nested_model_agreement_training_scores.csv",
    index=False
)

M_fold_summary = (
    M_nested_training_df.groupby("outer_fold")
    .agg(
        n=("M_model_agreement", "size"),
        mean_M=("M_model_agreement", "mean"),
        std_M=("M_model_agreement", "std"),
        hard_agreement_rate=("M_hard_agreement", "mean"),
        mean_confidence=("M_model_confidence", "mean")
    )
    .reset_index()
)

M_fold_summary.to_csv(
    "final_nested_model_agreement_fold_summary.csv",
    index=False
)

print("\n[C] Saved files:")
print("1. final_nested_model_agreement_training_scores.csv")
print("2. final_nested_model_agreement_fold_summary.csv")

print("\nSTEP 4 COMPLETE")
print("Only cross-fitted outer-training scores are saved here.")
print("No held-out outer labels were used to calculate training M_i.")
print("=" * 90)


STEP 4 — NESTED MODEL-AGREEMENT RELIABILITY

[A] Input verification
Feature matrix: (1349, 2048)
Outer folds: 10
Curated groups: 774
Training target distribution: {0: 613, 1: 736}
Outer fold 1/10 | train=1190 | test=159 | mean M=0.5568 | inner hard agreement=0.6294
Outer fold 2/10 | train=1215 | test=134 | mean M=0.5697 | inner hard agreement=0.6716
Outer fold 3/10 | train=1217 | test=132 | mean M=0.5569 | inner hard agreement=0.6163
Outer fold 4/10 | train=1217 | test=132 | mean M=0.5510 | inner hard agreement=0.6302
Outer fold 5/10 | train=1217 | test=132 | mean M=0.5637 | inner hard agreement=0.6508
Outer fold 6/10 | train=1217 | test=132 | mean M=0.5553 | inner hard agreement=0.6401
Outer fold 7/10 | train=1217 | test=132 | mean M=0.5662 | inner hard agreement=0.6606
Outer fold 8/10 | train=1217 | test=132 | mean M=0.5637 | inner hard agreement=0.6631
Outer fold 9/10 | train=1217 | test=132 | mean M=0.5541 | inner hard agreement=0.6574
Outer fold 10/10 | train=1217 | test=132 | mea

,n,mean_M,std_M,mean_hard_agreement
unitox_label,,,,
0,5517,0.49889,0.17069,0.46692
1,6624,0.61142,0.14337,0.79771



[C] Saved files:
1. final_nested_model_agreement_training_scores.csv
2. final_nested_model_agreement_fold_summary.csv

STEP 4 COMPLETE
Only cross-fitted outer-training scores are saved here.
No held-out outer labels were used to calculate training M_i.


In [106]:

# ============================================================
# STEP 5.1 — EXACT-MATCH EXTERNAL REFERENCE AGREEMENT (A_i)
# Corrected for DILIrank dataset containing SMILES, not InChIKey
# ============================================================

import numpy as np
import pandas as pd
import re
from pathlib import Path
from rdkit import Chem

print("=" * 90)
print("STEP 5 — EXTERNAL REFERENCE AGREEMENT")
print("=" * 90)

# ------------------------------------------------------------
# 1. INPUT VERIFICATION
# ------------------------------------------------------------

required_objects = [
    "work_df",
    "dilirank",
    "curated_outer_fold_ids",
    "curated_component_ids"
]

for obj in required_objects:
    assert obj in globals(), f"Missing object: {obj}"

assert "inchikey" in work_df.columns
assert "SMILES" in dilirank.columns
assert "DILIConcern" in dilirank.columns
assert "liver_toxicity_binary_rating_0_1" in work_df.columns

base_df = work_df.reset_index(drop=True).copy()

base_df["row_index"] = np.arange(len(base_df))
base_df["outer_fold"] = np.asarray(curated_outer_fold_ids)
base_df["component_id"] = np.asarray(curated_component_ids)

assert len(base_df) == len(curated_outer_fold_ids)
assert len(base_df) == len(curated_component_ids)

base_df["unitox_label"] = pd.to_numeric(
    base_df["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
)

assert base_df["unitox_label"].notna().all()
assert set(base_df["unitox_label"].unique()).issubset({0, 1})

# ------------------------------------------------------------
# 2. GENERATE DILIRANK INCHIKEY FROM SMILES
# ------------------------------------------------------------

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return np.nan

    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())

        if mol is None:
            return np.nan

        return Chem.MolToInchiKey(mol)

    except Exception:
        return np.nan


dili_df = dilirank[
    ["SMILES", "DILIConcern"]
].copy()

dili_df["original_inchikey"] = dili_df["SMILES"].apply(
    smiles_to_inchikey
)

print("\n[A] DILIrank InChIKey generation")
print("Total DILIrank records:", len(dili_df))
print(
    "Successfully generated InChIKeys:",
    dili_df["original_inchikey"].notna().sum()
)
print(
    "Failed / missing SMILES:",
    dili_df["original_inchikey"].isna().sum()
)

# ------------------------------------------------------------
# 3. NORMALIZE DILIRANK CATEGORIES
# ------------------------------------------------------------

def normalize_dilirank_category(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if "most" in value:
        return "Most"

    if "less" in value:
        return "Less"

    if re.search(r"\bno\b", value) or value.startswith("no-"):
        return "No"

    return np.nan


dili_df["dilirank_category"] = dili_df[
    "DILIConcern"
].apply(normalize_dilirank_category)

print("\nOriginal DILIConcern distribution:")
print(dilirank["DILIConcern"].value_counts(dropna=False))

unknown_categories = dili_df.loc[
    dili_df["DILIConcern"].notna()
    & dili_df["dilirank_category"].isna(),
    "DILIConcern"
].unique()

if len(unknown_categories):
    raise ValueError(
        "Unrecognized DILIConcern values. Review these before proceeding: "
        f"{unknown_categories}"
    )

dili_df = dili_df.dropna(
    subset=["original_inchikey", "dilirank_category"]
).copy()

print("\nNormalized DILIrank category distribution:")
print(dili_df["dilirank_category"].value_counts())

# ------------------------------------------------------------
# 4. CHECK DUPLICATE INCHIKEYS AND CATEGORY CONFLICTS
# ------------------------------------------------------------

category_counts = (
    dili_df.groupby("original_inchikey")["dilirank_category"]
    .nunique()
)

conflicting_keys = category_counts[category_counts > 1]

print("\n[B] DILIrank InChIKey consistency")
print("Unique generated InChIKeys:", len(category_counts))
print("Keys with conflicting categories:", len(conflicting_keys))

if len(conflicting_keys):
    raise ValueError(
        "The same InChIKey has multiple DILIrank categories. "
        "Resolve these records before calculating A_i.\n"
        + str(conflicting_keys.head(20))
    )

dili_lookup = (
    dili_df.drop_duplicates(subset="original_inchikey")
    [["original_inchikey", "dilirank_category"]]
    .copy()
)

# ------------------------------------------------------------
# 5. PREPARE UNITOX INCHIKEYS
# ------------------------------------------------------------

base_df["original_inchikey"] = (
    base_df["inchikey"]
    .astype("string")
    .str.strip()
    .str.upper()
)

base_df.loc[
    base_df["original_inchikey"].isin(["", "NAN", "NONE"]),
    "original_inchikey"
] = pd.NA

# ------------------------------------------------------------
# 6. EXACT FULL-INCHIKEY MATCH
# ------------------------------------------------------------

base_df = base_df.merge(
    dili_lookup,
    on="original_inchikey",
    how="left",
    validate="many_to_one"
)

base_df["dilirank_matched"] = (
    base_df["dilirank_category"].notna()
)

# DILIrank expected label:
# No   -> 0
# Most -> 1
# Less -> unavailable / ambiguous

base_df["external_expected_label"] = (
    base_df["dilirank_category"]
    .map({
        "No": 0,
        "Most": 1
    })
)

# Independent-reference agreement:
# A_i = 1 if UniTox agrees with DILIrank
# A_i = 0 if UniTox disagrees with DILIrank
# A_i = NaN for Less or unmatched records

base_df["A_i"] = np.where(
    base_df["external_expected_label"].notna(),
    (
        base_df["unitox_label"]
        == base_df["external_expected_label"]
    ).astype(float),
    np.nan
)

base_df["A_available"] = base_df["A_i"].notna()

# ------------------------------------------------------------
# 7. MATCHING AND AGREEMENT SUMMARY
# ------------------------------------------------------------

print("\n[C] Exact full-InChIKey matching")

print("Total UniTox compounds:", len(base_df))

print(
    "Exact DILIrank matched UniTox rows:",
    int(base_df["dilirank_matched"].sum())
)

print(
    "Unique exact-matched InChIKeys:",
    base_df.loc[
        base_df["dilirank_matched"],
        "original_inchikey"
    ].nunique()
)

print("\nMatched DILIrank category distribution:")
print(
    base_df.loc[
        base_df["dilirank_matched"],
        "dilirank_category"
    ].value_counts()
)

print("\nA_i availability:")
print(
    base_df["A_available"].value_counts().rename({
        True: "Available",
        False: "Missing"
    })
)

print("\nA_i distribution:")
print(base_df["A_i"].value_counts(dropna=False).sort_index())

print("\nDILIrank category versus UniTox label:")
print(
    pd.crosstab(
        base_df["dilirank_category"],
        base_df["unitox_label"],
        margins=True,
        dropna=False
    )
)

print("\nA_i by DILIrank category:")
print(
    base_df.loc[base_df["A_available"]]
    .groupby("dilirank_category")
    .agg(
        n=("A_i", "size"),
        mean_A=("A_i", "mean"),
        agreements=("A_i", "sum")
    )
)

# ------------------------------------------------------------
# 8. CHECK EXACT IDENTITY DOES NOT CROSS OUTER FOLDS
# ------------------------------------------------------------

key_fold_counts = (
    base_df.dropna(subset=["original_inchikey"])
    .groupby("original_inchikey")["outer_fold"]
    .nunique()
)

cross_fold_keys = key_fold_counts[key_fold_counts > 1]

print("\n[D] Identity leakage check")
print(
    "InChIKeys appearing in multiple outer folds:",
    len(cross_fold_keys)
)

assert len(cross_fold_keys) == 0, (
    "Exact InChIKey identity crosses outer folds. "
    "Do not proceed until the split is fixed."
)

# ------------------------------------------------------------
# 9. CREATE OUTER-TRAINING AND HELD-OUT TABLES
# ------------------------------------------------------------

training_parts = []
heldout_parts = []

for fold in sorted(base_df["outer_fold"].unique()):

    # This fold's training set: exclude the current outer test fold
    train_part = base_df.loc[
        base_df["outer_fold"] != fold
    ].copy()

    train_part["role"] = "outer_training"
    train_part["scoring_outer_fold"] = int(fold)

    training_parts.append(train_part)

    # This fold's test set: retained separately for evaluation
    test_part = base_df.loc[
        base_df["outer_fold"] == fold
    ].copy()

    test_part["role"] = "outer_heldout_evaluation"
    test_part["scoring_outer_fold"] = int(fold)

    heldout_parts.append(test_part)


A_nested_training_df = pd.concat(
    training_parts,
    ignore_index=True
)

A_heldout_evaluation_df = pd.concat(
    heldout_parts,
    ignore_index=True
)

A_columns = [
    "row_index",
    "scoring_outer_fold",
    "role",
    "unitox_label",
    "original_inchikey",
    "component_id",
    "dilirank_category",
    "external_expected_label",
    "A_i",
    "A_available"
]

A_nested_training_df = A_nested_training_df[A_columns].copy()
A_heldout_evaluation_df = A_heldout_evaluation_df[A_columns].copy()

# ------------------------------------------------------------
# 10. FOLD-WISE SUMMARY
# ------------------------------------------------------------

fold_summary = (
    A_nested_training_df
    .groupby("scoring_outer_fold")
    .agg(
        training_rows=("row_index", "size"),
        A_available=("A_available", "sum"),
        A_mean=("A_i", "mean"),
        A_agreements=("A_i", "sum")
    )
    .reset_index()
)

print("\n[E] Outer-training A_i summary")
print(fold_summary.to_string(index=False))

print(
    "\nTotal outer-training score rows:",
    len(A_nested_training_df)
)

print(
    "Expected outer-training rows:",
    len(base_df) * 9
)

print(
    "Outer-heldout evaluation rows:",
    len(A_heldout_evaluation_df)
)

print(
    "Held-out compounds with evaluable A_i:",
    int(A_heldout_evaluation_df["A_available"].sum())
)

# ------------------------------------------------------------
# 11. SAVE OUTPUTS
# ------------------------------------------------------------

training_path = Path(
    "final_nested_external_agreement_training_scores.csv"
)

heldout_path = Path(
    "final_heldout_external_agreement_evaluation.csv"
)

summary_path = Path(
    "final_external_agreement_fold_summary.csv"
)

A_nested_training_df.to_csv(training_path, index=False)
A_heldout_evaluation_df.to_csv(heldout_path, index=False)
fold_summary.to_csv(summary_path, index=False)

print("\n[F] Saved files:")
print("1.", training_path)
print("2.", heldout_path)
print("3.", summary_path)

print("\n" + "=" * 90)
print("STEP 5.1 COMPLETE")
print("A_i = 1: external agreement")
print("A_i = 0: external disagreement")
print("A_i = NaN: Less category or no exact match")
print("Held-out A_i is stored separately from training A_i.")
print("=" * 90)


STEP 5 — EXTERNAL REFERENCE AGREEMENT

[A] DILIrank InChIKey generation
Total DILIrank records: 1036
Successfully generated InChIKeys: 960
Failed / missing SMILES: 76

Original DILIConcern distribution:
DILIConcern
Less-DILI-Concern    488
No-DILI-Concern      321
Most-DILI-Concern    208
No-DILI-concern       18
Most-DILI-concern      1
Name: count, dtype: int64

Normalized DILIrank category distribution:
dilirank_category
Less    471
No      291
Most    198
Name: count, dtype: int64

[B] DILIrank InChIKey consistency
Unique generated InChIKeys: 958
Keys with conflicting categories: 0

[C] Exact full-InChIKey matching
Total UniTox compounds: 1349
Exact DILIrank matched UniTox rows: 267
Unique exact-matched InChIKeys: 267

Matched DILIrank category distribution:
dilirank_category
Less    145
No       67
Most     55
Name: count, dtype: int64

A_i availability:
A_available
Missing      1227
Available     122
Name: count, dtype: int64

A_i distribution:
A_i
0.0      17
1.0     105
NaN    

In [108]:

# ============================================================
# STEP 5.2 — RECONCILE HISTORICAL AND NEW DILIRANK MATCHES
# ============================================================

import pandas as pd
import numpy as np
import re

assert "external_253" in globals(), (
    "The historical external_253 object is not available."
)
assert "base_df" in globals(), (
    "Run the corrected Step 5.1 cell first."
)

def detect_inchikey_column(df, preferred_names=None):
    preferred_names = preferred_names or []

    normalized = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in df.columns
    }

    for name in preferred_names:
        key = re.sub(r"[^a-z0-9]", "", name.lower())
        if key in normalized:
            return normalized[key]

    # Fallback: identify a column containing valid-looking InChIKeys.
    pattern = re.compile(r"^[A-Z]{14}-[A-Z]{10}-[A-Z]$")

    scores = {}
    for col in df.columns:
        values = df[col].dropna().astype(str).str.strip().str.upper()
        scores[col] = int(values.apply(
            lambda x: bool(pattern.match(x))
        ).sum())

    best_col = max(scores, key=scores.get)

    if scores[best_col] == 0:
        raise ValueError(
            "Could not find an InChIKey column. "
            f"Available columns: {list(df.columns)}"
        )

    return best_col


# Identify key columns.
old_key_col = detect_inchikey_column(
    external_253,
    [
        "unitox_inchikey",
        "original_inchikey",
        "inchikey",
        "InChIKey"
    ]
)

new_key_col = "original_inchikey"

print("Historical key column:", old_key_col)
print("Current key column:", new_key_col)

# Extract unique key sets.
old_keys = set(
    external_253[old_key_col]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

new_keys = set(
    base_df.loc[
        base_df["dilirank_matched"],
        new_key_col
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

old_only = old_keys - new_keys
new_only = new_keys - old_keys
common = old_keys & new_keys

print("\n" + "=" * 75)
print("MATCH RECONCILIATION")
print("=" * 75)
print("Historical unique keys:", len(old_keys))
print("Current unique keys:", len(new_keys))
print("Common keys:", len(common))
print("Historical keys missing from current:", len(old_only))
print("Additional keys in current:", len(new_only))

# Show any historical matches that are unexpectedly absent.
if old_only:
    print("\nWARNING — Historical keys absent from the current match:")
    print(sorted(old_only)[:30])
else:
    print("\nPASS — Every historical exact match is present in the current set.")

# Inspect the additional records.
if new_only:
    print("\nAdditional exact matches found in the full DILIrank search:")

    extra_df = base_df.loc[
        base_df["original_inchikey"].isin(new_only),
        [
            "generic_name",
            "original_inchikey",
            "unitox_label",
            "dilirank_category",
            "A_i"
        ]
    ].drop_duplicates()

    print(extra_df.to_string(index=False))
else:
    print("\nNo additional exact matches.")

# Check category consistency for common keys if old category exists.
old_category_candidates = [
    "dilirank_category",
    "DILIConcern",
    "DILIrank",
    "category"
]

old_category_col = None
normalized_cols = {
    re.sub(r"[^a-z0-9]", "", str(c).lower()): c
    for c in external_253.columns
}

for candidate in old_category_candidates:
    key = re.sub(r"[^a-z0-9]", "", candidate.lower())
    if key in normalized_cols:
        old_category_col = normalized_cols[key]
        break

if old_category_col is not None:
    old_check = external_253[
        [old_key_col, old_category_col]
    ].drop_duplicates().copy()

    old_check["key"] = (
        old_check[old_key_col].astype(str).str.strip().str.upper()
    )

    current_check = base_df[
        ["original_inchikey", "dilirank_category"]
    ].drop_duplicates().copy()

    merged_check = old_check.merge(
        current_check,
        left_on="key",
        right_on="original_inchikey",
        how="inner"
    )

    print("\nHistorical/current category comparison:")
    print("Historical category column:", old_category_col)
    print("Common-key category comparisons:", len(merged_check))
    print(
        merged_check[
            [old_category_col, "dilirank_category"]
        ].value_counts(dropna=False).to_string()
    )

print("\n" + "=" * 75)
if not old_only:
    print("RECONCILIATION PASSED: historical matches are retained.")
else:
    print("RECONCILIATION NEEDS REVIEW.")
print("=" * 75)


Historical key column: unitox_inchikey
Current key column: original_inchikey

MATCH RECONCILIATION
Historical unique keys: 253
Current unique keys: 267
Common keys: 253
Historical keys missing from current: 0
Additional keys in current: 14

PASS — Every historical exact match is present in the current set.

Additional exact matches found in the full DILIrank search:
        generic_name           original_inchikey  unitox_label dilirank_category  A_i
ACETYLSALICYLIC ACID BSYNRYMUTXBXSQ-UHFFFAOYSA-N             0                No  1.0
           ACYCLOVIR MKUXAQIIEYXACX-UHFFFAOYSA-N             0              Less  NaN
  ADEFOVIR DIPIVOXIL WOZSCQDILHKSGG-UHFFFAOYSA-N             1              Less  NaN
           ALBUTEROL NDAUXUAQIAJITI-UHFFFAOYSA-N             0                No  1.0
           GLYBURIDE ZNNLBTZKUZBEKO-UHFFFAOYSA-N             1              Less  NaN
          MESALAMINE KBOPZPXVLCULAV-UHFFFAOYSA-N             1              Less  NaN
OLMESARTAN MEDOXOMIL UQGKUQLK

In [109]:

# ============================================================
# STEP 6.1 — COMBINE A_i, M_i, S_i INTO RELIABILITY SCORE R_i
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 90)
print("STEP 6 — COMBINED RELIABILITY SCORE")
print("=" * 90)

# ------------------------------------------------------------
# 1. INPUT VERIFICATION
# ------------------------------------------------------------

required_objects = [
    "A_nested_training_df",
    "A_heldout_evaluation_df",
    "M_nested_training_df",
    "S_nested_df"
]

for obj in required_objects:
    assert obj in globals(), f"Missing object: {obj}. Run previous steps first."

print("\n[A] Input tables")
print("A table:", A_nested_training_df.shape)
print("M table:", M_nested_training_df.shape)
print("S table:", S_nested_df.shape)

# ------------------------------------------------------------
# 2. HELPER: STANDARDIZE OUTER FOLD COLUMN
# ------------------------------------------------------------

def standardize_fold_column(df, table_name):
    df = df.copy()

    possible_fold_cols = [
        "scoring_outer_fold",
        "outer_fold"
    ]

    fold_col = next(
        (c for c in possible_fold_cols if c in df.columns),
        None
    )

    if fold_col is None:
        raise ValueError(
            f"{table_name}: Could not identify outer-fold column. "
            f"Available columns: {list(df.columns)}"
        )

    if fold_col != "scoring_outer_fold":
        df = df.rename(
            columns={fold_col: "scoring_outer_fold"}
        )

    df["row_index"] = pd.to_numeric(
        df["row_index"], errors="raise"
    ).astype(int)

    df["scoring_outer_fold"] = pd.to_numeric(
        df["scoring_outer_fold"], errors="raise"
    ).astype(int)

    return df


A_df = standardize_fold_column(
    A_nested_training_df, "A_nested_training_df"
)

M_df = standardize_fold_column(
    M_nested_training_df, "M_nested_training_df"
)

S_df = standardize_fold_column(
    S_nested_df, "S_nested_df"
)

# ------------------------------------------------------------
# 3. IDENTIFY THE CONTINUOUS M_i COLUMN
# ------------------------------------------------------------

M_candidates = [
    "M_i",
    "M_model_agreement",
    "M_true_class_probability",
    "M_soft_agreement",
    "M_probability_true_class",
    "M_train"
]

M_score_col = next(
    (c for c in M_candidates if c in M_df.columns),
    None
)

if M_score_col is None:
    raise ValueError(
        "Could not identify the continuous M_i column.\n"
        f"Available M columns: {list(M_df.columns)}"
    )

print("\n[B] Selected reliability columns")
print("M score column:", M_score_col)

assert "S_neighbour_final" in S_df.columns, (
    "S_neighbour_final is missing from S_nested_df."
)

assert "A_i" in A_df.columns
assert "A_available" in A_df.columns

# ------------------------------------------------------------
# 4. SELECT ONLY REQUIRED FIELDS
# ------------------------------------------------------------

M_score_df = M_df[
    ["row_index", "scoring_outer_fold", M_score_col]
].copy()

M_score_df = M_score_df.rename(
    columns={M_score_col: "M_i"}
)

S_score_df = S_df[
    ["row_index", "scoring_outer_fold", "S_neighbour_final"]
].copy()

S_score_df = S_score_df.rename(
    columns={
        "S_neighbour_final": "S_i"
    }
)

A_score_df = A_df[
    [
        "row_index",
        "scoring_outer_fold",
        "A_i",
        "A_available"
    ]
].copy()

# ------------------------------------------------------------
# 5. CHECK DUPLICATE KEYS
# ------------------------------------------------------------

key_cols = [
    "row_index",
    "scoring_outer_fold"
]

for name, table in [
    ("A", A_score_df),
    ("M", M_score_df),
    ("S", S_score_df)
]:
    duplicate_count = table.duplicated(key_cols).sum()

    print(f"{name} duplicate row/fold keys:", duplicate_count)

    assert duplicate_count == 0, (
        f"{name} table has duplicate row/fold keys."
    )

# ------------------------------------------------------------
# 6. MERGE ALL THREE COMPONENTS
# ------------------------------------------------------------

combined = M_score_df.merge(
    S_score_df,
    on=key_cols,
    how="inner",
    validate="one_to_one"
)

combined = combined.merge(
    A_score_df,
    on=key_cols,
    how="inner",
    validate="one_to_one"
)

print("\n[C] Merge verification")
print("Combined rows:", len(combined))

expected_rows = len(work_df) * 9

print("Expected rows:", expected_rows)

assert len(combined) == expected_rows, (
    "Unexpected combined row count. Check fold alignment."
)

# Check that A availability flag agrees with A_i missingness.
combined["A_available"] = combined["A_available"].astype(bool)

assert (
    combined["A_available"]
    == combined["A_i"].notna()
).all(), "A_i and A_available are inconsistent."

# ------------------------------------------------------------
# 7. VALIDATE COMPONENT RANGES AND COMPLETENESS
# ------------------------------------------------------------

for col in ["M_i", "S_i"]:
    combined[col] = pd.to_numeric(
        combined[col], errors="coerce"
    )

    assert combined[col].notna().all(), (
        f"{col} contains missing values."
    )

    assert combined[col].between(0, 1).all(), (
        f"{col} contains values outside [0,1]."
    )

assert combined.loc[
    combined["A_available"], "A_i"
].between(0, 1).all()

assert combined.loc[
    ~combined["A_available"], "A_i"
].isna().all()

# ------------------------------------------------------------
# 8. CALCULATE R_i WITH MISSING-AWARE EQUAL WEIGHTS
# ------------------------------------------------------------

# All three components available:
# R_i = (A_i + M_i + S_i) / 3
#
# A_i unavailable:
# R_i = (M_i + S_i) / 2
#
# Missing A_i is excluded from numerator and denominator.

combined["R_i"] = (
    combined["M_i"]
    + combined["S_i"]
    + combined["A_i"].fillna(0)
) / (
    2 + combined["A_available"].astype(int)
)

assert combined["R_i"].notna().all()
assert combined["R_i"].between(0, 1).all()

combined["n_reliability_components"] = (
    2 + combined["A_available"].astype(int)
)

# ------------------------------------------------------------
# 9. CONVERT RELIABILITY TO SAMPLE WEIGHTS
# ------------------------------------------------------------

WEIGHT_FLOOR = 0.10

combined["raw_sample_weight"] = (
    WEIGHT_FLOOR
    + (1 - WEIGHT_FLOOR) * combined["R_i"]
)

# Normalize weights to mean 1 independently within each
# outer-training fold.

combined["sample_weight"] = (
    combined["raw_sample_weight"]
    /
    combined.groupby("scoring_outer_fold")[
        "raw_sample_weight"
    ].transform("mean")
)

assert combined["sample_weight"].notna().all()
assert (combined["sample_weight"] > 0).all()

# ------------------------------------------------------------
# 10. COMPONENT AND RELIABILITY SUMMARY
# ------------------------------------------------------------

print("\n[D] Component summary")

print(
    combined[
        ["A_i", "M_i", "S_i", "R_i", "sample_weight"]
    ].describe().round(4)
)

print("\nA availability:")
print(combined["A_available"].value_counts())

print("\nNumber of available reliability components:")
print(
    combined["n_reliability_components"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 11. SUMMARY BY OUTER FOLD
# ------------------------------------------------------------

fold_summary = (
    combined.groupby("scoring_outer_fold")
    .agg(
        training_rows=("row_index", "size"),
        A_available=("A_available", "sum"),
        mean_A=("A_i", "mean"),
        mean_M=("M_i", "mean"),
        mean_S=("S_i", "mean"),
        mean_R=("R_i", "mean"),
        std_R=("R_i", "std"),
        min_R=("R_i", "min"),
        max_R=("R_i", "max"),
        mean_raw_weight=("raw_sample_weight", "mean"),
        mean_sample_weight=("sample_weight", "mean"),
        min_sample_weight=("sample_weight", "min"),
        max_sample_weight=("sample_weight", "max")
    )
    .reset_index()
)

print("\n[E] Outer-fold reliability summary")
print(fold_summary.round(4).to_string(index=False))

# Each fold's normalized sample weights should average 1.
assert np.allclose(
    fold_summary["mean_sample_weight"].values,
    1.0,
    atol=1e-10
), "Sample weights are not normalized to mean 1."

# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

R_nested_training_df = combined.copy()

combined_path = Path(
    "final_combined_reliability_training_scores.csv"
)

summary_path = Path(
    "final_combined_reliability_fold_summary.csv"
)

R_nested_training_df.to_csv(
    combined_path,
    index=False
)

fold_summary.to_csv(
    summary_path,
    index=False
)

print("\n[F] Saved files:")
print("1.", combined_path)
print("2.", summary_path)

print("\n" + "=" * 90)
print("STEP 6.1 COMPLETE")
print("R_i combines A_i, M_i and S_i using missing-aware equal weighting.")
print("Sample weights have a 0.10 floor and mean 1 per outer fold.")
print("No outer-heldout A_i values were used in training reliability scores.")
print("=" * 90)


STEP 6 — COMBINED RELIABILITY SCORE

[A] Input tables
A table: (12141, 10)
M table: (12141, 9)
S table: (13490, 10)

[B] Selected reliability columns
M score column: M_model_agreement
A duplicate row/fold keys: 0
M duplicate row/fold keys: 0
S duplicate row/fold keys: 0

[C] Merge verification
Combined rows: 12141
Expected rows: 12141

[D] Component summary
             A_i         M_i         S_i         R_i  sample_weight
count  1098.0000  12141.0000  12141.0000  12141.0000     12141.0000
mean      0.8607      0.5603      0.5717      0.5749         1.0000
std       0.3465      0.1661      0.2234      0.1842         0.2683
min       0.0000      0.0620      0.0000      0.0310         0.2088
25%       1.0000      0.4360      0.4041      0.4388         0.8021
50%       1.0000      0.5780      0.5920      0.5922         1.0254
75%       1.0000      0.6840      0.7358      0.7174         1.2076
max       1.0000      0.9560      1.0000      0.9780         1.6144

A availability:
A_available

In [110]:

# ============================================================
# STEP 7.1 — FINAL MATCHED OUTER-FOLD EXPERIMENT
# Plain Random Forest vs Reliability-Weighted Random Forest
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    log_loss,
    brier_score_loss
)

from scipy.stats import ttest_rel, wilcoxon

print("=" * 95)
print("STEP 7 — FINAL PLAIN RF VS RELIABILITY-WEIGHTED RF")
print("=" * 95)

# ------------------------------------------------------------
# 1. INPUT VERIFICATION
# ------------------------------------------------------------

required_objects = [
    "work_df",
    "X_morgan",
    "curated_outer_fold_ids",
    "curated_component_ids",
    "R_nested_training_df"
]

for obj in required_objects:
    assert obj in globals(), f"Missing required object: {obj}"

target_col = "liver_toxicity_binary_rating_0_1"

data = work_df.reset_index(drop=True).copy()

X = np.asarray(X_morgan)
y = pd.to_numeric(
    data[target_col], errors="coerce"
).to_numpy(dtype=int)

outer_ids = np.asarray(curated_outer_fold_ids)
component_ids = np.asarray(curated_component_ids)

assert X.ndim == 2
assert X.shape == (len(data), 2048)
assert len(y) == len(data)
assert len(outer_ids) == len(data)
assert len(component_ids) == len(data)
assert np.isfinite(X).all()
assert set(np.unique(y)).issubset({0, 1})

n_samples = len(y)
outer_folds = sorted(np.unique(outer_ids).tolist())

assert len(outer_folds) == 10, (
    f"Expected 10 outer folds, found {len(outer_folds)}"
)

print("\n[A] Input verification")
print("Feature matrix:", X.shape)
print("Target distribution:", pd.Series(y).value_counts().sort_index().to_dict())
print("Number of outer folds:", len(outer_folds))
print("Number of curated components:", len(np.unique(component_ids)))

# ------------------------------------------------------------
# 2. VERIFY OUTER SPLIT IDENTITY / SCAFFOLD ISOLATION
# ------------------------------------------------------------

assert "scaffold" in data.columns, "Scaffold column is missing."

split_audit = pd.DataFrame({
    "outer_fold": outer_ids,
    "component_id": component_ids,
    "scaffold": data["scaffold"].astype(str).to_numpy()
})

component_fold_counts = (
    split_audit.groupby("component_id")["outer_fold"].nunique()
)

scaffold_fold_counts = (
    split_audit.groupby("scaffold")["outer_fold"].nunique()
)

component_crossings = int((component_fold_counts > 1).sum())
scaffold_crossings = int((scaffold_fold_counts > 1).sum())

print("\n[B] Split leakage checks")
print("Curated components crossing folds:", component_crossings)
print("Scaffolds crossing folds:", scaffold_crossings)

assert component_crossings == 0
assert scaffold_crossings == 0

# ------------------------------------------------------------
# 3. VERIFY RELIABILITY TABLE
# ------------------------------------------------------------

rel = R_nested_training_df.copy()

required_rel_cols = [
    "row_index",
    "scoring_outer_fold",
    "R_i",
    "sample_weight"
]

for col in required_rel_cols:
    assert col in rel.columns, (
        f"Missing column in R_nested_training_df: {col}"
    )

rel["row_index"] = rel["row_index"].astype(int)
rel["scoring_outer_fold"] = rel["scoring_outer_fold"].astype(int)

assert not rel.duplicated(
    ["row_index", "scoring_outer_fold"]
).any()

assert rel["sample_weight"].notna().all()
assert (rel["sample_weight"] > 0).all()
assert rel["R_i"].between(0, 1).all()

# ------------------------------------------------------------
# 4. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_metrics(y_true, probabilities):

    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "log_loss": log_loss(
            y_true,
            np.column_stack([
                1 - probabilities,
                probabilities
            ]),
            labels=[0, 1]
        ),
        "brier_score": brier_score_loss(
            y_true, probabilities
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def positive_class_probability(model, X_test):

    probabilities = model.predict_proba(X_test)

    positive_positions = np.where(model.classes_ == 1)[0]

    assert len(positive_positions) == 1

    return probabilities[:, positive_positions[0]]


# ------------------------------------------------------------
# 5. MODEL CONFIGURATION
# ------------------------------------------------------------

N_ESTIMATORS = 500
MODEL_RANDOM_STATE = 42

print("\n[C] Model configuration")
print("Model: RandomForestClassifier")
print("Trees:", N_ESTIMATORS)
print("Random state:", MODEL_RANDOM_STATE)
print("Features: Morgan fingerprints, radius 2, 2048 bits")
print("Plain model: no sample weights")
print("Weighted model: R_i-derived sample weights")

# ------------------------------------------------------------
# 6. PREALLOCATE OUT-OF-FOLD PREDICTIONS
# ------------------------------------------------------------

oof_plain = np.full(n_samples, np.nan, dtype=float)
oof_weighted = np.full(n_samples, np.nan, dtype=float)

fold_metric_records = []
fold_weight_records = []

# ------------------------------------------------------------
# 7. OUTER-FOLD TRAINING AND EVALUATION
# ------------------------------------------------------------

for position, fold in enumerate(outer_folds, start=1):

    train_idx = np.where(outer_ids != fold)[0]
    test_idx = np.where(outer_ids == fold)[0]

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Retrieve weights specifically calculated for this
    # outer fold's training set.
    fold_rel = rel.loc[
        rel["scoring_outer_fold"] == int(fold)
    ].copy()

    assert len(fold_rel) == len(train_idx), (
        f"Fold {fold}: reliability row count mismatch. "
        f"Expected {len(train_idx)}, got {len(fold_rel)}"
    )

    assert fold_rel["row_index"].is_unique

    fold_rel = fold_rel.set_index("row_index")

    # Verify exact row-index alignment.
    assert set(fold_rel.index) == set(train_idx), (
        f"Fold {fold}: reliability scores do not match "
        "the outer training row indices."
    )

    sample_weights = (
        fold_rel.loc[train_idx, "sample_weight"]
        .to_numpy(dtype=float)
    )

    assert np.isfinite(sample_weights).all()
    assert (sample_weights > 0).all()
    assert np.isclose(sample_weights.mean(), 1.0, atol=1e-10)

    # Same model configuration and random state for both models.
    plain_model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=MODEL_RANDOM_STATE,
        n_jobs=-1
    )

    weighted_model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=MODEL_RANDOM_STATE,
        n_jobs=-1
    )

    # Plain Random Forest.
    plain_model.fit(X_train, y_train)

    # Reliability-weighted Random Forest.
    weighted_model.fit(
        X_train,
        y_train,
        sample_weight=sample_weights
    )

    # Held-out predictions.
    p_plain = positive_class_probability(
        plain_model, X_test
    )

    p_weighted = positive_class_probability(
        weighted_model, X_test
    )

    assert len(p_plain) == len(test_idx)
    assert len(p_weighted) == len(test_idx)

    oof_plain[test_idx] = p_plain
    oof_weighted[test_idx] = p_weighted

    # Calculate fold metrics.
    metrics_plain = calculate_metrics(y_test, p_plain)
    metrics_weighted = calculate_metrics(y_test, p_weighted)

    fold_metric_records.append({
        "outer_fold": int(fold),
        "model": "Plain_RF",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        **metrics_plain
    })

    fold_metric_records.append({
        "outer_fold": int(fold),
        "model": "Reliability_Weighted_RF",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        **metrics_weighted
    })

    fold_weight_records.append({
        "outer_fold": int(fold),
        "n_train": len(train_idx),
        "mean_weight": sample_weights.mean(),
        "std_weight": sample_weights.std(),
        "min_weight": sample_weights.min(),
        "max_weight": sample_weights.max()
    })

    print(
        f"Outer fold {position}/{len(outer_folds)} | "
        f"train={len(train_idx)} | test={len(test_idx)} | "
        f"Plain ROC={metrics_plain['roc_auc']:.4f} | "
        f"Weighted ROC={metrics_weighted['roc_auc']:.4f} | "
        f"Delta={metrics_weighted['roc_auc'] - metrics_plain['roc_auc']:+.4f} | "
        f"Plain PR={metrics_plain['pr_auc']:.4f} | "
        f"Weighted PR={metrics_weighted['pr_auc']:.4f}"
    )

# ------------------------------------------------------------
# 8. VERIFY COMPLETE OOF PREDICTIONS
# ------------------------------------------------------------

assert np.isfinite(oof_plain).all()
assert np.isfinite(oof_weighted).all()

print("\n[D] OOF prediction verification")
print("Total compounds:", n_samples)
print("Plain predictions:", np.isfinite(oof_plain).sum())
print("Weighted predictions:", np.isfinite(oof_weighted).sum())

# ------------------------------------------------------------
# 9. POOLED OUTER-HELDOUT METRICS
# ------------------------------------------------------------

pooled_plain = calculate_metrics(y, oof_plain)
pooled_weighted = calculate_metrics(y, oof_weighted)

pooled_metrics_df = pd.DataFrame([
    {
        "model": "Plain_RF",
        **pooled_plain
    },
    {
        "model": "Reliability_Weighted_RF",
        **pooled_weighted
    }
])

metric_names = [
    "roc_auc",
    "pr_auc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "log_loss",
    "brier_score"
]

pooled_delta = {
    metric: pooled_weighted[metric] - pooled_plain[metric]
    for metric in metric_names
}

print("\n[E] POOLED OUTER-HELDOUT PERFORMANCE")
print(
    pooled_metrics_df.set_index("model")[
        metric_names + ["tn", "fp", "fn", "tp"]
    ].round(4).to_string()
)

print("\nWeighted minus plain pooled metric differences:")
for metric, delta in pooled_delta.items():
    print(f"{metric:20s}: {delta:+.5f}")

# ------------------------------------------------------------
# 10. FOLD-WISE PAIRED COMPARISON
# ------------------------------------------------------------

fold_metrics_df = pd.DataFrame(fold_metric_records)

wide = fold_metrics_df.pivot(
    index="outer_fold",
    columns="model",
    values=metric_names
)

paired_records = []

for metric in metric_names:

    plain_values = wide[metric]["Plain_RF"].to_numpy()
    weighted_values = wide[metric][
        "Reliability_Weighted_RF"
    ].to_numpy()

    deltas = weighted_values - plain_values

    paired_records.append({
        "metric": metric,
        "plain_mean": np.mean(plain_values),
        "plain_std": np.std(plain_values, ddof=1),
        "weighted_mean": np.mean(weighted_values),
        "weighted_std": np.std(weighted_values, ddof=1),
        "mean_delta": np.mean(deltas),
        "std_delta": np.std(deltas, ddof=1),
        "folds_improved": int((deltas > 0).sum()),
        "folds_equal": int((deltas == 0).sum()),
        "folds_worsened": int((deltas < 0).sum())
    })

paired_comparison_df = pd.DataFrame(paired_records)

print("\n[F] FOLD-WISE PAIRED COMPARISON")
print(paired_comparison_df.round(4).to_string(index=False))

# ------------------------------------------------------------
# 11. PAIRED TESTS FOR PRIMARY METRICS
# ------------------------------------------------------------

print("\n[G] PAIRED FOLD TESTS")
print("Note: Only 10 outer folds are available; treat p-values cautiously.")

for metric in ["roc_auc", "pr_auc"]:

    plain_values = wide[metric]["Plain_RF"].to_numpy()
    weighted_values = wide[metric][
        "Reliability_Weighted_RF"
    ].to_numpy()

    differences = weighted_values - plain_values

    t_result = ttest_rel(
        weighted_values,
        plain_values
    )

    try:
        if np.allclose(differences, 0):
            wilcoxon_stat = 0.0
            wilcoxon_p = 1.0
        else:
            wilcoxon_result = wilcoxon(
                weighted_values,
                plain_values,
                alternative="two-sided"
            )
            wilcoxon_stat = wilcoxon_result.statistic
            wilcoxon_p = wilcoxon_result.pvalue

    except ValueError:
        wilcoxon_stat = np.nan
        wilcoxon_p = np.nan

    print(f"\n{metric.upper()}")
    print(f"Mean paired delta: {differences.mean():+.5f}")
    print(f"Improved folds: {(differences > 0).sum()}/10")
    print(f"Paired t-test p-value: {t_result.pvalue:.5f}")
    print(f"Wilcoxon p-value: {wilcoxon_p:.5f}")

# ------------------------------------------------------------
# 12. SAVE PREDICTIONS AND REPORTS
# ------------------------------------------------------------

prediction_df = pd.DataFrame({
    "row_index": np.arange(n_samples),
    "outer_fold": outer_ids,
    "component_id": component_ids,
    "scaffold": data["scaffold"].astype(str).to_numpy(),
    "unitox_label": y,
    "plain_probability": oof_plain,
    "weighted_probability": oof_weighted
})

prediction_df["plain_prediction"] = (
    prediction_df["plain_probability"] >= 0.5
).astype(int)

prediction_df["weighted_prediction"] = (
    prediction_df["weighted_probability"] >= 0.5
).astype(int)

fold_weight_df = pd.DataFrame(fold_weight_records)

prediction_path = Path(
    "final_outer_oof_plain_vs_weighted_predictions.csv"
)

fold_metrics_path = Path(
    "final_outer_fold_plain_vs_weighted_metrics.csv"
)

pooled_metrics_path = Path(
    "final_pooled_plain_vs_weighted_metrics.csv"
)

paired_path = Path(
    "final_paired_fold_comparison.csv"
)

weights_path = Path(
    "final_outer_fold_training_weight_audit.csv"
)

prediction_df.to_csv(prediction_path, index=False)
fold_metrics_df.to_csv(fold_metrics_path, index=False)
pooled_metrics_df.to_csv(pooled_metrics_path, index=False)
paired_comparison_df.to_csv(paired_path, index=False)
fold_weight_df.to_csv(weights_path, index=False)

print("\n[H] Saved files:")
print("1.", prediction_path)
print("2.", fold_metrics_path)
print("3.", pooled_metrics_path)
print("4.", paired_path)
print("5.", weights_path)

print("\n" + "=" * 95)
print("STEP 7.1 COMPLETE")
print("Each compound has exactly one outer-heldout prediction per model.")
print("Plain and weighted models use identical outer folds and features.")
print("No outer-heldout labels were used to construct training sample weights.")
print("=" * 95)


STEP 7 — FINAL PLAIN RF VS RELIABILITY-WEIGHTED RF

[A] Input verification
Feature matrix: (1349, 2048)
Target distribution: {0: 613, 1: 736}
Number of outer folds: 10
Number of curated components: 774

[B] Split leakage checks
Curated components crossing folds: 0
Scaffolds crossing folds: 0

[C] Model configuration
Model: RandomForestClassifier
Trees: 500
Random state: 42
Features: Morgan fingerprints, radius 2, 2048 bits
Plain model: no sample weights
Weighted model: R_i-derived sample weights
Outer fold 1/10 | train=1190 | test=159 | Plain ROC=0.6878 | Weighted ROC=0.7218 | Delta=+0.0340 | Plain PR=0.7029 | Weighted PR=0.7481
Outer fold 2/10 | train=1215 | test=134 | Plain ROC=0.6324 | Weighted ROC=0.6475 | Delta=+0.0150 | Plain PR=0.6830 | Weighted PR=0.6969
Outer fold 3/10 | train=1217 | test=132 | Plain ROC=0.6866 | Weighted ROC=0.6760 | Delta=-0.0106 | Plain PR=0.6951 | Weighted PR=0.6984
Outer fold 4/10 | train=1217 | test=132 | Plain ROC=0.7649 | Weighted ROC=0.7775 | Delta=+0

In [111]:

# ============================================================
# STEP 8.1 — HELD-OUT EXTERNAL VALIDATION
# A) Test reliability scores against DILIrank agreement
# B) Test RF predictions against DILIrank No vs Most
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    brier_score_loss
)

print("=" * 95)
print("STEP 8 — INDEPENDENT EXTERNAL VALIDATION")
print("=" * 95)

# ------------------------------------------------------------
# 1. INPUT VERIFICATION
# ------------------------------------------------------------

required_objects = [
    "A_heldout_evaluation_df",
    "prediction_df",
    "S_test_by_outer_fold"
]

for obj in required_objects:
    assert obj in globals(), f"Missing object: {obj}"

# ------------------------------------------------------------
# 2. PREPARE HELD-OUT S_i SCORES
# ------------------------------------------------------------

S_test_df = pd.concat(
    [
        S_test_by_outer_fold[fold]
        for fold in sorted(S_test_by_outer_fold)
    ],
    ignore_index=True
).copy()

assert "row_index" in S_test_df.columns
assert "outer_fold" in S_test_df.columns
assert "S_neighbour_final" in S_test_df.columns

S_test_df = S_test_df[
    ["row_index", "outer_fold", "S_neighbour_final"]
].rename(
    columns={
        "outer_fold": "scoring_outer_fold",
        "S_neighbour_final": "S_test"
    }
)

S_test_df["row_index"] = S_test_df["row_index"].astype(int)
S_test_df["scoring_outer_fold"] = (
    S_test_df["scoring_outer_fold"].astype(int)
)

assert not S_test_df.duplicated(
    ["row_index", "scoring_outer_fold"]
).any()

# Every compound should have exactly one held-out S score.
assert S_test_df["row_index"].nunique() == len(work_df)

# ------------------------------------------------------------
# 3. PREPARE HELD-OUT RF PREDICTIONS
# ------------------------------------------------------------

pred_test_df = prediction_df.copy()

pred_test_df = pred_test_df.rename(
    columns={
        "outer_fold": "scoring_outer_fold"
    }
)

pred_test_df["row_index"] = pred_test_df["row_index"].astype(int)
pred_test_df["scoring_outer_fold"] = (
    pred_test_df["scoring_outer_fold"].astype(int)
)

assert not pred_test_df.duplicated(
    ["row_index", "scoring_outer_fold"]
).any()

# M_test uses the probability assigned by the plain outer-trained RF
# to the observed UniTox label.
#
# This is a held-out analogue of model agreement:
# P(Y_i = observed UniTox label | X_i).

pred_test_df["M_test"] = np.where(
    pred_test_df["unitox_label"].to_numpy() == 1,
    pred_test_df["plain_probability"].to_numpy(),
    1 - pred_test_df["plain_probability"].to_numpy()
)

pred_test_df["M_test_hard"] = (
    pred_test_df["plain_prediction"]
    == pred_test_df["unitox_label"]
).astype(int)

# ------------------------------------------------------------
# 4. MERGE S_i, M_i AND HELD-OUT A_i
# ------------------------------------------------------------

A_test_df = A_heldout_evaluation_df.copy()

A_test_df["row_index"] = A_test_df["row_index"].astype(int)
A_test_df["scoring_outer_fold"] = (
    A_test_df["scoring_outer_fold"].astype(int)
)

test_validation_df = A_test_df.merge(
    S_test_df,
    on=["row_index", "scoring_outer_fold"],
    how="inner",
    validate="one_to_one"
)

test_validation_df = test_validation_df.merge(
    pred_test_df[
        [
            "row_index",
            "scoring_outer_fold",
            "plain_probability",
            "weighted_probability",
            "M_test",
            "M_test_hard",
            "unitox_label"
        ]
    ],
    on=["row_index", "scoring_outer_fold"],
    how="inner",
    validate="one_to_one"
)

assert len(test_validation_df) == len(work_df)

assert (
    test_validation_df["unitox_label_x"]
    == test_validation_df["unitox_label_y"]
).all()

test_validation_df = test_validation_df.rename(
    columns={
        "unitox_label_x": "unitox_label"
    }
).drop(
    columns=["unitox_label_y"]
)

# Missing-aware test reliability proxy.
# Do NOT include A_i: A_i is the independent evaluation target.
test_validation_df["R_test_MS"] = (
    test_validation_df["M_test"]
    + test_validation_df["S_test"]
) / 2

# ------------------------------------------------------------
# 5. SELECT EXTERNAL EVALUATION SUBSET
# ------------------------------------------------------------

external_test = test_validation_df.loc[
    test_validation_df["A_available"].astype(bool)
].copy()

external_test["A_i"] = pd.to_numeric(
    external_test["A_i"], errors="coerce"
)

external_test["external_expected_label"] = pd.to_numeric(
    external_test["external_expected_label"],
    errors="coerce"
)

assert external_test["A_i"].notna().all()
assert external_test["external_expected_label"].notna().all()

assert set(external_test["A_i"].unique()).issubset({0.0, 1.0})
assert set(
    external_test["external_expected_label"].unique()
).issubset({0, 1})

assert len(external_test) == 122

print("\n[A] External evaluation subset")
print("Number of compounds:", len(external_test))
print(
    "DILIrank expected-label distribution:",
    external_test["external_expected_label"].value_counts().sort_index().to_dict()
)
print(
    "A_i agreement distribution:",
    external_test["A_i"].value_counts().sort_index().to_dict()
)

# ------------------------------------------------------------
# 6. BOOTSTRAP UTILITIES
# ------------------------------------------------------------

RNG = np.random.default_rng(20261030)
N_BOOT = 2000


def stratified_bootstrap_auc_ap(y_true, scores, n_boot=N_BOOT):

    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    positive_idx = np.where(y_true == 1)[0]
    negative_idx = np.where(y_true == 0)[0]

    assert len(positive_idx) > 0
    assert len(negative_idx) > 0

    auc_values = []
    ap_values = []

    for _ in range(n_boot):

        sampled_pos = RNG.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        )

        sampled_neg = RNG.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )

        sampled_idx = np.concatenate([
            sampled_pos,
            sampled_neg
        ])

        y_boot = y_true[sampled_idx]
        score_boot = scores[sampled_idx]

        auc_values.append(
            roc_auc_score(y_boot, score_boot)
        )

        ap_values.append(
            average_precision_score(y_boot, score_boot)
        )

    return {
        "auc_lower_95": np.percentile(auc_values, 2.5),
        "auc_upper_95": np.percentile(auc_values, 97.5),
        "ap_lower_95": np.percentile(ap_values, 2.5),
        "ap_upper_95": np.percentile(ap_values, 97.5)
    }


def stratified_bootstrap_metric_differences(
    y_true,
    score_plain,
    score_weighted,
    n_boot=N_BOOT
):

    y_true = np.asarray(y_true, dtype=int)
    score_plain = np.asarray(score_plain, dtype=float)
    score_weighted = np.asarray(score_weighted, dtype=float)

    positive_idx = np.where(y_true == 1)[0]
    negative_idx = np.where(y_true == 0)[0]

    auc_deltas = []
    ap_deltas = []

    for _ in range(n_boot):

        sampled_pos = RNG.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        )

        sampled_neg = RNG.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )

        sampled_idx = np.concatenate([
            sampled_pos,
            sampled_neg
        ])

        y_boot = y_true[sampled_idx]

        auc_plain = roc_auc_score(
            y_boot, score_plain[sampled_idx]
        )

        auc_weighted = roc_auc_score(
            y_boot, score_weighted[sampled_idx]
        )

        ap_plain = average_precision_score(
            y_boot, score_plain[sampled_idx]
        )

        ap_weighted = average_precision_score(
            y_boot, score_weighted[sampled_idx]
        )

        auc_deltas.append(auc_weighted - auc_plain)
        ap_deltas.append(ap_weighted - ap_plain)

    return {
        "auc_delta_lower_95": np.percentile(auc_deltas, 2.5),
        "auc_delta_upper_95": np.percentile(auc_deltas, 97.5),
        "ap_delta_lower_95": np.percentile(ap_deltas, 2.5),
        "ap_delta_upper_95": np.percentile(ap_deltas, 97.5)
    }


# ------------------------------------------------------------
# 7. RELIABILITY VALIDATION AGAINST A_i
# ------------------------------------------------------------

print("\n[B] Reliability score validation against held-out A_i")

reliability_scores = {
    "M_test": external_test["M_test"].to_numpy(),
    "S_test": external_test["S_test"].to_numpy(),
    "R_test_MS": external_test["R_test_MS"].to_numpy()
}

a_true = external_test["A_i"].astype(int).to_numpy()

reliability_records = []
reliability_ci_records = []

for score_name, score_values in reliability_scores.items():

    auc = roc_auc_score(a_true, score_values)
    ap = average_precision_score(a_true, score_values)

    rho, rho_p = spearmanr(a_true, score_values)

    ci = stratified_bootstrap_auc_ap(
        a_true,
        score_values
    )

    reliability_records.append({
        "score": score_name,
        "n": len(a_true),
        "roc_auc_for_A": auc,
        "pr_auc_for_A": ap,
        "spearman_rho": rho,
        "spearman_p": rho_p
    })

    reliability_ci_records.append({
        "score": score_name,
        **ci
    })

reliability_results_df = pd.DataFrame(reliability_records)
reliability_ci_df = pd.DataFrame(reliability_ci_records)

print(reliability_results_df.round(4).to_string(index=False))

print("\nBootstrap 95% confidence intervals:")
print(reliability_ci_df.round(4).to_string(index=False))

# ------------------------------------------------------------
# 8. DIRECT EXTERNAL PREDICTION VALIDATION
# DILIrank No=0 and Most=1
# ------------------------------------------------------------

print("\n[C] Plain vs weighted RF on held-out DILIrank labels")


def external_prediction_metrics(y_true, probabilities):

    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = np.clip(probabilities, 1e-7, 1 - 1e-7)

    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "specificity": specificity,
        "log_loss": log_loss(
            y_true,
            probabilities,
            labels=[0, 1]
        ),
        "brier_score": brier_score_loss(
            y_true, probabilities
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


dili_y = external_test[
    "external_expected_label"
].astype(int).to_numpy()

plain_external_p = external_test[
    "plain_probability"
].to_numpy(dtype=float)

weighted_external_p = external_test[
    "weighted_probability"
].to_numpy(dtype=float)

plain_external_metrics = external_prediction_metrics(
    dili_y,
    plain_external_p
)

weighted_external_metrics = external_prediction_metrics(
    dili_y,
    weighted_external_p
)

external_model_results_df = pd.DataFrame([
    {
        "model": "Plain_RF",
        **plain_external_metrics
    },
    {
        "model": "Reliability_Weighted_RF",
        **weighted_external_metrics
    }
])

print(
    external_model_results_df.set_index("model")
    .round(4)
    .to_string()
)

# ------------------------------------------------------------
# 9. BOOTSTRAP CONFIDENCE INTERVALS FOR DIRECT EXTERNAL AUC/AP
# ------------------------------------------------------------

plain_dili_ci = stratified_bootstrap_auc_ap(
    dili_y,
    plain_external_p
)

weighted_dili_ci = stratified_bootstrap_auc_ap(
    dili_y,
    weighted_external_p
)

dili_delta_ci = stratified_bootstrap_metric_differences(
    dili_y,
    plain_external_p,
    weighted_external_p
)

direct_external_ci_df = pd.DataFrame([
    {
        "model": "Plain_RF",
        **plain_dili_ci
    },
    {
        "model": "Reliability_Weighted_RF",
        **weighted_dili_ci
    }
])

print("\n[D] Direct DILIrank prediction bootstrap 95% CIs")
print(direct_external_ci_df.round(4).to_string(index=False))

print("\n[E] Paired bootstrap difference: weighted minus plain")
print(pd.Series(dili_delta_ci).round(4).to_string())

# ------------------------------------------------------------
# 10. SAVE EXTERNAL VALIDATION OUTPUTS
# ------------------------------------------------------------

validation_path = Path(
    "final_heldout_external_reliability_validation.csv"
)

reliability_results_path = Path(
    "final_external_reliability_association_metrics.csv"
)

reliability_ci_path = Path(
    "final_external_reliability_bootstrap_ci.csv"
)

external_model_path = Path(
    "final_dilirank_heldout_prediction_metrics.csv"
)

external_ci_path = Path(
    "final_dilirank_heldout_prediction_bootstrap_ci.csv"
)

test_validation_df.to_csv(validation_path, index=False)
reliability_results_df.to_csv(reliability_results_path, index=False)
reliability_ci_df.to_csv(reliability_ci_path, index=False)
external_model_results_df.to_csv(external_model_path, index=False)
direct_external_ci_df.to_csv(external_ci_path, index=False)

print("\n[F] Saved files:")
print("1.", validation_path)
print("2.", reliability_results_path)
print("3.", reliability_ci_path)
print("4.", external_model_path)
print("5.", external_ci_path)

print("\n" + "=" * 95)
print("STEP 8.1 COMPLETE")
print("Reliability association was evaluated without using A_i in R_test_MS.")
print("Held-out DILIrank labels were not used to train the corresponding outer-fold models.")
print("Bootstrap intervals use stratified resampling.")
print("=" * 95)


STEP 8 — INDEPENDENT EXTERNAL VALIDATION

[A] External evaluation subset
Number of compounds: 122
DILIrank expected-label distribution: {0.0: 67, 1.0: 55}
A_i agreement distribution: {0.0: 17, 1.0: 105}

[B] Reliability score validation against held-out A_i
    score   n  roc_auc_for_A  pr_auc_for_A  spearman_rho  spearman_p
   M_test 122         0.6202        0.9171        0.1442      0.1131
   S_test 122         0.6487        0.9203        0.1784      0.0492
R_test_MS 122         0.6459        0.9220        0.1751      0.0538

Bootstrap 95% confidence intervals:
    score  auc_lower_95  auc_upper_95  ap_lower_95  ap_upper_95
   M_test        0.4840        0.7597       0.8844       0.9509
   S_test        0.5059        0.7849       0.8799       0.9567
R_test_MS        0.5059        0.7782       0.8856       0.9552

[C] Plain vs weighted RF on held-out DILIrank labels
                         roc_auc  pr_auc  accuracy  balanced_accuracy  precision  recall  specificity  log_loss  brier_

In [112]:

# ============================================================
# STEP 9.1 — RELIABILITY COMPONENT ABLATION STUDY
# Plain / A-only / M-only / S-only / MS / AMS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("=" * 100)
print("STEP 9 — RELIABILITY COMPONENT ABLATION")
print("=" * 100)

# ------------------------------------------------------------
# 1. INPUT VERIFICATION
# ------------------------------------------------------------

assert "work_df" in globals()
assert "X_morgan" in globals()
assert "curated_outer_fold_ids" in globals()
assert "R_nested_training_df" in globals()

data = work_df.reset_index(drop=True).copy()

X = np.asarray(X_morgan)

y = pd.to_numeric(
    data["liver_toxicity_binary_rating_0_1"],
    errors="coerce"
).to_numpy(dtype=int)

outer_ids = np.asarray(curated_outer_fold_ids)

assert X.shape == (len(data), 2048)
assert len(y) == len(data)
assert len(outer_ids) == len(data)

outer_folds = sorted(np.unique(outer_ids).tolist())

rel = R_nested_training_df.copy()

required_cols = [
    "row_index",
    "scoring_outer_fold",
    "A_i",
    "A_available",
    "M_i",
    "S_i",
    "R_i"
]

for col in required_cols:
    assert col in rel.columns, f"Missing reliability column: {col}"

rel["row_index"] = rel["row_index"].astype(int)
rel["scoring_outer_fold"] = (
    rel["scoring_outer_fold"].astype(int)
)

assert not rel.duplicated(
    ["row_index", "scoring_outer_fold"]
).any()

assert rel["M_i"].between(0, 1).all()
assert rel["S_i"].between(0, 1).all()
assert rel["R_i"].between(0, 1).all()

print("\n[A] Inputs")
print("Samples:", len(y))
print("Features:", X.shape[1])
print("Outer folds:", len(outer_folds))
print("Training reliability rows:", len(rel))

# ------------------------------------------------------------
# 2. MODEL CONFIGURATION
# ------------------------------------------------------------

N_ESTIMATORS = 500
MODEL_RANDOM_STATE = 42
WEIGHT_FLOOR = 0.10

model_names = [
    "Plain_RF",
    "A_only",
    "M_only",
    "S_only",
    "MS_only",
    "AMS_full"
]

oof_predictions = {
    name: np.full(len(y), np.nan, dtype=float)
    for name in model_names
}

fold_records = []
weight_records = []

# ------------------------------------------------------------
# 3. HELPER FUNCTIONS
# ------------------------------------------------------------

def normalize_weights(raw_weights):

    raw_weights = np.asarray(raw_weights, dtype=float)

    assert np.isfinite(raw_weights).all()
    assert (raw_weights > 0).all()

    weights = raw_weights / raw_weights.mean()

    assert np.isclose(weights.mean(), 1.0, atol=1e-10)

    return weights


def positive_probability(model, X_eval):

    probs = model.predict_proba(X_eval)

    pos_idx = np.where(model.classes_ == 1)[0]

    assert len(pos_idx) == 1

    return probs[:, pos_idx[0]]


def fold_metrics(y_true, probs):

    predictions = (probs >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, probs),
        "pr_auc": average_precision_score(y_true, probs),
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        )
    }


# ------------------------------------------------------------
# 4. OUTER-FOLD TRAINING
# ------------------------------------------------------------

for fold_position, fold in enumerate(outer_folds, start=1):

    train_idx = np.where(outer_ids != fold)[0]
    test_idx = np.where(outer_ids == fold)[0]

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    fold_rel = rel.loc[
        rel["scoring_outer_fold"] == int(fold)
    ].copy()

    assert fold_rel["row_index"].is_unique
    fold_rel = fold_rel.set_index("row_index")

    assert set(fold_rel.index) == set(train_idx)

    fold_rel = fold_rel.loc[train_idx]

    A = fold_rel["A_i"].to_numpy(dtype=float)
    A_available = (
        fold_rel["A_available"].to_numpy(dtype=bool)
    )
    M = fold_rel["M_i"].to_numpy(dtype=float)
    S = fold_rel["S_i"].to_numpy(dtype=float)
    R = fold_rel["R_i"].to_numpy(dtype=float)

    assert np.isfinite(M).all()
    assert np.isfinite(S).all()
    assert np.isfinite(R).all()

    # --------------------------------------------------------
    # 5. COMPONENT-SPECIFIC RAW SAMPLE WEIGHTS
    # --------------------------------------------------------

    # A-only:
    # Missing external evidence is neutral (raw weight = 1).
    raw_A = np.where(
        A_available,
        WEIGHT_FLOOR + (1 - WEIGHT_FLOOR) * A,
        1.0
    )

    # M-only.
    raw_M = (
        WEIGHT_FLOOR
        + (1 - WEIGHT_FLOOR) * M
    )

    # S-only.
    raw_S = (
        WEIGHT_FLOOR
        + (1 - WEIGHT_FLOOR) * S
    )

    # MS-only.
    R_MS = (M + S) / 2

    raw_MS = (
        WEIGHT_FLOOR
        + (1 - WEIGHT_FLOOR) * R_MS
    )

    # Full AMS score.
    raw_AMS = (
        WEIGHT_FLOOR
        + (1 - WEIGHT_FLOOR) * R
    )

    weights_by_model = {
        "A_only": normalize_weights(raw_A),
        "M_only": normalize_weights(raw_M),
        "S_only": normalize_weights(raw_S),
        "MS_only": normalize_weights(raw_MS),
        "AMS_full": normalize_weights(raw_AMS)
    }

    # Verify model-specific mean-one weights.
    for model_name, weights in weights_by_model.items():

        assert len(weights) == len(train_idx)
        assert np.isclose(weights.mean(), 1.0, atol=1e-10)

        weight_records.append({
            "outer_fold": int(fold),
            "model": model_name,
            "mean_weight": weights.mean(),
            "std_weight": weights.std(),
            "min_weight": weights.min(),
            "max_weight": weights.max()
        })

    # --------------------------------------------------------
    # 6. FIT PLAIN AND WEIGHTED MODELS
    # --------------------------------------------------------

    for model_name in model_names:

        model = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1
        )

        if model_name == "Plain_RF":
            model.fit(X_train, y_train)
        else:
            model.fit(
                X_train,
                y_train,
                sample_weight=weights_by_model[model_name]
            )

        probs = positive_probability(model, X_test)

        oof_predictions[model_name][test_idx] = probs

        metrics = fold_metrics(y_test, probs)

        fold_records.append({
            "outer_fold": int(fold),
            "model": model_name,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            **metrics
        })

    # --------------------------------------------------------
    # 7. FOLD PROGRESS SUMMARY
    # --------------------------------------------------------

    fold_now = pd.DataFrame(
        [
            r for r in fold_records
            if r["outer_fold"] == int(fold)
        ]
    ).set_index("model")

    plain_roc = fold_now.loc["Plain_RF", "roc_auc"]

    print(
        f"\nFold {fold_position}/10 | "
        f"train={len(train_idx)} | test={len(test_idx)}"
    )

    for model_name in model_names[1:]:

        roc = fold_now.loc[model_name, "roc_auc"]
        pr = fold_now.loc[model_name, "pr_auc"]

        print(
            f"  {model_name:10s} | "
            f"ROC={roc:.4f} | "
            f"PR={pr:.4f} | "
            f"ROC delta vs plain={roc - plain_roc:+.4f}"
        )

# ------------------------------------------------------------
# 8. VERIFY PREDICTIONS
# ------------------------------------------------------------

print("\n[B] OOF prediction verification")

for model_name, predictions in oof_predictions.items():

    count = int(np.isfinite(predictions).sum())

    print(f"{model_name}: {count}/{len(y)}")

    assert count == len(y)

# ------------------------------------------------------------
# 9. POOLED PERFORMANCE
# ------------------------------------------------------------

pooled_records = []

for model_name in model_names:

    metrics = fold_metrics(
        y,
        oof_predictions[model_name]
    )

    pooled_records.append({
        "model": model_name,
        **metrics
    })

pooled_df = pd.DataFrame(pooled_records)

print("\n[C] POOLED OUTER-HELDOUT PERFORMANCE")
print(pooled_df.round(4).to_string(index=False))

# ------------------------------------------------------------
# 10. FOLD-WISE MEAN AND IMPROVEMENT COUNTS
# ------------------------------------------------------------

fold_metrics_df = pd.DataFrame(fold_records)

fold_mean = (
    fold_metrics_df
    .groupby("model")
    .agg({
        "roc_auc": ["mean", "std"],
        "pr_auc": ["mean", "std"],
        "accuracy": ["mean", "std"],
        "balanced_accuracy": ["mean", "std"],
        "f1": ["mean", "std"]
    })
)

fold_mean.columns = [
    "_".join(col)
    for col in fold_mean.columns
]

fold_mean = fold_mean.reset_index()

print("\n[D] FOLD-WISE MEAN PERFORMANCE")
print(fold_mean.round(4).to_string(index=False))

wide_roc = fold_metrics_df.pivot(
    index="outer_fold",
    columns="model",
    values="roc_auc"
)

wide_pr = fold_metrics_df.pivot(
    index="outer_fold",
    columns="model",
    values="pr_auc"
)

comparison_records = []

for model_name in model_names[1:]:

    roc_delta = (
        wide_roc[model_name] - wide_roc["Plain_RF"]
    )

    pr_delta = (
        wide_pr[model_name] - wide_pr["Plain_RF"]
    )

    comparison_records.append({
        "model": model_name,
        "mean_ROC_delta": roc_delta.mean(),
        "std_ROC_delta": roc_delta.std(ddof=1),
        "ROC_folds_improved": int((roc_delta > 0).sum()),
        "mean_PR_delta": pr_delta.mean(),
        "std_PR_delta": pr_delta.std(ddof=1),
        "PR_folds_improved": int((pr_delta > 0).sum())
    })

comparison_df = pd.DataFrame(comparison_records)

print("\n[E] COMPONENT CONTRIBUTION VS PLAIN RF")
print(comparison_df.round(5).to_string(index=False))

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

prediction_output = pd.DataFrame({
    "row_index": np.arange(len(y)),
    "outer_fold": outer_ids,
    "unitox_label": y
})

for model_name, predictions in oof_predictions.items():

    prediction_output[
        f"{model_name}_probability"
    ] = predictions

    prediction_output[
        f"{model_name}_prediction"
    ] = (predictions >= 0.5).astype(int)

fold_metrics_path = Path(
    "final_ablation_outer_fold_metrics.csv"
)

pooled_path = Path(
    "final_ablation_pooled_metrics.csv"
)

comparison_path = Path(
    "final_ablation_component_comparison.csv"
)

weights_path = Path(
    "final_ablation_sample_weight_audit.csv"
)

predictions_path = Path(
    "final_ablation_outer_oof_predictions.csv"
)

fold_metrics_df.to_csv(fold_metrics_path, index=False)
pooled_df.to_csv(pooled_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
pd.DataFrame(weight_records).to_csv(weights_path, index=False)
prediction_output.to_csv(predictions_path, index=False)

print("\n[F] Saved files:")
print("1.", fold_metrics_path)
print("2.", pooled_path)
print("3.", comparison_path)
print("4.", weights_path)
print("5.", predictions_path)

print("\n" + "=" * 100)
print("STEP 9.1 COMPLETE")
print("All six configurations use identical outer folds and test compounds.")
print("A-only treats missing external evidence neutrally.")
print("All weighted configurations have mean training weight 1 per fold.")
print("=" * 100)


STEP 9 — RELIABILITY COMPONENT ABLATION

[A] Inputs
Samples: 1349
Features: 2048
Outer folds: 10
Training reliability rows: 12141

Fold 1/10 | train=1190 | test=159
  A_only     | ROC=0.6887 | PR=0.7056 | ROC delta vs plain=+0.0009
  M_only     | ROC=0.7203 | PR=0.7524 | ROC delta vs plain=+0.0324
  S_only     | ROC=0.7217 | PR=0.7509 | ROC delta vs plain=+0.0338
  MS_only    | ROC=0.7147 | PR=0.7397 | ROC delta vs plain=+0.0268
  AMS_full   | ROC=0.7218 | PR=0.7481 | ROC delta vs plain=+0.0340

Fold 2/10 | train=1215 | test=134
  A_only     | ROC=0.6314 | PR=0.6805 | ROC delta vs plain=-0.0010
  M_only     | ROC=0.6340 | PR=0.7092 | ROC delta vs plain=+0.0016
  S_only     | ROC=0.6421 | PR=0.6909 | ROC delta vs plain=+0.0097
  MS_only    | ROC=0.6282 | PR=0.7007 | ROC delta vs plain=-0.0042
  AMS_full   | ROC=0.6475 | PR=0.6969 | ROC delta vs plain=+0.0150

Fold 3/10 | train=1217 | test=132
  A_only     | ROC=0.6679 | PR=0.6712 | ROC delta vs plain=-0.0188
  M_only     | ROC=0.6810 | 

In [113]:

# ============================================================
# STEP 9.2 — PAIRED ABLATION STATISTICAL ANALYSIS
# Holm correction across 5 variants x 2 primary metrics
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import ttest_rel, wilcoxon, t

assert "fold_metrics_df" in globals(), (
    "Run Step 9.1 first; fold_metrics_df is missing."
)

metrics_to_test = ["roc_auc", "pr_auc"]

variants = [
    "A_only",
    "M_only",
    "S_only",
    "MS_only",
    "AMS_full"
]

wide = fold_metrics_df.pivot(
    index="outer_fold",
    columns="model",
    values=metrics_to_test
).sort_index()

assert "Plain_RF" in wide["roc_auc"].columns

n_folds = len(wide)
assert n_folds == 10

# Fold-size correction information.
fold_sizes = (
    fold_metrics_df[
        fold_metrics_df["model"] == "Plain_RF"
    ][["outer_fold", "n_train", "n_test"]]
    .sort_values("outer_fold")
)

mean_test_train_ratio = np.mean(
    fold_sizes["n_test"].to_numpy()
    / fold_sizes["n_train"].to_numpy()
)

# Nadeau-Bengio-style corrected resampled t-test factor.
# It accounts approximately for the dependence caused by
# overlapping training sets in cross-validation.
correction_factor = (
    (1 / n_folds) + mean_test_train_ratio
)

records = []

for metric in metrics_to_test:

    plain = wide[metric]["Plain_RF"].to_numpy(dtype=float)

    for variant in variants:

        candidate = wide[metric][variant].to_numpy(dtype=float)
        differences = candidate - plain

        mean_delta = differences.mean()
        sd_delta = differences.std(ddof=1)

        # Conventional paired t-test.
        paired_t = ttest_rel(candidate, plain)

        # Corrected resampled t-test.
        corrected_se = np.sqrt(
            correction_factor * (sd_delta ** 2)
        )

        if corrected_se > 0:
            corrected_t_stat = mean_delta / corrected_se
            corrected_p = 2 * t.sf(
                abs(corrected_t_stat),
                df=n_folds - 1
            )
        else:
            corrected_t_stat = 0.0
            corrected_p = 1.0

        # Paired Wilcoxon signed-rank test.
        if np.allclose(differences, 0):
            wilcoxon_stat = 0.0
            wilcoxon_p = 1.0
        else:
            w_result = wilcoxon(
                candidate,
                plain,
                alternative="two-sided",
                zero_method="wilcox"
            )
            wilcoxon_stat = w_result.statistic
            wilcoxon_p = w_result.pvalue

        # Conventional paired t confidence interval.
        critical_t = t.ppf(
            0.975,
            df=n_folds - 1
        )

        conventional_se = sd_delta / np.sqrt(n_folds)

        ci_lower = mean_delta - critical_t * conventional_se
        ci_upper = mean_delta + critical_t * conventional_se

        records.append({
            "metric": metric,
            "variant": variant,
            "mean_delta_vs_plain": mean_delta,
            "sd_paired_delta": sd_delta,
            "ci95_lower_conventional": ci_lower,
            "ci95_upper_conventional": ci_upper,
            "folds_improved": int((differences > 0).sum()),
            "folds_worsened": int((differences < 0).sum()),
            "paired_t_p": paired_t.pvalue,
            "corrected_t_p": corrected_p,
            "wilcoxon_p": wilcoxon_p
        })

results = pd.DataFrame(records)

# ------------------------------------------------------------
# HOLM CORRECTION: 10 WILCOXON TESTS
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    m = len(p_values)
    order = np.argsort(p_values)

    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (m - np.arange(m)) * sorted_p
    )

    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted

    return adjusted


results["wilcoxon_holm_p"] = holm_adjust(
    results["wilcoxon_p"].to_numpy()
)

results["significant_holm_0_05"] = (
    results["wilcoxon_holm_p"] < 0.05
)

print("=" * 100)
print("STEP 9.2 — PAIRED ABLATION INFERENCE")
print("=" * 100)

print("\n[A] Test configuration")
print("Outer folds:", n_folds)
print("Variants:", len(variants))
print("Primary metrics:", metrics_to_test)
print("Total Wilcoxon tests corrected by Holm:", len(results))
print("Mean test/train size ratio:", round(mean_test_train_ratio, 5))
print("Corrected t-test factor:", round(correction_factor, 5))

print("\n[B] ROC-AUC comparisons")
print(
    results.loc[
        results["metric"] == "roc_auc"
    ].round(5).to_string(index=False)
)

print("\n[C] PR-AUC comparisons")
print(
    results.loc[
        results["metric"] == "pr_auc"
    ].round(5).to_string(index=False)
)

print("\n[D] Holm-adjusted significance")
significant = results.loc[
    results["significant_holm_0_05"]
]

if len(significant) == 0:
    print("No variant shows a statistically significant improvement")
    print("after Holm correction at alpha = 0.05.")
else:
    print(significant[
        ["metric", "variant", "mean_delta_vs_plain", "wilcoxon_holm_p"]
    ].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output_path = Path(
    "final_ablation_statistical_inference.csv"
)

results.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\n" + "=" * 100)
print("STEP 9.2 COMPLETE")
print("Holm correction applied to the 10 paired Wilcoxon tests.")
print("Corrected t-test is included as an additional sensitivity analysis.")
print("Ten-fold inference remains exploratory because training sets overlap.")
print("=" * 100)


STEP 9.2 — PAIRED ABLATION INFERENCE

[A] Test configuration
Outer folds: 10
Variants: 5
Primary metrics: ['roc_auc', 'pr_auc']
Total Wilcoxon tests corrected by Holm: 10
Mean test/train size ratio: 0.11116
Corrected t-test factor: 0.21116

[B] ROC-AUC comparisons
 metric  variant  mean_delta_vs_plain  sd_paired_delta  ci95_lower_conventional  ci95_upper_conventional  folds_improved  folds_worsened  paired_t_p  corrected_t_p  wilcoxon_p  wilcoxon_holm_p  significant_holm_0_05
roc_auc   A_only              0.00313          0.00972                 -0.00382                  0.01009               7               3     0.33490        0.50096     0.27539              1.0                  False
roc_auc   M_only              0.00022          0.01653                 -0.01160                  0.01205               4               6     0.96697        0.97726     0.92188              1.0                  False
roc_auc   S_only              0.00515          0.01658                 -0.00671        